# MathVisionKid — chuẩn bị toàn bộ dữ liệu trên Colab

Notebook đọc toàn bộ member ứng viên được công bố theo STATE/PROGRESS và hash, gồm mọi gói bổ sung. Gói cân bằng nhỏ cũ giữ làm lịch sử/diagnostic, không dùng thay toàn kho.

Chỉ kiểm dữ liệu: không weights, trainer, GPU hoặc huấn luyện. Đăng nhập tài khoản có quyền đọc cùng ID Drive. Payload tạm nằm trong runtime Colab, tối đa một payload cache mỗi lúc, không tải kho về máy cá nhân.

Mỗi lần xử lý một đoạn tiếp nối; chạy tiếp sẽ tới các dòng/member chưa xử lý, không chọn tập nhỏ cố định. Train/validation/test giữ split gốc; sealed/pilot/challenge và history không tự vào train. Tổng candidate chưa phải tổng train-only đã kiểm. Full archive token/linkage/split audit vẫn pending.


In [2]:
# Chỉ thư viện tokenizer đã pin từ pipeline.
%pip -q install transformers==5.16.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 71.1 MB/s eta 0:00:00


In [3]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import json,sys,sqlite3,hashlib
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
ROOT = Path('/content/mathvisionkid_data_only')
ROOT.mkdir(parents=True,exist_ok=True)
exec(compile('"""Account-neutral, data-only Colab bootstrap over fixed Drive IDs and hashes.\n\nNo training, weights, sharing edits or writes to Drive. Payloads are cached only\nin the caller\'s runtime directory; this module is not executed on the PC.\n"""\nfrom collections.abc import Mapping\nfrom pathlib import Path\nimport hashlib,io,json,zipfile\n\nSTATE_ID=\'1TLoQIbrRVmgsg-2uiUujCfM3HrK62Yme\'\nPROGRESS_ID=\'1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX\'\nCODE_FILES=(\'exhaustive_colab_loader.py\',\'math_data_portable_resume.py\',\'math_data_archive_audit.py\',\'math_data_candidate_stream.py\')\ndef sha(data):return hashlib.sha256(data).hexdigest()\ndef verify(data,ref):\n    if len(data)!=ref[\'bytes\'] or sha(data)!=ref[\'sha256\']:raise ValueError(\'pinned bytes/SHA256 mismatch\')\n    return data\n\nclass GoogleDriveReader:\n    def __init__(self,service):self.service=service\n    def read(self,file_id,max_bytes):\n        from googleapiclient.http import MediaIoBaseDownload\n        meta=self.service.files().get(fileId=file_id,fields=\'id,size,mimeType\',supportsAllDrives=True).execute()\n        size=int(meta.get(\'size\',-1))\n        if size<0 or size>max_bytes:raise ValueError(\'Drive raw size exceeds bounded request\')\n        output=io.BytesIO();down=MediaIoBaseDownload(output,self.service.files().get_media(fileId=file_id,supportsAllDrives=True),chunksize=4*1024*1024)\n        done=False\n        while not done:\n            _,done=down.next_chunk(num_retries=3)\n            if output.tell()>max_bytes:raise ValueError(\'download exceeds bounded request\')\n        raw=output.getvalue()\n        if len(raw)!=size:raise ValueError(\'Drive raw size changed during download\')\n        return raw\n\ndef latest_controls(reader):\n    """Mutable STATE read twice; recipe itself is pinned by its exact hash."""\n    before=reader.read(STATE_ID,16*1024*1024);state=json.loads(before)\n    pointer=state[\'latest_all_data_union_recipe\']\n    if pointer[\'progress_id\']!=PROGRESS_ID:raise ValueError(\'unexpected progress control ID\')\n    member=pointer[\'member\']\n    if not isinstance(member,str) or member.startswith(\'/\') or \'..\' in Path(member).parts:raise ValueError(\'unsafe recipe member\')\n    raw=reader.read(PROGRESS_ID,128*1024*1024)\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if archive.namelist().count(member)!=1:raise ValueError(\'missing/ambiguous latest union recipe\')\n        data=archive.read(member)\n    if sha(data)!=pointer[\'sha256\']:raise ValueError(\'STATE recipe hash differs from PROGRESS\')\n    recipe=json.loads(data)\n    if recipe.get(\'pending_discovery\'):raise ValueError(\'unresolved declared supplements; refusing partial data\')\n    if reader.read(STATE_ID,16*1024*1024)!=before:raise ValueError(\'STATE changed during read; resume from latest controls\')\n    if state.get(\'training_started\'):raise ValueError(\'unexpected training-started state for data-only notebook\')\n    return state,recipe,pointer\n\ndef bootstrap_code(reader,state,directory):\n    """Load exact code members from latest accepted bundle; no archive extractall."""\n    runs=[r for r in state.get(\'sgk_finish_runs\',[]) if r.get(\'bundle\')]\n    if not runs:raise ValueError(\'no explicit accepted code bundle\')\n    run=runs[-1];ref=run[\'bundle\'];fid=ref.get(\'id\',ref.get(\'file_id\'))\n    if not fid:raise ValueError(\'bundle file ID required\')\n    raw=verify(reader.read(fid,ref[\'bytes\']),ref)\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    receipts=[]\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError(\'duplicate bundle member\')\n        manifest=json.loads(archive.read(\'manifest.json\'))\n        for name in CODE_FILES:\n            path=\'code/\'+name;pin=manifest[\'members\'].get(path)\n            if not pin:raise ValueError(\'latest bundle lacks candidate reader dependency: \'+name)\n            data=verify(archive.read(path),pin)\n            (directory/name).write_bytes(data);receipts.append(dict(member=path,sha256=sha(data),bytes=len(data)))\n    return dict(bundle_id=fid,bundle_sha256=ref[\'sha256\'],code_members=receipts,training_started=False)\n\nclass LazyDriveCache(Mapping):\n    """One hash-verified payload cached at a time in ephemeral Colab storage.\n\n    source/chat members remain inside their archives. Checkpoints are separate\n    and never removed by cache eviction. No files are created on Drive.\n    """\n    def __init__(self,reader,recipe,directory):\n        self.reader=reader;self.directory=Path(directory);self.directory.mkdir(parents=True,exist_ok=True)\n        self.artifacts=recipe[\'artifacts\'];self.lookup={};self.current=None\n        for digest,ref in self.artifacts.items():\n            if digest!=ref[\'sha256\'] or not ref.get(\'file_ids\'):raise ValueError(\'hash and existing Drive ID required\')\n            self.lookup[digest]=digest\n            for fid in ref[\'file_ids\']:self.lookup[fid]=digest\n    def __iter__(self):return iter(self.lookup)\n    def __len__(self):return len(self.lookup)\n    def __getitem__(self,key):\n        digest=self.lookup[key];ref=self.artifacts[digest]\n        if self.current and self.current[0]==digest:\n            path=self.current[1]\n            if path.stat().st_size!=ref[\'bytes\'] or sha(path.read_bytes())!=digest:raise ValueError(\'cached payload changed\')\n            return path\n        data=verify(self.reader.read(ref[\'file_ids\'][0],ref[\'bytes\']),ref)\n        suffix=\'.gz\' if any(n.endswith(\'.gz\') for n in ref.get(\'file_names\',[])) else \'.zip\'\n        dest=self.directory/(digest+suffix);dest.write_bytes(data)\n        if self.current:\n            old=self.current[1]\n            if old.parent.resolve()!=self.directory.resolve():raise ValueError(\'cache deletion escaped runtime directory\')\n            old.unlink()\n        self.current=(digest,dest)\n        return dest\n\ndef scope_receipt(state,recipe,pointer):\n    return dict(accepted_cases=state[\'accepted_cases\'],declared_candidate_chat_rows=state[\'training_chat_rows\'],payload_references=len(recipe[\'artifacts\']),chat_members=len(recipe[\'chat_members\']),recipe_sha256=pointer[\'sha256\'],historical_train_only_total_verified=False,full_archive_token_split_audit_complete=False,TRAINING_DATASET_READY=False,training_started=False,selection=\'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET\',runtime_accounts=\'Authenticate any account with access to the same existing IDs; no credentials in dataset\')\n', 'math_data_drive_bootstrap.py', 'exec'), globals())
reader = GoogleDriveReader(drive_api)
state,recipe,recipe_pointer = latest_controls(reader)
code_receipt = bootstrap_code(reader,state,ROOT/'code')
sys.path.insert(0,str(ROOT/'code'))
from math_data_candidate_stream import CandidateStream,DataOnlyConsumer
from exhaustive_colab_loader import load_qwen_tokenizer,tokenize_assistant_only
print(scope_receipt(state,recipe,recipe_pointer))


{'accepted_cases': 1629922, 'declared_candidate_chat_rows': 6502344, 'payload_references': 38, 'chat_members': 32, 'recipe_sha256': '1bb69c394c7eaaea7034125c6b3361eebbbe0d77e0715a8454331fbe7cee3ff8', 'historical_train_only_total_verified': False, 'full_archive_token_split_audit_complete': False, 'TRAINING_DATASET_READY': False, 'training_started': False, 'selection': 'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET', 'runtime_accounts': 'Authenticate any account with access to the same existing IDs; no credentials in dataset'}


Checkpoint chỉ chứa tiến độ/hash/ID. Mỗi recipe/split dùng một SQLite trong thư mục Drive gốc, chạy lại cập nhật cùng ID. Chỉ một notebook ghi mỗi lúc; SQLite không là khóa phân tán Drive. Recipe thay đổi cần chuyển checkpoint append-only, không xóa bản cũ để ép chạy.


In [ ]:
FOLDER_ID='19ZxkfKVIggdcbSfsAdCtG1ibfJ0CRN3n'
SPLIT='train'  # Chỉ đổi rõ thành validation/test khi cần kiểm split đó.
ROWS_PER_RUN=20000  # Đoạn công việc tiếp nối, không chọn tập train cố định.
TOKENIZE=True  # Chỉ tokenizer pinned, không model weights.
cache=LazyDriveCache(reader,recipe,ROOT/'payload_cache')
stream=CandidateStream(recipe,cache,split=SPLIT)
checkpoint_name=f'MathVisionKid_DATA_ONLY_{SPLIT}_{stream.identity[:24]}.sqlite'
query=f"'{FOLDER_ID}' in parents and name = '{checkpoint_name}' and trashed = false"
matches=drive_api.files().list(q=query,fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(matches)>1:
    raise RuntimeError('Checkpoint trùng tên: giữ nguyên tất cả, cần xác định ID trước khi tiếp.')
checkpoint_ref=matches[0] if matches else None
DB=ROOT/checkpoint_name
if not DB.exists() and checkpoint_ref:
    raw_db=reader.read(checkpoint_ref['id'],int(checkpoint_ref['size']))
    pin=checkpoint_ref.get('appProperties',{}).get('data_sha256')
    if not pin or hashlib.sha256(raw_db).hexdigest()!=pin:
        raise RuntimeError('Checkpoint thiếu hoặc sai hash; không tự khởi tạo lại.')
    DB.write_bytes(raw_db)
consumer=DataOnlyConsumer(DB,stream)
tokenizer=load_qwen_tokenizer() if TOKENIZE else None
span={'rows':0,'max_sequence_tokens':0,'max_completion_tokens':0,'source_links_not_reverified_here':0}
def data_only_check(row,provenance):
    if tokenizer is not None:
        item=tokenize_assistant_only(row,tokenizer)
        span['max_sequence_tokens']=max(span['max_sequence_tokens'],len(item['input_ids']))
        span['max_completion_tokens']=max(span['max_completion_tokens'],item['completion_tokens'])
    span['rows']+=1
    if provenance.get('sourcehash_origin')!='ORIGINAL_FIELD_VERIFIED_AGAINST_PINNED_SOURCE_MEMBER':
        span['source_links_not_reverified_here']+=1
try:
    progress=consumer.consume(data_only_check,max_rows=ROWS_PER_RUN)
finally:
    consumer.close()
print({'progress':progress,'this_work_span_only':span,'full_archive_linkage_split_audit_complete':False,'training_started':False})


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'full_archive_linkage_split_audit_complete': False, 'training_started': False}


In [ ]:
# Lưu cùng ID sau khi xử lý thành công và SQLite đã đóng.
from googleapiclient.http import MediaFileUpload
checkpoint_sha=hashlib.sha256(DB.read_bytes()).hexdigest()
properties={'data_sha256':checkpoint_sha,'recipe_identity':stream.identity,'split':SPLIT,'purpose':'DATA_ONLY_NO_TRAINING'}
if checkpoint_ref:
    fresh=drive_api.files().get(fileId=checkpoint_ref['id'],fields='id,appProperties',supportsAllDrives=True).execute()
    if fresh.get('appProperties',{}).get('data_sha256')!=checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Checkpoint đã được cập nhật; không ghi đè tiến độ mới.')
    checkpoint_result=drive_api.files().update(fileId=checkpoint_ref['id'],body={'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
else:
    checkpoint_result=drive_api.files().create(body={'name':checkpoint_name,'parents':[FOLDER_ID],'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
verified_db=reader.read(checkpoint_result['id'],int(checkpoint_result['size']))
if hashlib.sha256(verified_db).hexdigest()!=checkpoint_sha:
    raise RuntimeError('Raw readback checkpoint không khớp; không tuyên bố lưu thành công.')
checkpoint_ref=checkpoint_result
receipt={'checkpoint_id':checkpoint_result['id'],'checkpoint_sha256':checkpoint_sha,'progress':progress,'this_work_span_only':span,'training_started':False,'TRAINING_DATASET_READY':False,'full_archive_token_linkage_split_audit_complete':False}
(ROOT/'last_data_only_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print(receipt)


{'checkpoint_id': '1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr', 'checkpoint_sha256': '7b9babd8710bb598c2ec051073dc0d34de888686de97e3610cc10f1931b4a626', 'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'training_started': False, 'TRAINING_DATASET_READY': False, 'full_archive_token_linkage_split_audit_complete': False}


Runtime mất kết nối: chạy lại để tải checkpoint đúng ID/hash và tiếp nối. stream_exhausted chỉ chứng minh đọc hết member của split/recipe này; không tự chứng minh mọi nguồn liên kết đúng hoặc các split độc lập. Cần source–chat/group/question/canonical audit theo queue trước công bố sẵn sàng train. Không đổi split hoặc bỏ bài hợp lệ để qua gate. Notebook chưa được chạy trên Colab trong phiên chuẩn bị này.


In [ ]:
# DATA ONLY: deepen source/hash/link/split audit over the entire published recipe.
# Does not invoke model weights, Trainer, GPU, or training.
import io,zipfile,importlib,os
from datetime import datetime,timezone
state_all,recipe_all,pointer_all=latest_controls(reader)
assert state_all['accepted_cases']==1629852 and state_all['training_chat_rows']==6502064
audit_bundle=state_all['sgk_finish_runs'][-1]['bundle']
audit_bytes=reader.read(audit_bundle['id'],audit_bundle['bytes'])
assert hashlib.sha256(audit_bytes).hexdigest()==audit_bundle['sha256']
audit_dir=ROOT/'full_audit_code'
audit_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(audit_bytes)) as za:
    ma=json.loads(za.read('manifest.json'))
    for filename in ['exhaustive_colab_loader.py','math_data_portable_resume.py','math_data_archive_audit.py','math_data_candidate_stream.py','math_data_full_audit.py']:
        member='code/'+filename
        payload=za.read(member);pin=ma['members'][member]
        assert len(payload)==pin['bytes'] and hashlib.sha256(payload).hexdigest()==pin['sha256']
        (audit_dir/filename).write_bytes(payload)
sys.path.insert(0,str(audit_dir))
import math_data_full_audit as full_audit
AUDIT_ID=full_audit.plan_identity(recipe_all)
AUDIT_NAME=f'MathVisionKid_FULL_DATA_AUDIT_{AUDIT_ID[:24]}.sqlite'
AUDIT_DB=ROOT/AUDIT_NAME
audit_query=f"'{FOLDER_ID}' in parents and name = '{AUDIT_NAME}' and trashed = false"
audit_matches=drive_api.files().list(q=audit_query,fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(audit_matches)>1:
    raise RuntimeError('Ambiguous audit checkpoint IDs; do not reset or overwrite.')
audit_checkpoint_ref=audit_matches[0] if audit_matches else None
if not AUDIT_DB.exists() and audit_checkpoint_ref:
    prior_audit=reader.read(audit_checkpoint_ref['id'],int(audit_checkpoint_ref['size']))
    if hashlib.sha256(prior_audit).hexdigest()!=audit_checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Audit checkpoint hash mismatch; preserve evidence.')
    AUDIT_DB.write_bytes(prior_audit)
audit_cache=LazyDriveCache(reader,recipe_all,ROOT/'full_audit_payload_cache')
auditor=full_audit.FullArchiveAudit(AUDIT_DB,recipe_all,tokenizer)
entry=auditor.next_entry()
try:
    if entry is not None:
        result=auditor.consume(entry,audit_cache[entry['sha256']],max_rows=1000)
    else:
        result={'already_complete':True}
    audit_report=auditor.report()
finally:
    auditor.close()
print({'purpose':'DATA_PREPARATION_ONLY','runtime_accelerator':os.environ.get('COLAB_ACCELERATOR','none'),'old_20k_token_checkpoint_preserved':checkpoint_result['id'],'latest_recipe':pointer_all['sha256'],'first_deep_audit_span':result,'audit':audit_report,'training_started':False})


{'purpose': 'DATA_PREPARATION_ONLY', 'runtime_accelerator': 'none', 'old_20k_token_checkpoint_preserved': '1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr', 'latest_recipe': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'first_deep_audit_span': {'member_key': 'c9fa305a1bf0904b794e243ad39409d5e5616e081777e3431349365955e209d5', 'rows': 1000, 'processed': 1000, 'bounded_pause': True}, 'audit': {'schema': 'MathVisionKid.full_current_archive_audit.v1', 'identity': 'e9609c3b8626b9234f884a2fdb8372166e0fd0ffbb3a1ff210d63a4bb93715b5', 'source_versions_indexed': 1000, 'unique_chat_rows_processed': 0, 'actual_rows_by_split': {}, 'source_hash_origins': {}, 'physical_chat_occurrences_processed': 0, 'exact_duplicate_references': 0, 'members_complete': 0, 'members_expected': 247, 'all_declared_members_scanned': False, 'max_sequence_tokens': None, 'max_completion_tokens': None, 'source_linkage_unverified_rows': 0, 'incomplete_four_mode_cases': 0, 'issues': {'SOURCE_SPLIT_MISSING_OR_UNKNOWN': 1

In [ ]:
# Persist audit evidence even when a format issue needs adjudication; never promote it.
from googleapiclient.http import MediaFileUpload
import math_data_archive_audit as archive_audit
with archive_audit.payload(entry,audit_cache[entry['sha256']]) as audit_source_stream:
    first_source=json.loads(next(audit_source_stream))
source_format_observation={'member':entry.get('member'),'first_source_id':first_source.get('sample_id'),'explicit_split':first_source.get('split'),'keys':sorted(first_source),'source_version_count':audit_report['source_versions_indexed'],'finding':'Source records without an explicit known split remain flagged; no default train assignment or gate bypass applied.'}
audit_sha=hashlib.sha256(AUDIT_DB.read_bytes()).hexdigest()
audit_properties={'data_sha256':audit_sha,'recipe_identity':AUDIT_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
if audit_checkpoint_ref:
    fresh_audit=drive_api.files().get(fileId=audit_checkpoint_ref['id'],fields='id,appProperties',supportsAllDrives=True).execute()
    if fresh_audit.get('appProperties',{}).get('data_sha256')!=audit_checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Audit checkpoint changed on Drive; retain local evidence, do not overwrite.')
    audit_saved=drive_api.files().update(fileId=audit_checkpoint_ref['id'],body={'appProperties':audit_properties},media_body=MediaFileUpload(str(AUDIT_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
else:
    audit_saved=drive_api.files().create(body={'name':AUDIT_NAME,'parents':[FOLDER_ID],'appProperties':audit_properties},media_body=MediaFileUpload(str(AUDIT_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
audit_readback=reader.read(audit_saved['id'],int(audit_saved['size']))
assert hashlib.sha256(audit_readback).hexdigest()==audit_sha
audit_checkpoint_ref=audit_saved
audit_receipt={'checkpoint_id':audit_saved['id'],'checkpoint_sha256':audit_sha,'checkpoint_bytes':len(audit_readback),'all_bytes_hash_readback_verified':True,'latest_recipe_sha256':pointer_all['sha256'],'source_format_observation':source_format_observation,'report':audit_report,'old_token_checkpoint_id':checkpoint_result['id'],'training_started':False,'TRAINING_DATASET_READY':False}
(ROOT/'last_full_audit_receipt.json').write_text(json.dumps(audit_receipt,ensure_ascii=False,indent=2),encoding='utf8')
print(audit_receipt)


{'checkpoint_id': '18vXFDzqBNxLSawKWvAUhCgYZPW2fNEb3', 'checkpoint_sha256': '33d3a07ef2bc51ded1cc1958c869c64f8a88999803e57c908ea69c9b3ca47224', 'checkpoint_bytes': 1830912, 'all_bytes_hash_readback_verified': True, 'latest_recipe_sha256': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'source_format_observation': {'member': 'source.jsonl.gz', 'first_source_id': 'exhaustive_v1_div_0_100', 'explicit_split': 'train_candidate', 'keys': ['acceptance_status', 'can_solve', 'canonical_signatures', 'case_group_id', 'case_kind', 'coverage', 'curriculum_assignment', 'evaluation_eligible', 'expected_answer', 'givens', 'grade', 'hints', 'initial_guidance', 'operation_metadata', 'problem', 'provenance', 'quality_status', 'quality_validation', 'requested_quantity', 'sample_id', 'split', 'student_work', 'topic', 'training_eligible', 'training_exclusions', 'worked_steps'], 'source_version_count': 1000, 'finding': 'Source records without an explicit known split remain flagged; no de

## Mốc kiểm dữ liệu qua MCP — 21/10/2026
Đã xác thực Drive trên CPU và kiểm tokenizer đủ20.000chat, checkpointđãlưu Drive ID1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr. Kiểm sâu v1 index1.000source và ghi rõ nhãnnguồn train_candidate khác nhãnchat train; checkpointv1đãlưu riêng ID18vXFDzqBNxLSawKWvAUhCgYZPW2fNEb3,khôngxóa. Cần kiểm tươngthíchschema bằngauditorv2dẫnxuấttrướckhikiểm toànbộ247members. Chưa train/GPU/weights;TRAINING_DATASET_READY=false. Toànbộ71bàimới/284chatđãkiểm riêng vàcôngbốtrênDrive; không dùng20Kđểthay thếtoànkho.

In [ ]:
# DATA ONLY v2: explicit source-pool label validation, no dataset rewriting.
import base64,importlib
audit_code_v2=base64.b64decode("IiIiRGVyaXZlZCB2MiBjdXJyZW50LWNhbmRpZGF0ZSBDUFUgYXVkaXQuIE5vIHRyYWluaW5nLCB3ZWlnaHRzLCBuZXR3b3JrIG9yIGRlbGV0aW9ucy4NCg0KVXNlIG9ubHkgaW4gYSBjbG91ZC9ydW50aW1lIGRpcmVjdG9yeSBmb3IgdGhlIGNvcnB1cy4gQWNjb3VudCBhbmQgbW91bnRlZCBwYXRocw0KYXJlIG91dHNpZGUgY29udGVudCBpZGVudGl0eS4gRXZlcnkgZGVjbGFyZWQgc291cmNlIG1lbWJlciBpcyBpbmRleGVkIGJlZm9yZQ0KY2hhdHM7IGFsbCBleHBsaWNpdCBzcGxpdHMgYXJlIGNoZWNrZWQgdG9nZXRoZXIuIE9yaWdpbmFsIGJ5dGVzIG5ldmVyIGNoYW5nZS4NCiIiIg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyDQppbXBvcnQganNvbixzcWxpdGUzLGhhc2hsaWINCmltcG9ydCBtYXRoX2RhdGFfYXJjaGl2ZV9hdWRpdCBhcyBBDQppbXBvcnQgZXhoYXVzdGl2ZV9jb2xhYl9sb2FkZXIgYXMgTA0KZnJvbSBtYXRoX2RhdGFfY2FuZGlkYXRlX3N0cmVhbSBpbXBvcnQgbWVtYmVyX2tleSxjb250ZW50X2lkZW50aXR5LFNQTElUUyxfY2hlY2tfZW50cnkNCmZyb20gbWF0aF9kYXRhX3BvcnRhYmxlX3Jlc3VtZSBpbXBvcnQgY2Fub25pY2FsLGRpZ2VzdA0KDQpERVJJVkFUSU9OX1BST1ZFTkFOQ0UgPSB7DQogICAgJ3BhcmVudF9maWxlJzogJ21hdGhfZGF0YV9mdWxsX2F1ZGl0LnB5JywNCiAgICAncGFyZW50X3NoYTI1Nic6ICc2NDIzNDM5MzdjNTRjMWM3NjFlYzM4NzA4ZDc2ZmI0NmUyNjE2NTU2M2FiMjA0NWU5YmM2YzQ1YWRmMDYyNGExJywNCiAgICAnc291cmNlX2xhYmVsX2V2aWRlbmNlX2ZpbGUnOiAnZXhoYXVzdGl2ZV9wcmltYXJ5X21hdGgucHknLA0KICAgICdzb3VyY2VfbGFiZWxfZXZpZGVuY2Vfc2hhMjU2JzogJ2QwYWYxZTRhZGY3M2E1NWI4NDE2NzU3ZWU0M2RlOTAwNDc0ZjM3OGFmN2ZmODI5MGZmZmRkMGIwMTVhY2I4YjInLA0KICAgICdzb3VyY2VfbGFiZWxfZXZpZGVuY2UnOiAnbWFrZV9yb3cgcmV0dXJucyB0cmFpbl9jYW5kaWRhdGU7IGNoYXRfcmVjb3JkcyBleHBsaWNpdGx5IHJldHVybnMgdHJhaW4uJywNCiAgICAncmF3X3NvdXJjZV9yZXdyaXR0ZW4nOiBGYWxzZSwNCiAgICAnY2FuZGlkYXRlX3Bvb2xfcHJvbW90ZWRfdG9fdHJhaW4nOiBGYWxzZSwNCiAgICAndjFfY2hlY2twb2ludF9hbmRfaXNzdWVzX3ByZXNlcnZlZCc6IFRydWUsDQp9DQpTT1VSQ0VfTEFCRUxTID0gZnJvemVuc2V0KFNQTElUUykgfCB7J3RyYWluX2NhbmRpZGF0ZSd9DQoNCmRlZiBzb3VyY2VfY2hhdF9zcGxpdF9jb21wYXRpYmxlKHNvdXJjZV9sYWJlbCwgY2hhdF9zcGxpdCk6DQogICAgIyBUaGlzIGlzIGEgYnVpbGRlci1iYWNrZWQgY29tcGF0aWJpbGl0eSBjaGVjaywgbm90IGEgc291cmNlIHNwbGl0IHJld3JpdGUuDQogICAgIyBVbmtub3duL21pc3Npbmcgc291cmNlIGxhYmVscyBzdGF5IHVudmVyaWZpZWQ7IGNoYXRzIHN0aWxsIHJlcXVpcmUgU1BMSVRTLg0KICAgIHJldHVybiAoc291cmNlX2xhYmVsID09ICd0cmFpbl9jYW5kaWRhdGUnIGFuZCBjaGF0X3NwbGl0ID09ICd0cmFpbicpIG9yICgNCiAgICAgICAgc291cmNlX2xhYmVsIGluIFNQTElUUyBhbmQgc291cmNlX2xhYmVsID09IGNoYXRfc3BsaXQpDQoNCmRlZiByYXdoYXNoKGxpbmUpOnJldHVybiBoYXNobGliLnNoYTI1NihsaW5lLnJlbW92ZXN1ZmZpeCgnXG4nKS5yZW1vdmVzdWZmaXgoJ1xyJykuZW5jb2RlKCd1dGY4JykpLmhleGRpZ2VzdCgpDQpkZWYgcGxhbl9pZGVudGl0eShyZWNpcGUpOg0KICAgIHJldHVybiBkaWdlc3QoZGljdChzY2hlbWE9J2Z1bGxfY3VycmVudF9hcmNoaXZlX2F1ZGl0LnYyJyxzb3VyY2VfbWVtYmVycz1bY29udGVudF9pZGVudGl0eSh4KSBmb3IgeCBpbiByZWNpcGVbJ3NvdXJjZV9tZW1iZXJzJ11dLGNoYXRfbWVtYmVycz1bY29udGVudF9pZGVudGl0eSh4KSBmb3IgeCBpbiByZWNpcGVbJ2NoYXRfbWVtYmVycyddXSx0b2tlbml6ZXJfcmV2aXNpb249TC5SRVZJU0lPTixzb3VyY2VfbGFiZWxfcG9saWN5PSdidWlsZGVyX3Bpbm5lZF9jYW5kaWRhdGVfcG9vbF90b190cmFpbl9jaGF0X29ubHkudjEnKSkNCmNsYXNzIEZ1bGxBcmNoaXZlQXVkaXQ6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsZGF0YWJhc2UscmVjaXBlLHRva2VuaXplcixzZXF1ZW5jZV9saW1pdD00MDk2MCk6DQogICAgICAgIGlmIHJlY2lwZS5nZXQoJ3BlbmRpbmdfZGlzY292ZXJ5Jyk6cmFpc2UgVmFsdWVFcnJvcignVW5yZXNvbHZlZCBtZW1iZXIgZGlzY292ZXJ5JykNCiAgICAgICAgaWYgdG9rZW5pemVyIGlzIE5vbmU6cmFpc2UgVmFsdWVFcnJvcignQWN0dWFsIHRva2VuaXplciByZXF1aXJlZCwgbm8gZnVsbCB0b2tlbiBhdWRpdCB3aXRob3V0IGl0JykNCiAgICAgICAgc2VsZi5yZWNpcGU9cmVjaXBlO3NlbGYudG9rZW5pemVyPXRva2VuaXplcjtzZWxmLmxpbWl0PXNlcXVlbmNlX2xpbWl0DQogICAgICAgIHNlbGYuZW50cmllcz1yZWNpcGVbJ3NvdXJjZV9tZW1iZXJzJ10rcmVjaXBlWydjaGF0X21lbWJlcnMnXQ0KICAgICAgICBmb3IgZSBpbiBzZWxmLmVudHJpZXM6X2NoZWNrX2VudHJ5KGUpDQogICAgICAgIHNlbGYua2V5cz1bbWVtYmVyX2tleShlKSBmb3IgZSBpbiBzZWxmLmVudHJpZXNdDQogICAgICAgIGlmIGxlbihzZXQoc2VsZi5rZXlzKSkhPWxlbihzZWxmLmtleXMpOnJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBleGFjdCBtZW1iZXIga2V5JykNCiAgICAgICAgc2VsZi5pZGVudGl0eT1wbGFuX2lkZW50aXR5KHJlY2lwZSk7c2VsZi5kYj1zcWxpdGUzLmNvbm5lY3QoZGF0YWJhc2UpDQogICAgICAgICMgUmVmdXNlIGFuIG9sZCBpZGVudGl0eSBiZWZvcmUgYW55IHNjaGVtYSB3cml0ZSwgcHJlc2VydmluZyBpdHMgaXNzdWUgbG9nLg0KICAgICAgICBpZiBzZWxmLmRiLmV4ZWN1dGUoIlNFTEVDVCAxIEZST00gc3FsaXRlX21hc3RlciBXSEVSRSB0eXBlPSd0YWJsZScgQU5EIG5hbWU9J2NvbmZpZyciKS5mZXRjaG9uZSgpOg0KICAgICAgICAgICAgcHJpb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHksc2VxdWVuY2VfbGltaXQgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgICAgICBpZiBwcmlvciBhbmQgcHJpb3IhPShzZWxmLmlkZW50aXR5LHNlbGYubGltaXQpOg0KICAgICAgICAgICAgICAgIHNlbGYuZGIuY2xvc2UoKTtyYWlzZSBWYWx1ZUVycm9yKCdEaWZmZXJlbnQgaW1tdXRhYmxlIGF1ZGl0IGlkZW50aXR5OyBwcmVzZXJ2ZSBwcmlvciBjaGVja3BvaW50JykNCiAgICAgICAgc2VsZi5kYi5leGVjdXRlc2NyaXB0KCIiIg0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBjb25maWcoaWRlbnRpdHkgVEVYVCBQUklNQVJZIEtFWSwgc2VxdWVuY2VfbGltaXQgSU5URUdFUik7DQogICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIG1lbWJlcnMoaWQgVEVYVCBQUklNQVJZIEtFWSwgcm9sZSBURVhULCBuZXh0X2xpbmUgSU5URUdFUiwgY29tcGxldGUgSU5URUdFUiwgZXhwZWN0ZWRfcm93cyBJTlRFR0VSKTsNCiAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgc291cmNlX3ZlcnNpb25zKGlkIFRFWFQscmF3X3NoYSBURVhULGNhbm9uaWNhbF9zaGEgVEVYVCxwcm9qZWN0aW9uIFRFWFQsc2lnbmF0dXJlcyBURVhULHNvdXJjZV9zcGxpdCBURVhULFBSSU1BUlkgS0VZKGlkLHJhd19zaGEpKTsNCiAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgc2FtcGxlcyhpZCBURVhUIFBSSU1BUlkgS0VZLGNhbm9uaWNhbF9zaGEgVEVYVCxjYXNlX2lkIFRFWFQsbW9kZSBURVhULHNwbGl0IFRFWFQsc291cmNlX3Jhd19zaGEgVEVYVCxsaW5rYWdlIElOVEVHRVIscmF3X2hhc2hfb3JpZ2luIFRFWFQsc2VxdWVuY2VfdG9rZW5zIElOVEVHRVIsY29tcGxldGlvbl90b2tlbnMgSU5URUdFUiwgVU5JUVVFKGNhc2VfaWQsbW9kZSkpOw0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBvY2N1cnJlbmNlcyhtZW1iZXIgVEVYVCxsaW5lIElOVEVHRVIsc2FtcGxlX2lkIFRFWFQscmF3X3NoYSBURVhULGtpbmQgVEVYVCxQUklNQVJZIEtFWShtZW1iZXIsbGluZSkpOw0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBzcGxpdF9rZXlzKGtpbmQgVEVYVCxrZXkgVEVYVCxzcGxpdCBURVhULFBSSU1BUlkgS0VZKGtpbmQsa2V5KSk7DQogICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIGlzc3VlcyhtZW1iZXIgVEVYVCxsaW5lIElOVEVHRVIsa2luZCBURVhULGRldGFpbCBURVhULFBSSU1BUlkgS0VZKG1lbWJlcixsaW5lLGtpbmQsZGV0YWlsKSk7DQogICAgICAgICIiIikNCiAgICAgICAgcHJpb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHksc2VxdWVuY2VfbGltaXQgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgIGlmIHByaW9yIGFuZCBwcmlvciE9KHNlbGYuaWRlbnRpdHksc2VsZi5saW1pdCk6c2VsZi5kYi5jbG9zZSgpO3JhaXNlIFZhbHVlRXJyb3IoJ0RpZmZlcmVudCBpbW11dGFibGUgYXVkaXQgaWRlbnRpdHk7IHByZXNlcnZlIHByaW9yIGNoZWNrcG9pbnQnKQ0KICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBPUiBJR05PUkUgSU5UTyBjb25maWcgVkFMVUVTICg/LD8pJywoc2VsZi5pZGVudGl0eSxzZWxmLmxpbWl0KSk7c2VsZi5kYi5jb21taXQoKQ0KICAgIGRlZiBpc3N1ZShzZWxmLGtleSxsaW5lLGtpbmQsZGV0YWlsKToNCiAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdJTlNFUlQgT1IgSUdOT1JFIElOVE8gaXNzdWVzIFZBTFVFUyAoPyw/LD8sPyknLChrZXksbGluZSxraW5kLHN0cihkZXRhaWwpKSkNCiAgICBkZWYgX3NvdXJjZShzZWxmLHJvdyxrZXksbGluZSx0ZXh0KToNCiAgICAgICAgaWYgcm93LmdldCgnc3BsaXQnKSBub3QgaW4gU09VUkNFX0xBQkVMUzpzZWxmLmlzc3VlKGtleSxsaW5lLCdTT1VSQ0VfU1BMSVRfTUlTU0lOR19PUl9VTktOT1dOJyxyb3cuZ2V0KCdzYW1wbGVfaWQnKSkNCiAgICAgICAgcHJvamVjdGlvbj1BLnNvdXJjZV9wcm9qZWN0aW9uKHJvdyk7aD1yYXdoYXNoKHRleHQpDQogICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIHNvdXJjZV92ZXJzaW9ucyBWQUxVRVMgKD8sPyw/LD8sPyw/KScsKHJvd1snc2FtcGxlX2lkJ10saCxkaWdlc3Qocm93KSxjYW5vbmljYWwocHJvamVjdGlvbiksY2Fub25pY2FsKEwuc291cmNlX3NpZ25hdHVyZXMocm93KSkscm93LmdldCgnc3BsaXQnKSkpDQogICAgZGVmIF9jaGF0KHNlbGYscm93LGtleSxsaW5lLHRleHQpOg0KICAgICAgICBpZiByb3cuZ2V0KCdzcGxpdCcpIG5vdCBpbiBTUExJVFM6cmFpc2UgVmFsdWVFcnJvcignRXhwbGljaXQga25vd24gY2hhdCBzcGxpdCByZXF1aXJlZDsgbm8gZGVmYXVsdCB0cmFpbicpDQogICAgICAgIEwudmFsaWRhdGVfY2hhdChyb3cpO3NpZD1MLmNhc2VfaWQocm93KTtoPWRpZ2VzdChyb3cpDQogICAgICAgIHByaW9yPXNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIGNhbm9uaWNhbF9zaGEgRlJPTSBzYW1wbGVzIFdIRVJFIGlkPT8nLChyb3dbJ3NhbXBsZV9pZCddLCkpLmZldGNob25lKCkNCiAgICAgICAgaWYgcHJpb3I6DQogICAgICAgICAgICBraW5kPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJyBpZiBwcmlvclswXT09aCBlbHNlICdDT05GTElDVElOR19TQU1QTEVfSUQnDQogICAgICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBJTlRPIG9jY3VycmVuY2VzIFZBTFVFUyAoPyw/LD8sPyw/KScsKGtleSxsaW5lLHJvd1snc2FtcGxlX2lkJ10scmF3aGFzaCh0ZXh0KSxraW5kKSkNCiAgICAgICAgICAgIGlmIGtpbmQhPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJzpzZWxmLmlzc3VlKGtleSxsaW5lLGtpbmQscm93WydzYW1wbGVfaWQnXSkNCiAgICAgICAgICAgIHJldHVybg0KICAgICAgICBjYW5kaWRhdGVzPXNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIHJhd19zaGEscHJvamVjdGlvbixzaWduYXR1cmVzLHNvdXJjZV9zcGxpdCBGUk9NIHNvdXJjZV92ZXJzaW9ucyBXSEVSRSBpZD0/Jywoc2lkLCkpLmZldGNoYWxsKCkNCiAgICAgICAgb3JpZ2luYWw9cm93LmdldCgnc291cmNlX3Jvd19zaGEyNTYnKQ0KICAgICAgICBpZiBvcmlnaW5hbDoNCiAgICAgICAgICAgIGNhbmRpZGF0ZXM9W3MgZm9yIHMgaW4gY2FuZGlkYXRlcyBpZiBzWzBdPT1vcmlnaW5hbF0NCiAgICAgICAgICAgIGlmIG5vdCBjYW5kaWRhdGVzOnNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9SQVdfSEFTSF9OT1RfRk9VTkQnLHNpZCkNCiAgICAgICAgdGFyZ2V0X21hdGNoZXM9W3MgZm9yIHMgaW4gY2FuZGlkYXRlcyBpZiBub3QgQS5saW5rYWdlX2Vycm9ycyhyb3csanNvbi5sb2FkcyhzWzFdKSldDQogICAgICAgIG1hdGNoZXM9W3MgZm9yIHMgaW4gdGFyZ2V0X21hdGNoZXMgaWYgc291cmNlX2NoYXRfc3BsaXRfY29tcGF0aWJsZShzWzNdLHJvd1snc3BsaXQnXSldDQogICAgICAgIGlmIHRhcmdldF9tYXRjaGVzIGFuZCBub3QgbWF0Y2hlczoNCiAgICAgICAgICAgIHNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9DSEFUX1NQTElUX01JU01BVENIJyxjYW5vbmljYWwoZGljdChjYXNlX2lkPXNpZCxzb3VyY2VfbGFiZWxzPXNvcnRlZCh7c3RyKHNbM10pIGZvciBzIGluIHRhcmdldF9tYXRjaGVzfSksY2hhdF9zcGxpdD1yb3dbJ3NwbGl0J10pKSkNCiAgICAgICAgaWYgbGVuKG1hdGNoZXMpPT0xOg0KICAgICAgICAgICAgc291cmNlPW1hdGNoZXNbMF07b3JpZ2luPSdPUklHSU5BTF9SQVdfSEFTSF9WRVJJRklFRCcgaWYgb3JpZ2luYWwgZWxzZSAnUkVBRF9USU1FX0RFUklWRURfQllfRlVMTF9QUk9NUFRfVEFSR0VUX0xJTktBR0UnDQogICAgICAgIGVsaWYgbGVuKG1hdGNoZXMpPjE6DQogICAgICAgICAgICBzZWxmLmlzc3VlKGtleSxsaW5lLCdBTUJJR1VPVVNfU09VUkNFX1JBV19IQVNIJyxzaWQpO3NvdXJjZT1Ob25lO29yaWdpbj0nVU5WRVJJRklFRCcNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9DSEFUX0xJTktBR0VfVU5WRVJJRklFRCcsc2lkKTtzb3VyY2U9Tm9uZTtvcmlnaW49J1VOVkVSSUZJRUQnDQogICAgICAgIGVuY29kZWQ9TC50b2tlbml6ZV9hc3Npc3RhbnRfb25seShyb3csc2VsZi50b2tlbml6ZXIsc2VsZi5saW1pdCkNCiAgICAgICAgcmF3X3NoYT1zb3VyY2VbMF0gaWYgc291cmNlIGVsc2UgTm9uZTtzaWduYXR1cmVzPWpzb24ubG9hZHMoc291cmNlWzJdKSBpZiBzb3VyY2UgZWxzZSByb3cuZ2V0KCdjYW5vbmljYWxfc2lnbmF0dXJlcycsW10pDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIElOVE8gc2FtcGxlcyBWQUxVRVMgKD8sPyw/LD8sPyw/LD8sPyw/LD8pJywocm93WydzYW1wbGVfaWQnXSxoLHNpZCxyb3dbJ21vZGUnXSxyb3dbJ3NwbGl0J10scmF3X3NoYSxpbnQoc291cmNlIGlzIG5vdCBOb25lKSxvcmlnaW4sbGVuKGVuY29kZWRbJ2lucHV0X2lkcyddKSxlbmNvZGVkWydjb21wbGV0aW9uX3Rva2VucyddKSkNCiAgICAgICAgZXhjZXB0IHNxbGl0ZTMuSW50ZWdyaXR5RXJyb3I6DQogICAgICAgICAgICBzZWxmLmlzc3VlKGtleSxsaW5lLCdDT05GTElDVElOR19DQVNFX01PREUnLHNpZCsnOicrcm93Wydtb2RlJ10pO3JldHVybg0KICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBJTlRPIG9jY3VycmVuY2VzIFZBTFVFUyAoPyw/LD8sPyw/KScsKGtleSxsaW5lLHJvd1snc2FtcGxlX2lkJ10scmF3aGFzaCh0ZXh0KSwnQ1VSUkVOVF9DQU5ESURBVEUnKSkNCiAgICAgICAga2V5cz1bKCdncm91cCcscm93WydjYXNlX2dyb3VwX2lkJ10pLCgncXVlc3Rpb24nLEwucXVlc3Rpb25fa2V5KHJvdykpXQ0KICAgICAgICBrZXlzLmV4dGVuZCgoJ2Nhbm9uaWNhbCcscykgZm9yIHMgaW4gc2lnbmF0dXJlcykNCiAgICAgICAgZm9yIGtpbmQsdmFsdWUgaW4ga2V5czoNCiAgICAgICAgICAgIHA9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1Qgc3BsaXQgRlJPTSBzcGxpdF9rZXlzIFdIRVJFIGtpbmQ9PyBBTkQga2V5PT8nLChraW5kLHZhbHVlKSkuZmV0Y2hvbmUoKQ0KICAgICAgICAgICAgaWYgcCBhbmQgcFswXSE9cm93WydzcGxpdCddOnNlbGYuaXNzdWUoa2V5LGxpbmUsJ0NST1NTX1NQTElUXycra2luZC51cHBlcigpLHZhbHVlKQ0KICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdJTlNFUlQgT1IgSUdOT1JFIElOVE8gc3BsaXRfa2V5cyBWQUxVRVMgKD8sPyw/KScsKGtpbmQsdmFsdWUscm93WydzcGxpdCddKSkNCiAgICBkZWYgY29uc3VtZShzZWxmLGVudHJ5LHBhdGgsbWF4X3Jvd3M9MjAwMDAsY29tbWl0X2V2ZXJ5PTI1Nik6DQogICAgICAgIGlmIG1heF9yb3dzIGlzIG5vdCBOb25lIGFuZCBtYXhfcm93czw9MCBvciBjb21taXRfZXZlcnk8PTA6cmFpc2UgVmFsdWVFcnJvcignUG9zaXRpdmUgd29yayBib3VuZHMgcmVxdWlyZWQnKQ0KICAgICAgICBrZXk9bWVtYmVyX2tleShlbnRyeSkNCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWxmLmtleXM6cmFpc2UgVmFsdWVFcnJvcignTWVtYmVyIG5vdCBkZWNsYXJlZCBieSBleGFjdCByZWNpcGUnKQ0KICAgICAgICBpZiBlbnRyeVsncm9sZSddPT0nY2FuZGlkYXRlX2NoYXQnOg0KICAgICAgICAgICAgbWlzc2luZz1bbWVtYmVyX2tleShlKSBmb3IgZSBpbiBzZWxmLnJlY2lwZVsnc291cmNlX21lbWJlcnMnXSBpZiBub3Qgc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY29tcGxldGUgRlJPTSBtZW1iZXJzIFdIRVJFIGlkPT8nLChtZW1iZXJfa2V5KGUpLCkpLmZldGNob25lKCkgb3Igbm90IHNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIGNvbXBsZXRlIEZST00gbWVtYmVycyBXSEVSRSBpZD0/JywobWVtYmVyX2tleShlKSwpKS5mZXRjaG9uZSgpWzBdXQ0KICAgICAgICAgICAgaWYgbWlzc2luZzpyYWlzZSBWYWx1ZUVycm9yKCdJbmRleCBldmVyeSBkZWNsYXJlZCBzb3VyY2UgbWVtYmVyIGJlZm9yZSBhbnkgY2hhdHMnKQ0KICAgICAgICBjdXJzb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgbmV4dF9saW5lLGNvbXBsZXRlIEZST00gbWVtYmVycyBXSEVSRSBpZD0/Jywoa2V5LCkpLmZldGNob25lKCkNCiAgICAgICAgaWYgY3Vyc29yIGFuZCBjdXJzb3JbMV06cmV0dXJuIGRpY3QoYWxyZWFkeV9jb21wbGV0ZT1UcnVlLHJvd3M9Y3Vyc29yWzBdLG1lbWJlcl9rZXk9a2V5KQ0KICAgICAgICBzdGFydD1jdXJzb3JbMF0gaWYgY3Vyc29yIGVsc2UgMDtwcm9jZXNzZWQ9MDtjb3VudD0wDQogICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIG1lbWJlcnMgVkFMVUVTICg/LD8sMCwwLD8pJywoa2V5LGVudHJ5Wydyb2xlJ10sZW50cnlbJ3Jvd3MnXSkpO3NlbGYuZGIuY29tbWl0KCkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgd2l0aCBBLnBheWxvYWQoZW50cnkscGF0aCkgYXMgc3RyZWFtOg0KICAgICAgICAgICAgICAgIGZvciBjb3VudCx0ZXh0IGluIGVudW1lcmF0ZShzdHJlYW0sMSk6DQogICAgICAgICAgICAgICAgICAgIGlmIGNvdW50PD1zdGFydDpjb250aW51ZQ0KICAgICAgICAgICAgICAgICAgICBpZiBub3QgdGV4dC5zdHJpcCgpOnJhaXNlIFZhbHVlRXJyb3IoJ0JsYW5rIEpTT05MIHJvdycpDQogICAgICAgICAgICAgICAgICAgIHJvdz1qc29uLmxvYWRzKHRleHQpDQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdyxkaWN0KTpyYWlzZSBWYWx1ZUVycm9yKCdKU09OTCBvYmplY3QgcmVxdWlyZWQnKQ0KICAgICAgICAgICAgICAgICAgICBpZiBlbnRyeVsncm9sZSddPT0nY2FuZGlkYXRlX3NvdXJjZSc6c2VsZi5fc291cmNlKHJvdyxrZXksY291bnQsdGV4dCkNCiAgICAgICAgICAgICAgICAgICAgZWxzZTpzZWxmLl9jaGF0KHJvdyxrZXksY291bnQsdGV4dCkNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdVUERBVEUgbWVtYmVycyBTRVQgbmV4dF9saW5lPT8gV0hFUkUgaWQ9PycsKGNvdW50LGtleSkpO3Byb2Nlc3NlZCs9MQ0KICAgICAgICAgICAgICAgICAgICBpZiBwcm9jZXNzZWQlY29tbWl0X2V2ZXJ5PT0wOnNlbGYuZGIuY29tbWl0KCkNCiAgICAgICAgICAgICAgICAgICAgaWYgbWF4X3Jvd3MgaXMgbm90IE5vbmUgYW5kIHByb2Nlc3NlZD49bWF4X3Jvd3M6DQogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmRiLmNvbW1pdCgpO3JldHVybiBkaWN0KG1lbWJlcl9rZXk9a2V5LHJvd3M9Y291bnQscHJvY2Vzc2VkPXByb2Nlc3NlZCxib3VuZGVkX3BhdXNlPVRydWUpDQogICAgICAgICAgICBpZiBjb3VudCE9ZW50cnlbJ3Jvd3MnXSBvciBzdGFydD5jb3VudDpyYWlzZSBWYWx1ZUVycm9yKCdFeGFjdCBkZWNsYXJlZCBtZW1iZXIgcm93IGNvdW50IGRpZmZlcnMnKQ0KICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdVUERBVEUgbWVtYmVycyBTRVQgbmV4dF9saW5lPT8sY29tcGxldGU9MSBXSEVSRSBpZD0/JywoY291bnQsa2V5KSk7c2VsZi5kYi5jb21taXQoKQ0KICAgICAgICAgICAgcmV0dXJuIGRpY3QobWVtYmVyX2tleT1rZXkscm93cz1jb3VudCxwcm9jZXNzZWQ9cHJvY2Vzc2VkLGNvbXBsZXRlPVRydWUpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBzZWxmLmRiLnJvbGxiYWNrKCk7cmFpc2UNCiAgICBkZWYgbmV4dF9lbnRyeShzZWxmKToNCiAgICAgICAgZm9yIGUgaW4gc2VsZi5lbnRyaWVzOg0KICAgICAgICAgICAgZG9uZT1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb21wbGV0ZSBGUk9NIG1lbWJlcnMgV0hFUkUgaWQ9PycsKG1lbWJlcl9rZXkoZSksKSkuZmV0Y2hvbmUoKQ0KICAgICAgICAgICAgaWYgbm90IGRvbmUgb3Igbm90IGRvbmVbMF06cmV0dXJuIGUNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIGV4dGVuZF9yZWNpcGUoZGF0YWJhc2UscHJldmlvdXNfcmVjaXBlLG5leHRfcmVjaXBlKToNCiAgICAgICAgIiIiRXhwbGljaXQgYXBwZW5kLW9ubHkgbWlncmF0aW9uOyByZXRhaW4gYWxsIHByaW9yIHNvdXJjZS9jaGF0IGV2aWRlbmNlLiIiIg0KICAgICAgICBpZiBuZXh0X3JlY2lwZS5nZXQoJ3BlbmRpbmdfZGlzY292ZXJ5Jyk6cmFpc2UgVmFsdWVFcnJvcignVW5yZXNvbHZlZCBuZXcgbWVtYmVyIGRpc2NvdmVyeScpDQogICAgICAgIGZvciBmaWVsZCBpbiAoJ3NvdXJjZV9tZW1iZXJzJywnY2hhdF9tZW1iZXJzJyk6DQogICAgICAgICAgICBvbGQ9W2NvbnRlbnRfaWRlbnRpdHkoZSkgZm9yIGUgaW4gcHJldmlvdXNfcmVjaXBlW2ZpZWxkXV0NCiAgICAgICAgICAgIG5ldz1bY29udGVudF9pZGVudGl0eShlKSBmb3IgZSBpbiBuZXh0X3JlY2lwZVtmaWVsZF1dDQogICAgICAgICAgICBpZiBsZW4obmV3KTxsZW4ob2xkKSBvciBuZXdbOmxlbihvbGQpXSE9b2xkOnJhaXNlIFZhbHVlRXJyb3IoJ09sZCBzb3VyY2UvY2hhdCBwcmVmaXhlcyBtdXN0IHJlbWFpbiBpbW11dGFibGUnKQ0KICAgICAgICBiZWZvcmUsYWZ0ZXI9cGxhbl9pZGVudGl0eShwcmV2aW91c19yZWNpcGUpLHBsYW5faWRlbnRpdHkobmV4dF9yZWNpcGUpDQogICAgICAgIGRiPXNxbGl0ZTMuY29ubmVjdChkYXRhYmFzZSkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgd2l0aCBkYjoNCiAgICAgICAgICAgICAgICBwcmlvcj1kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHkgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgICAgICAgICAgaWYgbm90IHByaW9yIG9yIHByaW9yWzBdIT1iZWZvcmU6cmFpc2UgVmFsdWVFcnJvcignV3JvbmcgcHJpb3IgYXVkaXQgaWRlbnRpdHknKQ0KICAgICAgICAgICAgICAgIGRiLmV4ZWN1dGUoJ0NSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIHJlY2lwZV9oaXN0b3J5KHByZXZpb3VzX2lkZW50aXR5IFRFWFQsbmV4dF9pZGVudGl0eSBURVhULFBSSU1BUlkgS0VZKHByZXZpb3VzX2lkZW50aXR5LG5leHRfaWRlbnRpdHkpKScpDQogICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIHJlY2lwZV9oaXN0b3J5IFZBTFVFUyAoPyw/KScsKGJlZm9yZSxhZnRlcikpDQogICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnVVBEQVRFIGNvbmZpZyBTRVQgaWRlbnRpdHk9PycsKGFmdGVyLCkpDQogICAgICAgIGZpbmFsbHk6ZGIuY2xvc2UoKQ0KICAgICAgICByZXR1cm4gZGljdChwcmV2aW91c19pZGVudGl0eT1iZWZvcmUsbmV4dF9pZGVudGl0eT1hZnRlcixwcmlvcl9ldmlkZW5jZV9wcmVzZXJ2ZWQ9VHJ1ZSx0cmFpbmluZ19zdGFydGVkPUZhbHNlKQ0KICAgIGRlZiByZXBvcnQoc2VsZik6DQogICAgICAgIG4sbWF4c2VxLG1heGNvbXAsdW5saW5rZWQ9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKiksbWF4KHNlcXVlbmNlX3Rva2VucyksbWF4KGNvbXBsZXRpb25fdG9rZW5zKSxjb2FsZXNjZShzdW0obGlua2FnZT0wKSwwKSBGUk9NIHNhbXBsZXMnKS5mZXRjaG9uZSgpDQogICAgICAgIGNvbXBsZXRlZD1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb3VudCgqKSBGUk9NIG1lbWJlcnMgV0hFUkUgY29tcGxldGU9MScpLmZldGNob25lKClbMF0NCiAgICAgICAgaXNzdWVzPWRpY3Qoc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1Qga2luZCxjb3VudCgqKSBGUk9NIGlzc3VlcyBHUk9VUCBCWSBraW5kJykpDQogICAgICAgIG1pc3NpbmdfbW9kZXM9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSAoU0VMRUNUIGNhc2VfaWQgRlJPTSBzYW1wbGVzIEdST1VQIEJZIGNhc2VfaWQgSEFWSU5HIGNvdW50KCopIT00KScpLmZldGNob25lKClbMF0NCiAgICAgICAgc291cmNlcz1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb3VudCgqKSBGUk9NIHNvdXJjZV92ZXJzaW9ucycpLmZldGNob25lKClbMF0NCiAgICAgICAgYWxsX2RvbmU9Y29tcGxldGVkPT1sZW4oc2VsZi5lbnRyaWVzKQ0KICAgICAgICAjIENhbm9uaWNhbCBvdmVybGFwcyBuZWVkIGFuIGV4YWN0LWhhc2ggcG9saWN5IGFkanVkaWNhdGlvbiwgbmV2ZXIgYQ0KICAgICAgICAjIGJsYW5rZXQgYnlwYXNzLiBUaGlzIHNjYW5uZXIgZG9lcyBub3QgaW52ZW50IG93bmVyIGV4Y2VwdGlvbnMuDQogICAgICAgIGNsZWFuPW5vdCBpc3N1ZXMgYW5kIG5vdCB1bmxpbmtlZCBhbmQgbm90IG1pc3NpbmdfbW9kZXMNCiAgICAgICAgcmV0dXJuIGRpY3Qoc2NoZW1hPSdNYXRoVmlzaW9uS2lkLmZ1bGxfY3VycmVudF9hcmNoaXZlX2F1ZGl0LnYyJyxpZGVudGl0eT1zZWxmLmlkZW50aXR5LGRlcml2YXRpb25fcHJvdmVuYW5jZT1ERVJJVkFUSU9OX1BST1ZFTkFOQ0Usc291cmNlX3ZlcnNpb25zX2luZGV4ZWQ9c291cmNlcyxzb3VyY2VfbGFiZWxzX2luZGV4ZWQ9ZGljdChzZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb2FsZXNjZShzb3VyY2Vfc3BsaXQsXCc8bWlzc2luZz5cJyksY291bnQoKikgRlJPTSBzb3VyY2VfdmVyc2lvbnMgR1JPVVAgQlkgc291cmNlX3NwbGl0JykpLGNhbmRpZGF0ZV9zb3VyY2VfdHJhaW5fY2hhdF9saW5rcz1zZWxmLmRiLmV4ZWN1dGUoIlNFTEVDVCBjb3VudCgqKSBGUk9NIHNhbXBsZXMgcyBKT0lOIHNvdXJjZV92ZXJzaW9ucyB2IE9OIHMuY2FzZV9pZD12LmlkIEFORCBzLnNvdXJjZV9yYXdfc2hhPXYucmF3X3NoYSBXSEVSRSBzLmxpbmthZ2U9MSBBTkQgdi5zb3VyY2Vfc3BsaXQ9J3RyYWluX2NhbmRpZGF0ZScgQU5EIHMuc3BsaXQ9J3RyYWluJyIpLmZldGNob25lKClbMF0sdW5pcXVlX2NoYXRfcm93c19wcm9jZXNzZWQ9bixhY3R1YWxfcm93c19ieV9zcGxpdD1kaWN0KHNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIHNwbGl0LGNvdW50KCopIEZST00gc2FtcGxlcyBHUk9VUCBCWSBzcGxpdCcpKSxzb3VyY2VfaGFzaF9vcmlnaW5zPWRpY3Qoc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgcmF3X2hhc2hfb3JpZ2luLGNvdW50KCopIEZST00gc2FtcGxlcyBHUk9VUCBCWSByYXdfaGFzaF9vcmlnaW4nKSkscGh5c2ljYWxfY2hhdF9vY2N1cnJlbmNlc19wcm9jZXNzZWQ9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSBvY2N1cnJlbmNlcycpLmZldGNob25lKClbMF0sZXhhY3RfZHVwbGljYXRlX3JlZmVyZW5jZXM9c2VsZi5kYi5leGVjdXRlKCJTRUxFQ1QgY291bnQoKikgRlJPTSBvY2N1cnJlbmNlcyBXSEVSRSBraW5kPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJyIpLmZldGNob25lKClbMF0sbWVtYmVyc19jb21wbGV0ZT1jb21wbGV0ZWQsbWVtYmVyc19leHBlY3RlZD1sZW4oc2VsZi5lbnRyaWVzKSxhbGxfZGVjbGFyZWRfbWVtYmVyc19zY2FubmVkPWFsbF9kb25lLG1heF9zZXF1ZW5jZV90b2tlbnM9bWF4c2VxLG1heF9jb21wbGV0aW9uX3Rva2Vucz1tYXhjb21wLHNvdXJjZV9saW5rYWdlX3VudmVyaWZpZWRfcm93cz11bmxpbmtlZCxpbmNvbXBsZXRlX2ZvdXJfbW9kZV9jYXNlcz1taXNzaW5nX21vZGVzLGlzc3Vlcz1pc3N1ZXMsZnVsbF9hcmNoaXZlX3Rva2VuX2xpbmthZ2Vfc3BsaXRfYXVkaXRfY29tcGxldGU9Ym9vbChhbGxfZG9uZSBhbmQgY2xlYW4pLGFsbF9yb3dzX3NlbWFudGljYWxseV9odW1hbl9yZXZpZXdlZD1GYWxzZSx3aG9sZV9jdXJyaWN1bHVtX2NlcnRpZmllZD1GYWxzZSxUUkFJTklOR19EQVRBU0VUX1JFQURZPUZhbHNlLHRyYWluaW5nX3N0YXJ0ZWQ9RmFsc2UsZ3B1X3VzZWQ9RmFsc2UsbW9kZWxfd2VpZ2h0c19sb2FkZWQ9RmFsc2UpDQogICAgZGVmIGNsb3NlKHNlbGYpOnNlbGYuZGIuY2xvc2UoKQ0KDQo=").decode('utf8')
assert hashlib.sha256(audit_code_v2.encode()).hexdigest()=='81b1e3f04c5886f78282386ecb34d47ba4ba9da63edfec8856bab82034a093ee'
(audit_dir/'math_data_full_audit_v2.py').write_text(audit_code_v2,encoding='utf8')
import math_data_full_audit_v2 as audit_v2
assert audit_v2.plan_identity(recipe_all)!=AUDIT_ID
V2_ID=audit_v2.plan_identity(recipe_all)
V2_NAME=f'MathVisionKid_FULL_DATA_AUDIT_V2_{V2_ID[:24]}.sqlite'
V2_DB=ROOT/V2_NAME
v2_matches=drive_api.files().list(q=f"'{FOLDER_ID}' in parents and name = '{V2_NAME}' and trashed = false",fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(v2_matches)>1: raise RuntimeError('Ambiguous v2 checkpoints, preserve all')
v2_ref=v2_matches[0] if v2_matches else None
if not V2_DB.exists() and v2_ref:
 raw_v2=reader.read(v2_ref['id'],int(v2_ref['size']))
 assert hashlib.sha256(raw_v2).hexdigest()==v2_ref.get('appProperties',{}).get('data_sha256')
 V2_DB.write_bytes(raw_v2)
v2auditor=audit_v2.FullArchiveAudit(V2_DB,recipe_all,tokenizer)
v2entry=v2auditor.next_entry()
try:
 v2span=v2auditor.consume(v2entry,audit_cache[v2entry['sha256']],max_rows=1000) if v2entry else {'already_complete':True}
 v2report=v2auditor.report()
finally:
 v2auditor.close()
v2sha=hashlib.sha256(V2_DB.read_bytes()).hexdigest()
v2props={'data_sha256':v2sha,'recipe_identity':V2_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
if v2_ref:
 fv2=drive_api.files().get(fileId=v2_ref['id'],fields='appProperties',supportsAllDrives=True).execute()
 assert fv2.get('appProperties',{}).get('data_sha256')==v2_ref.get('appProperties',{}).get('data_sha256')
 v2_saved=drive_api.files().update(fileId=v2_ref['id'],body={'appProperties':v2props},media_body=MediaFileUpload(str(V2_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
else:
 v2_saved=drive_api.files().create(body={'name':V2_NAME,'parents':[FOLDER_ID],'appProperties':v2props},media_body=MediaFileUpload(str(V2_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
v2raw=reader.read(v2_saved['id'],int(v2_saved['size']))
assert hashlib.sha256(v2raw).hexdigest()==v2sha
v2_ref=v2_saved
v2receipt={'checkpoint_id':v2_saved['id'],'checkpoint_sha256':v2sha,'checkpoint_bytes':len(v2raw),'fresh_raw_readback_verified':True,'span':v2span,'report':v2report,'v1_checkpoint_preserved':audit_saved['id'],'old_token_checkpoint_preserved':checkpoint_result['id'],'runtime_accelerator':os.environ.get('COLAB_ACCELERATOR','none'),'training_started':False,'TRAINING_DATASET_READY':False}
(ROOT/'last_full_audit_v2_receipt.json').write_text(json.dumps(v2receipt,ensure_ascii=False,indent=2),encoding='utf8')
print(v2receipt)


{'checkpoint_id': '182zLHZU0XBpTfKm7ThJpfy37GgFJ7Ob2', 'checkpoint_sha256': '5ec824151d0acfec641a1ab15bc9f8ee2ea1995c1dedd95292c44741fb9f9e1b', 'checkpoint_bytes': 1540096, 'fresh_raw_readback_verified': True, 'span': {'member_key': 'c9fa305a1bf0904b794e243ad39409d5e5616e081777e3431349365955e209d5', 'rows': 1000, 'processed': 1000, 'bounded_pause': True}, 'report': {'schema': 'MathVisionKid.full_current_archive_audit.v2', 'identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'derivation_provenance': {'parent_file': 'math_data_full_audit.py', 'parent_sha256': '642343937c54c1c761ec38708d76fb46e26165563ab2045e9bc6c45adf0624a1', 'source_label_evidence_file': 'exhaustive_primary_math.py', 'source_label_evidence_sha256': 'd0af1e4adf73a55b8416757ee43de900474f378af7ff8290fffdd0b015acb8b2', 'source_label_evidence': 'make_row returns train_candidate; chat_records explicitly returns train.', 'raw_source_rewritten': False, 'candidate_pool_promoted_to_train': False, 'v1_ch

In [ ]:
# Finite CPU data audit over all247 declared source/chat members. No training.
import threading,time,traceback
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
if globals().get('full_audit_worker') is not None and full_audit_worker.is_alive():
    raise RuntimeError('An audit worker is already running; do not launch a duplicate.')
worker_status_path=ROOT/'full_audit_v2_worker_status.json'
full_audit_worker_state={'state':'STARTING','started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l','audit_identity':V2_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':v2_ref['id'],'training_started':False,'gpu_used':False,'model_weights_loaded':False}
def write_worker_status():
    full_audit_worker_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    temp_status=worker_status_path.with_suffix('.writing')
    temp_status.write_text(json.dumps(full_audit_worker_state,ensure_ascii=False,indent=2),encoding='utf8')
    temp_status.replace(worker_status_path)
def finite_full_data_audit_worker():
    work_api=build('drive','v3',credentials=credentials,cache_discovery=False)
    work_reader=GoogleDriveReader(work_api)
    work_cache=LazyDriveCache(work_reader,recipe_all,ROOT/'full_audit_worker_payload_cache')
    work_auditor=None
    checkpoint=dict(v2_ref)
    last_save=0.0
    def save_work_checkpoint():
        nonlocal checkpoint,last_save
        work_auditor.db.commit()
        snapshot=ROOT/'full_audit_v2_snapshot.sqlite'
        with sqlite3.connect(snapshot) as target_db:
            work_auditor.db.backup(target_db)
        hasher=hashlib.sha256()
        with snapshot.open('rb') as sf:
            for block in iter(lambda:sf.read(4*1024*1024),b''):hasher.update(block)
        snapshot_sha=hasher.hexdigest()
        remote=work_api.files().get(fileId=checkpoint['id'],fields='id,appProperties',supportsAllDrives=True).execute()
        if remote.get('appProperties',{}).get('data_sha256')!=checkpoint.get('appProperties',{}).get('data_sha256'):
            raise RuntimeError('Concurrent checkpoint update; stop without overwriting.')
        props={'data_sha256':snapshot_sha,'recipe_identity':V2_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
        saved=work_api.files().update(fileId=checkpoint['id'],body={'appProperties':props},media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
        class HashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):self.hasher.update(data);self.size+=len(data);return len(data)
        sink=HashSink()
        download=MediaIoBaseDownload(sink,work_api.files().get_media(fileId=saved['id'],supportsAllDrives=True),chunksize=4*1024*1024)
        done=False
        while not done:_,done=download.next_chunk(num_retries=3)
        if sink.hasher.hexdigest()!=snapshot_sha or sink.size!=snapshot.stat().st_size:
            raise RuntimeError('Checkpoint raw streaming readback mismatch.')
        checkpoint=saved;last_save=time.monotonic()
        full_audit_worker_state['checkpoint']={'id':saved['id'],'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True}
    try:
        work_auditor=audit_v2.FullArchiveAudit(V2_DB,recipe_all,tokenizer)
        full_audit_worker_state['state']='RUNNING_DATA_ONLY'
        write_worker_status()
        while True:
            next_member=work_auditor.next_entry()
            if next_member is None:break
            before_report=work_auditor.report()
            span_work=work_auditor.consume(next_member,work_cache[next_member['sha256']],max_rows=50000,commit_every=256)
            full_audit_worker_state['last_span']=span_work
            full_audit_worker_state['report']=work_auditor.report()
            full_audit_worker_state['current_member']={'role':next_member['role'],'member':next_member.get('member'),'payload_sha256':next_member['sha256']}
            if not span_work.get('processed',0) and not span_work.get('complete') and not span_work.get('already_complete'):
                raise RuntimeError('Audit made no cursor progress; refuse an endless loop.')
            if last_save==0 or time.monotonic()-last_save>=300:
                save_work_checkpoint()
            write_worker_status()
        save_work_checkpoint()
        final=work_auditor.report()
        full_audit_worker_state['report']=final
        full_audit_worker_state['state']='AUDIT_PASS_PENDING_CURRICULUM_ACCEPTANCE' if final['full_archive_token_linkage_split_audit_complete'] else 'AUDIT_FINISHED_WITH_FINDINGS'
    except Exception as exc:
        full_audit_worker_state['state']='STOPPED_REQUIRES_AUDIT_REPAIR'
        full_audit_worker_state['error_type']=type(exc).__name__
        full_audit_worker_state['error']=str(exc)
        if work_auditor is not None:
            full_audit_worker_state['report']=work_auditor.report()
            try:save_work_checkpoint()
            except Exception as checkpoint_error:
                full_audit_worker_state['checkpoint_save_error']=str(checkpoint_error)
    finally:
        if work_auditor is not None:work_auditor.close()
        write_worker_status()
write_worker_status()
full_audit_worker=threading.Thread(target=finite_full_data_audit_worker,name='MathVisionKidFiniteCPUDataAudit',daemon=True)
full_audit_worker.start()
print(full_audit_worker_state)



{'state': 'STARTING', 'started_utc': '2026-10-21T11:12:42.153326+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'recipe_sha256': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'checkpoint_id': '182zLHZU0XBpTfKm7ThJpfy37GgFJ7Ob2', 'training_started': False, 'gpu_used': False, 'model_weights_loaded': False, 'updated_utc': '2026-10-21T11:12:42.154959+00:00'}


In [ ]:
# Read-only progress; does not start another worker.
print(json.loads(worker_status_path.read_text(encoding='utf8')))


NameError: name 'json' is not defined

In [ ]:
import json
from pathlib import Path
p=Path('/content/mathvisionkid_data_only/full_audit_v3_worker_status.json')
print(json.loads(p.read_text(encoding='utf8')) if p.exists() else {'state':'RUNTIME_OR_STATUS_MISSING','training_started':False,'resume_checkpoint_required':True})


{'state': 'RUNNING_DATA_ONLY', 'started_utc': '2026-10-22T11:06:51.111748+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'writer_session': '49b86c37103a4c9db410fe085837063a', 'writer_guard': 'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS', 'restored_parent': {'state': 'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY', 'observed_utc': '2026-10-22T11:04:18.430512+00:00', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'checkpoint_bytes': 2321960960, 'checkpoint_sha256': 'ce71cd2e8d0c4e891333acf336e22152a7092cd4d0a1a3d3f42cf1d8a2409bde', 'fresh_raw_readback_verified': True, 'remote_metadata_unchanged_during_restore': True, 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4

### Tiếp tục chuẩn bị dữ liệu — 22/10/2026

Đã thêm 6 bài lớp 2 / 24 chat sau QA; bộ core vẫn 495 bài / 165 node. Chưa train.

Checkpoint audit V3 đã có trên Drive: `16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI`. Giữ nguyên checkpoint V2 và mọi dữ liệu cũ. Khôi phục đúng checkpoint V3 hiện có theo hash/recipe; không khởi tạo lại, không chạy tất cả hoặc chạy lại worker V2 cũ.

Mã khôi phục/worker V3 và evidence nằm trong PROGRESS cùng ID `1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX`, nhánh `data_resume_20261022`. Tổng candidate và recipe lấy từ STATE `1TLoQIbrRVmgsg-2uiUujCfM3HrK62Yme`. Ô trạng thái chỉ đọc sẽ báo thiếu runtime nếu phiên Colab đã mất.


In [ ]:
import os,json,threading
from pathlib import Path
import google.auth
try:
    probe_credentials,probe_project=google.auth.default()
    adc_available=True
except google.auth.exceptions.DefaultCredentialsError:
    adc_available=False
print({'stage':'MCP_CPU_DATA_ONLY_PROBE','gpu':os.environ.get('COLAB_GPU','0'),'adc_available':adc_available,'existing_globals':[k for k in ('ROOT','drive_api','recipe','recipe_pointer','v3_ref') if k in globals()],'audit_threads':[t.name for t in threading.enumerate() if t.name.startswith('MathVisionKid')],'training_started':False})


{'stage': 'MCP_CPU_DATA_ONLY_PROBE', 'gpu': '', 'adc_available': True, 'existing_globals': [], 'audit_threads': [], 'training_started': False}


In [ ]:
# Reuse Google credentials already granted by the owner; no new OAuth flow.
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import json,sys,sqlite3,hashlib
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
ROOT = Path('/content/mathvisionkid_data_only')
ROOT.mkdir(parents=True,exist_ok=True)
exec(compile('"""Account-neutral, data-only Colab bootstrap over fixed Drive IDs and hashes.\n\nNo training, weights, sharing edits or writes to Drive. Payloads are cached only\nin the caller\'s runtime directory; this module is not executed on the PC.\n"""\nfrom collections.abc import Mapping\nfrom pathlib import Path\nimport hashlib,io,json,zipfile\n\nSTATE_ID=\'1TLoQIbrRVmgsg-2uiUujCfM3HrK62Yme\'\nPROGRESS_ID=\'1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX\'\nCODE_FILES=(\'exhaustive_colab_loader.py\',\'math_data_portable_resume.py\',\'math_data_archive_audit.py\',\'math_data_candidate_stream.py\')\ndef sha(data):return hashlib.sha256(data).hexdigest()\ndef verify(data,ref):\n    if len(data)!=ref[\'bytes\'] or sha(data)!=ref[\'sha256\']:raise ValueError(\'pinned bytes/SHA256 mismatch\')\n    return data\n\nclass GoogleDriveReader:\n    def __init__(self,service):self.service=service\n    def read(self,file_id,max_bytes):\n        from googleapiclient.http import MediaIoBaseDownload\n        meta=self.service.files().get(fileId=file_id,fields=\'id,size,mimeType\',supportsAllDrives=True).execute()\n        size=int(meta.get(\'size\',-1))\n        if size<0 or size>max_bytes:raise ValueError(\'Drive raw size exceeds bounded request\')\n        output=io.BytesIO();down=MediaIoBaseDownload(output,self.service.files().get_media(fileId=file_id,supportsAllDrives=True),chunksize=4*1024*1024)\n        done=False\n        while not done:\n            _,done=down.next_chunk(num_retries=3)\n            if output.tell()>max_bytes:raise ValueError(\'download exceeds bounded request\')\n        raw=output.getvalue()\n        if len(raw)!=size:raise ValueError(\'Drive raw size changed during download\')\n        return raw\n\ndef latest_controls(reader):\n    """Mutable STATE read twice; recipe itself is pinned by its exact hash."""\n    before=reader.read(STATE_ID,16*1024*1024);state=json.loads(before)\n    pointer=state[\'latest_all_data_union_recipe\']\n    if pointer[\'progress_id\']!=PROGRESS_ID:raise ValueError(\'unexpected progress control ID\')\n    member=pointer[\'member\']\n    if not isinstance(member,str) or member.startswith(\'/\') or \'..\' in Path(member).parts:raise ValueError(\'unsafe recipe member\')\n    raw=reader.read(PROGRESS_ID,128*1024*1024)\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if archive.namelist().count(member)!=1:raise ValueError(\'missing/ambiguous latest union recipe\')\n        data=archive.read(member)\n    if sha(data)!=pointer[\'sha256\']:raise ValueError(\'STATE recipe hash differs from PROGRESS\')\n    recipe=json.loads(data)\n    if recipe.get(\'pending_discovery\'):raise ValueError(\'unresolved declared supplements; refusing partial data\')\n    if reader.read(STATE_ID,16*1024*1024)!=before:raise ValueError(\'STATE changed during read; resume from latest controls\')\n    if state.get(\'training_started\'):raise ValueError(\'unexpected training-started state for data-only notebook\')\n    return state,recipe,pointer\n\ndef bootstrap_code(reader,state,directory):\n    """Load exact code members from latest accepted bundle; no archive extractall."""\n    runs=[r for r in state.get(\'sgk_finish_runs\',[]) if r.get(\'bundle\')]\n    if not runs:raise ValueError(\'no explicit accepted code bundle\')\n    run=runs[-1];ref=run[\'bundle\'];fid=ref.get(\'id\',ref.get(\'file_id\'))\n    if not fid:raise ValueError(\'bundle file ID required\')\n    raw=verify(reader.read(fid,ref[\'bytes\']),ref)\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    receipts=[]\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError(\'duplicate bundle member\')\n        manifest=json.loads(archive.read(\'manifest.json\'))\n        for name in CODE_FILES:\n            path=\'code/\'+name;pin=manifest[\'members\'].get(path)\n            if not pin:raise ValueError(\'latest bundle lacks candidate reader dependency: \'+name)\n            data=verify(archive.read(path),pin)\n            (directory/name).write_bytes(data);receipts.append(dict(member=path,sha256=sha(data),bytes=len(data)))\n    return dict(bundle_id=fid,bundle_sha256=ref[\'sha256\'],code_members=receipts,training_started=False)\n\nclass LazyDriveCache(Mapping):\n    """One hash-verified payload cached at a time in ephemeral Colab storage.\n\n    source/chat members remain inside their archives. Checkpoints are separate\n    and never removed by cache eviction. No files are created on Drive.\n    """\n    def __init__(self,reader,recipe,directory):\n        self.reader=reader;self.directory=Path(directory);self.directory.mkdir(parents=True,exist_ok=True)\n        self.artifacts=recipe[\'artifacts\'];self.lookup={};self.current=None\n        for digest,ref in self.artifacts.items():\n            if digest!=ref[\'sha256\'] or not ref.get(\'file_ids\'):raise ValueError(\'hash and existing Drive ID required\')\n            self.lookup[digest]=digest\n            for fid in ref[\'file_ids\']:self.lookup[fid]=digest\n    def __iter__(self):return iter(self.lookup)\n    def __len__(self):return len(self.lookup)\n    def __getitem__(self,key):\n        digest=self.lookup[key];ref=self.artifacts[digest]\n        if self.current and self.current[0]==digest:\n            path=self.current[1]\n            if path.stat().st_size!=ref[\'bytes\'] or sha(path.read_bytes())!=digest:raise ValueError(\'cached payload changed\')\n            return path\n        data=verify(self.reader.read(ref[\'file_ids\'][0],ref[\'bytes\']),ref)\n        suffix=\'.gz\' if any(n.endswith(\'.gz\') for n in ref.get(\'file_names\',[])) else \'.zip\'\n        dest=self.directory/(digest+suffix);dest.write_bytes(data)\n        if self.current:\n            old=self.current[1]\n            if old.parent.resolve()!=self.directory.resolve():raise ValueError(\'cache deletion escaped runtime directory\')\n            old.unlink()\n        self.current=(digest,dest)\n        return dest\n\ndef scope_receipt(state,recipe,pointer):\n    return dict(accepted_cases=state[\'accepted_cases\'],declared_candidate_chat_rows=state[\'training_chat_rows\'],payload_references=len(recipe[\'artifacts\']),chat_members=len(recipe[\'chat_members\']),recipe_sha256=pointer[\'sha256\'],historical_train_only_total_verified=False,full_archive_token_split_audit_complete=False,TRAINING_DATASET_READY=False,training_started=False,selection=\'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET\',runtime_accounts=\'Authenticate any account with access to the same existing IDs; no credentials in dataset\')\n', 'math_data_drive_bootstrap.py', 'exec'), globals())
reader = GoogleDriveReader(drive_api)
state,recipe,recipe_pointer = latest_controls(reader)
code_receipt = bootstrap_code(reader,state,ROOT/'code')
sys.path.insert(0,str(ROOT/'code'))
from math_data_candidate_stream import CandidateStream,DataOnlyConsumer
from exhaustive_colab_loader import load_qwen_tokenizer,tokenize_assistant_only
print(scope_receipt(state,recipe,recipe_pointer))


RefreshError: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google_auth_httplib2._Response object at 0x7b1e6caecd70>)

In [ ]:
# Load the already published, exact V3 auditor. CPU data only.
import io,zipfile,importlib,hashlib,os
assert os.environ.get('COLAB_GPU','0') in ('0','')
progress_code=reader.read('1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX',128*1024*1024)
module_pins={
'data_resume_20261022/code/math_data_full_audit_v3.py':'02431da433060a6574127f9a8e14d90cabffe85d8d0150c3b2a331dc15b4be1f',
'data_resume_20261022/code/math_data_primary782_bindings_v3.py':'4d75e90c07eecccf7571d68015dda16e85a690e7ee2379b28d517d201472580b'}
with zipfile.ZipFile(io.BytesIO(progress_code)) as code_zip:
    assert hashlib.sha256(code_zip.read(recipe_pointer['member'])).hexdigest()==recipe_pointer['sha256']
    for member,pin in module_pins.items():
        module_bytes=code_zip.read(member)
        assert hashlib.sha256(module_bytes).hexdigest()==pin
        (ROOT/'code'/member.rsplit('/',1)[-1]).write_bytes(module_bytes)
importlib.invalidate_caches()
import math_data_full_audit_v3 as audit_v3
assert audit_v3.plan_identity(recipe)=='06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2'
# CPU data preparation only: restore the existing checkpoint, never create one.
import json,hashlib,sqlite3,threading,os
from datetime import datetime,timezone
from googleapiclient.http import MediaIoBaseDownload
import math_data_candidate_stream as C
assert os.environ.get('COLAB_GPU','0') in ('0','')
assert recipe_pointer['sha256']=='e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8'
assert len(recipe['source_members'])+len(recipe['chat_members'])==249
if any(t.name.startswith('MathVisionKidFiniteCPUDataAudit') and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError('Existing active data worker; inspect it without restoring or launching a duplicate.')
EXISTING_V3_ID='16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI'
EXISTING_V3_IDENTITY='06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2'
meta_before=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
props=meta_before['appProperties'];pin=props['data_sha256'];size=int(meta_before['size'])
assert props['recipe_identity']==EXISTING_V3_IDENTITY and len(pin)==64 and 0<size<64*1024**3
RESTORED_V3_DB=ROOT/('existing_v3_'+pin[:16]+'.sqlite')
assert not RESTORED_V3_DB.exists(), 'Checkpoint path already exists: verify it instead of overwriting/restarting.'
print({'stage':'STREAMING_EXISTING_V3_CHECKPOINT_TO_COLAB_ONLY','bytes':size,'training_started':False})
with RESTORED_V3_DB.open('wb') as target:
    class HashingFile:
        def __init__(self):self.h=hashlib.sha256();self.size=0
        def write(self,data):
            count=target.write(data);self.h.update(data);self.size+=count;return count
    sink=HashingFile();download=MediaIoBaseDownload(sink,drive_api.files().get_media(fileId=EXISTING_V3_ID,supportsAllDrives=True),chunksize=32*1024*1024)
    done=False
    while not done:
        _,done=download.next_chunk(num_retries=3)
        if sink.size%(256*1024*1024)==0 or done:
            print({'checkpoint_download_bytes':sink.size,'expected_bytes':size,'training_started':False},flush=True)
assert sink.size==size and sink.h.hexdigest()==pin
meta_after=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert meta_before==meta_after, 'Remote checkpoint advanced during restore; do not resume a stale writer.'
con=sqlite3.connect(RESTORED_V3_DB.as_uri()+'?mode=ro',uri=True)
try:
    assert con.execute('SELECT identity,sequence_limit FROM config').fetchone()==(EXISTING_V3_IDENTITY,40960)
    assert con.execute('PRAGMA quick_check').fetchone()==('ok',)
    entries=recipe['source_members']+recipe['chat_members'];declared={C.member_key(e):e for e in entries}
    member_rows=con.execute('SELECT id,role,next_line,complete,expected_rows FROM members').fetchall()
    for key,role,line,complete,expected in member_rows:
        e=declared[key];assert role==e['role'] and expected==e['rows'] and 0<=line<=expected
        if complete:assert line==expected
    done_keys={r[0] for r in member_rows if r[3]}
    next_entry=next((e for e in entries if C.member_key(e) not in done_keys),None)
    probe_receipt=dict(state='EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY',observed_utc=datetime.now(timezone.utc).isoformat(),checkpoint_id=EXISTING_V3_ID,checkpoint_bytes=size,checkpoint_sha256=pin,fresh_raw_readback_verified=True,remote_metadata_unchanged_during_restore=True,audit_identity=EXISTING_V3_IDENTITY,recipe_sha256=recipe_pointer['sha256'],source_versions_indexed=con.execute('SELECT count(*) FROM source_versions').fetchone()[0],unique_chat_rows_processed=con.execute('SELECT count(*) FROM samples').fetchone()[0],actual_rows_by_split=dict(con.execute('SELECT split,count(*) FROM samples GROUP BY split')),members_complete=len(done_keys),members_expected=249,issues=dict(con.execute('SELECT kind,count(*) FROM issues GROUP BY kind')),first_issue_metadata=con.execute('SELECT member,line,kind,detail FROM issues LIMIT 8').fetchall(),next_member=None if next_entry is None else dict(role=next_entry['role'],member=next_entry.get('member'),rows=next_entry['rows']),sqlite_quick_check='ok',checkpoint_mutated=False,worker_started_by_this_probe=False,training_started=False,gpu_used=False,model_weights_loaded=False,TRAINING_DATASET_READY=False)
finally:con.close()
(ROOT/'existing_v3_restore_readonly_receipt.json').write_text(json.dumps(probe_receipt,ensure_ascii=False,indent=2),encoding='utf8')
print(probe_receipt)


{'stage': 'STREAMING_EXISTING_V3_CHECKPOINT_TO_COLAB_ONLY', 'bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 268435456, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 536870912, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 805306368, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 1073741824, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 1342177280, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 1610612736, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 1879048192, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 2147483648, 'expected_bytes': 2321960960, 'training_started': False}
{'checkpoint_download_bytes': 2321960960, 'expected_bytes': 2321960960, 'training_started': False}
{'st

In [ ]:
# Attach to the existing, fully raw-verified V3 checkpoint; never create a file.
import threading,time,uuid,shutil
from datetime import datetime,timezone
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
assert os.environ.get('COLAB_GPU','0') in ('0','')
if any(t.name.startswith('MathVisionKidFiniteCPUDataAudit') and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError('A data audit worker is already active; do not duplicate it.')
assert probe_receipt['state']=='EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY'
assert probe_receipt['fresh_raw_readback_verified']
V3_DB=RESTORED_V3_DB
V3_ID=probe_receipt['audit_identity']
recipe_all=recipe;pointer_all=recipe_pointer
assert audit_v3.plan_identity(recipe_all)==V3_ID
assert len(recipe_all['source_members'])==218 and len(recipe_all['chat_members'])==31
def file_digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
assert file_digest(V3_DB)==probe_receipt['checkpoint_sha256']
if shutil.disk_usage(ROOT).free<3*V3_DB.stat().st_size+1024**3:
    raise RuntimeError('Insufficient temporary Colab disk for data checkpoint snapshots.')
fresh=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert fresh==meta_after,'Checkpoint changed after read-only restore; refuse a stale writer.'
from exhaustive_colab_loader import load_qwen_tokenizer
tokenizer=load_qwen_tokenizer()  # tokenizer assets only, no model weights
preclaim=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert preclaim==fresh,'Checkpoint advanced while loading tokenizer; refuse a stale writer.'
writer_session=uuid.uuid4().hex
claim=dict(fresh['appProperties'],writer_session=writer_session)
v3_ref=drive_api.files().update(fileId=EXISTING_V3_ID,body={'appProperties':claim},fields='id,size,appProperties',supportsAllDrives=True).execute()
assert v3_ref['id']==EXISTING_V3_ID
assert v3_ref['appProperties']['data_sha256']==probe_receipt['checkpoint_sha256']
worker_status_path=ROOT/'full_audit_v3_worker_status.json'
full_audit_worker_state={'state':'RESUMING_EXISTING_CHECKPOINT','started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l','audit_identity':V3_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':EXISTING_V3_ID,'writer_session':writer_session,'writer_guard':'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS','restored_parent':probe_receipt,'training_started':False,'gpu_used':False,'model_weights_loaded':False}
def write_worker_status():
    full_audit_worker_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    tmp=worker_status_path.with_suffix('.writing')
    tmp.write_text(json.dumps(full_audit_worker_state,ensure_ascii=False,indent=2),encoding='utf8')
    tmp.replace(worker_status_path)

def finite_full_data_audit_worker_v3():
    global v3_ref
    auditor=None;checkpoint=dict(v3_ref);last_save=0.0
    def verify_writer():
        remote=work_api.files().get(fileId=checkpoint['id'],fields='id,size,appProperties',supportsAllDrives=True).execute()
        props=remote.get('appProperties',{})
        if props.get('writer_session')!=writer_session or props.get('recipe_identity')!=V3_ID or props.get('data_sha256')!=checkpoint.get('appProperties',{}).get('data_sha256'):
            raise RuntimeError('Checkpoint writer or hash changed; stop without overwriting.')
        return remote
    def save_checkpoint():
        nonlocal checkpoint,last_save
        global v3_ref
        auditor.db.commit();snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
        target_db=sqlite3.connect(snapshot)
        try:auditor.db.backup(target_db)
        finally:target_db.close()
        snapshot_sha=file_digest(snapshot)
        verify_writer()
        props={'recipe_identity':V3_ID,'data_sha256':snapshot_sha,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING','writer_session':writer_session}
        saved=work_api.files().update(fileId=checkpoint['id'],body={'appProperties':props},media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
        # Retain the completed write identity even if the verification read fails.
        # It does not become a verified receipt until all raw bytes match.
        checkpoint=saved;v3_ref=saved
        class HashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):self.hasher.update(data);self.size+=len(data);return len(data)
        sink=HashSink();download=MediaIoBaseDownload(sink,work_api.files().get_media(fileId=saved['id'],supportsAllDrives=True),chunksize=8*1024*1024)
        done=False
        while not done:_,done=download.next_chunk(num_retries=3)
        if sink.hasher.hexdigest()!=snapshot_sha or sink.size!=snapshot.stat().st_size:
            raise RuntimeError('V3 raw checkpoint readback differs.')
        after=verify_writer()
        last_save=time.monotonic()
        proof={'id':saved['id'],'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True,'report_at_snapshot':auditor.report(),'observed_utc':datetime.now(timezone.utc).isoformat()}
        full_audit_worker_state['checkpoint']=proof
        (ROOT/'v3_latest_checkpoint_ref.json').write_text(json.dumps({'metadata':after,'proof':proof},ensure_ascii=False,indent=2),encoding='utf8')
    try:
        work_api=build('drive','v3',credentials=credentials,cache_discovery=False)
        work_reader=GoogleDriveReader(work_api)
        work_cache=LazyDriveCache(work_reader,recipe_all,ROOT/'full_audit_v3_worker_payload_cache')
        auditor=audit_v3.FullArchiveAudit(V3_DB,recipe_all,tokenizer)
        full_audit_worker_state['report']=auditor.report();write_worker_status()
        last_save=time.monotonic()
        full_audit_worker_state['state']='RUNNING_DATA_ONLY';write_worker_status()
        while True:
            entry=auditor.next_entry()
            if entry is None:break
            verify_writer()
            full_audit_worker_state['current_member']={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']}
            write_worker_status()
            span=auditor.consume(entry,work_cache[entry['sha256']],max_rows=50000,commit_every=256)
            full_audit_worker_state.update(last_span=span,report=auditor.report(),current_member={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']})
            if not span.get('processed',0) and not span.get('complete') and not span.get('already_complete'):
                raise RuntimeError('No finite cursor progress; refuse an endless loop.')
            if 'checkpoint' not in full_audit_worker_state or time.monotonic()-last_save>=300:save_checkpoint()
            write_worker_status()
        save_checkpoint();report=auditor.report()
        full_audit_worker_state.update(report=report,state='AUDIT_PASS_PENDING_CURRICULUM_ACCEPTANCE' if report['full_archive_token_linkage_split_audit_complete'] else 'AUDIT_FINISHED_WITH_FINDINGS')
    except Exception as exc:
        full_audit_worker_state.update(state='STOPPED_REQUIRES_AUDIT_REPAIR',error_type=type(exc).__name__,error=str(exc))
        if auditor is not None:
            full_audit_worker_state['report']=auditor.report()
            try:save_checkpoint()
            except Exception as checkpoint_error:full_audit_worker_state['checkpoint_save_error']=str(checkpoint_error)
    finally:
        if auditor is not None:auditor.close()
        write_worker_status()
write_worker_status()
full_audit_worker=threading.Thread(target=finite_full_data_audit_worker_v3,name='MathVisionKidFiniteCPUDataAuditV3',daemon=True)
full_audit_worker.start()
print(full_audit_worker_state)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{'state': 'RESUMING_EXISTING_CHECKPOINT', 'started_utc': '2026-10-22T11:06:51.111748+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'writer_session': '49b86c37103a4c9db410fe085837063a', 'writer_guard': 'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS', 'restored_parent': {'state': 'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY', 'observed_utc': '2026-10-22T11:04:18.430512+00:00', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'checkpoint_bytes': 2321960960, 'checkpoint_sha256': 'ce71cd2e8d0c4e891333acf336e22152a7092cd4d0a1a3d3f42cf1d8a2409bde', 'fresh_raw_readback_verified': True, 'remote_metadata_unchanged_during_restore': True, 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_s

In [ ]:
# Read actual finite data-audit thread, in-memory progress and persisted status.
import json,threading
from pathlib import Path
from datetime import datetime,timezone
p=Path('/content/mathvisionkid_data_only/full_audit_v3_worker_status.json')
live_status=globals().get('full_audit_worker_state')
print({'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':any(t.name=='MathVisionKidFiniteCPUDataAuditV3' and t.is_alive() for t in threading.enumerate()),'stored_status':dict(live_status) if live_status is not None else json.loads(p.read_text('utf8')) if p.exists() else None,'status_origin':'LIVE_KERNEL_MEMORY' if live_status is not None else 'PERSISTED_STATUS_FILE','training_started':False})


{'observed_utc': '2026-10-22T11:42:17.516675+00:00', 'worker_alive': False, 'stored_status': {'state': 'PAUSED_AT_VERIFIED_CHECKPOINT_FOR_VALIDATED_PERFORMANCE_UPGRADE', 'started_utc': '2026-10-22T11:06:51.111748+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'writer_session': '49b86c37103a4c9db410fe085837063a', 'writer_guard': 'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS', 'restored_parent': {'state': 'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY', 'observed_utc': '2026-10-22T11:04:18.430512+00:00', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'checkpoint_bytes': 2321960960, 'checkpoint_sha256': 'ce71cd2e8d0c4e891333acf336e22152a7092cd4d0a1a3d3f42cf1d8a2409bde', 'fresh_raw_readback_verified': True, 'remote_metadata_

In [ ]:
# Read-only diagnostic of our own CPU data worker: function locations and byte counters only.
import sys,threading
worker_diagnostic=[]
for thread in threading.enumerate():
    if thread.name=='MathVisionKidFiniteCPUDataAuditV3':
        frame=sys._current_frames().get(thread.ident);stack=[];transfer={}
        while frame is not None and len(stack)<20:
            stack.append({'file':frame.f_code.co_filename,'function':frame.f_code.co_name,'line':frame.f_lineno})
            if frame.f_code.co_name=='save_checkpoint':
                downloader=frame.f_locals.get('download')
                if downloader is not None:
                    transfer={'stage':'FRESH_RAW_CHECKPOINT_READBACK','bytes_received':getattr(downloader,'_progress',None),'expected_bytes':getattr(downloader,'_total_size',None),'chunk_bytes':getattr(downloader,'_chunksize',None)}
            frame=frame.f_back
        worker_diagnostic.append({'thread':thread.name,'alive':thread.is_alive(),'stack':stack,'transfer':transfer})
print({'worker_diagnostic':worker_diagnostic,'training_started':False})


{'worker_diagnostic': [{'thread': 'MathVisionKidFiniteCPUDataAuditV3', 'alive': True, 'stack': [{'file': '/usr/lib/python3.13/ssl.py', 'function': 'read', 'line': 1149}, {'file': '/usr/lib/python3.13/ssl.py', 'function': 'recv_into', 'line': 1315}, {'file': '/usr/lib/python3.13/socket.py', 'function': 'readinto', 'line': 723}, {'file': '/usr/lib/python3.13/http/client.py', 'function': '_read_status', 'line': 304}, {'file': '/usr/lib/python3.13/http/client.py', 'function': 'begin', 'line': 343}, {'file': '/usr/lib/python3.13/http/client.py', 'function': 'getresponse', 'line': 1478}, {'file': '/usr/local/lib/python3.13/dist-packages/httplib2/__init__.py', 'function': '_conn_request', 'line': 1427}, {'file': '/usr/local/lib/python3.13/dist-packages/httplib2/__init__.py', 'function': '_request', 'line': 1475}, {'file': '/usr/local/lib/python3.13/dist-packages/httplib2/__init__.py', 'function': 'request', 'line': 1755}, {'file': '/usr/local/lib/python3.13/dist-packages/google_auth_httplib2.

In [ ]:
# Read committed progress only; no original row contents and no writes.
import sqlite3,json
from datetime import datetime,timezone
current_spec=full_audit_worker_state.get('current_member',{})
entry_probe=next(e for e in recipe['source_members']+recipe['chat_members'] if e['role']==current_spec.get('role') and e['sha256']==current_spec.get('payload_sha256') and e.get('member')==current_spec.get('member'))
k=C.member_key(entry_probe)
con_probe=sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=2)
try:
    current_cursor=con_probe.execute('SELECT next_line,complete,expected_rows FROM members WHERE id=?',(k,)).fetchone()
    unique_chats=con_probe.execute('SELECT count(*) FROM samples').fetchone()[0]
    issues_committed=dict(con_probe.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
finally:con_probe.close()
print({'observed_utc':datetime.now(timezone.utc).isoformat(),'committed_cursor':current_cursor,'role':entry_probe['role'],'unique_chat_rows_committed':unique_chats,'issues_committed':issues_committed,'training_started':False})


{'observed_utc': '2026-10-22T11:45:44.619459+00:00', 'committed_cursor': (200000, 0, 963504), 'role': 'candidate_chat', 'unique_chat_rows_committed': 200000, 'issues_committed': {'SOURCE_SPLIT_MISSING_OR_UNKNOWN': 1}, 'training_started': False}


In [ ]:
# Cooperative pause only at a checkpoint whose committed cursors also match.
# This installs the controller; requesting a pause is a separate explicit cell.
import hashlib,json,sqlite3
assert globals().get('_mathvision_pause_controller_installed',False)
assert audit_v3.FullArchiveAudit.consume in (
    _mathvision_checkpoint_boundary_consume,
    globals().get('_mathvision_checkpoint_boundary_consume_v2'))

def _mathvision_checkpoint_boundary_consume_v2(self,entry,path,*args,**kwargs):
    if _mathvision_pause_requested:
        verified=full_audit_worker_state.get('checkpoint')
        if verified and verified.get('fresh_raw_readback_verified') and self.report()==verified['report_at_snapshot']:
            snapshot_path=ROOT/'full_audit_v3_snapshot.sqlite'
            with sqlite3.connect(snapshot_path.as_uri()+'?mode=ro',uri=True) as snapshot_check:
                query='SELECT id,role,next_line,complete,expected_rows FROM members ORDER BY id'
                saved_cursors=snapshot_check.execute(query).fetchall()
            current_cursors=self.db.execute(query).fetchall()
            if current_cursors==saved_cursors:
                cursor_digest=hashlib.sha256(json.dumps(saved_cursors,separators=(',',':')).encode()).hexdigest()
                full_audit_worker_state['pause_receipt']={
                    'checkpoint_sha256':verified['sha256'],
                    'committed_members_equal_verified_snapshot':True,
                    'member_cursors_sha256':cursor_digest,
                    'member_cursor_count':len(saved_cursors),
                    'working_database_byte_identity_claimed':False,
                    'safe_resume_source':'RAW_VERIFIED_SNAPSHOT',
                    'training_started':False}
                full_audit_worker_state['state']='PAUSED_AT_VERIFIED_CHECKPOINT_FOR_VALIDATED_PERFORMANCE_UPGRADE'
                full_audit_worker_state['pause_reason']='Exact batched tokenization and fewer large checkpoint copies; no data changes.'
                write_worker_status()
                raise SystemExit('Cooperative pause at verified snapshot with equal committed cursors')
    return _mathvision_original_consume(self,entry,path,*args,**kwargs)

audit_v3.FullArchiveAudit.consume=_mathvision_checkpoint_boundary_consume_v2
print({'pause_controller_version':2,'pause_requested':_mathvision_pause_requested,
       'requires_equal_committed_member_cursors':True,'training_started':False})


{'pause_controller_version': 2, 'pause_requested': False, 'requires_equal_committed_member_cursors': True, 'training_started': False}


In [ ]:
# Metadata-only source/exclusion evidence; do not print question/answer content.
import sqlite3,json
from datetime import datetime,timezone
diagnostic_id='exhaustive_v1_div_1563_43'
diagnostic_db=sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=3)
try:
    diagnostic_versions=diagnostic_db.execute('SELECT raw_sha,canonical_sha,source_split FROM source_versions WHERE id=?',(diagnostic_id,)).fetchall()
    diagnostic_occurrences=diagnostic_db.execute('SELECT split,count(*) FROM samples WHERE case_id=? GROUP BY split',(diagnostic_id,)).fetchall()
    diagnostic_complete=diagnostic_db.execute('SELECT role,count(*) FROM members WHERE complete=1 GROUP BY role').fetchall()
    print({'observed_utc':datetime.now(timezone.utc).isoformat(),'sample_id':diagnostic_id,
           'source_versions_metadata':diagnostic_versions,'chat_links_by_split_so_far':diagnostic_occurrences,
           'completed_members_by_role':diagnostic_complete,'zero_links_is_final_proof':False,
           'source_content_read':False,'training_started':False})
finally:
    diagnostic_db.close()


{'observed_utc': '2026-10-22T11:35:39.647619+00:00', 'sample_id': 'exhaustive_v1_div_1563_43', 'source_versions_metadata': [('1729ed713dfedcc0205b130c6573313dcc5784c7eaf5618d0cf32fdc728cf26b', '1729ed713dfedcc0205b130c6573313dcc5784c7eaf5618d0cf32fdc728cf26b', 'regression_only')], 'chat_links_by_split_so_far': [], 'completed_members_by_role': [('candidate_source', 218)], 'zero_links_is_final_proof': False, 'source_content_read': False, 'training_started': False}


In [ ]:
_mathvision_pause_requested=True
print({'pause_requested':True,'policy':'Finish current span and raw-verify next checkpoint, then compare all committed member cursors before stopping worker.','training_started':False})

{'pause_requested': True, 'policy': 'Finish current span and raw-verify next checkpoint, then compare all committed member cursors before stopping worker.', 'training_started': False}


In [ ]:
# Resume from the latest full raw-verified Colab snapshot, never re-download to PC.
import os,json,hashlib,sqlite3,shutil,threading,base64,importlib.util,sys
from pathlib import Path
assert os.environ.get('COLAB_GPU','0') in ('0','')
full_audit_worker.join(timeout=0)
assert not full_audit_worker.is_alive(),'Wait for cooperative checkpoint pause; do not interrupt worker.'
assert full_audit_worker_state['state']=='PAUSED_AT_VERIFIED_CHECKPOINT_FOR_VALIDATED_PERFORMANCE_UPGRADE'
assert full_audit_worker_state['pause_receipt']['committed_members_equal_verified_snapshot']
upgrade_parent_state=json.loads(json.dumps(full_audit_worker_state))
upgrade_proof=upgrade_parent_state['checkpoint']
assert upgrade_proof['fresh_raw_readback_verified']
upgrade_snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
assert upgrade_snapshot.stat().st_size==upgrade_proof['bytes']
assert file_digest(upgrade_snapshot)==upgrade_proof['sha256']
upgrade_remote=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert upgrade_remote['appProperties']['data_sha256']==upgrade_proof['sha256']
assert upgrade_remote['appProperties']['writer_session']==upgrade_parent_state['writer_session']
assert int(upgrade_remote['size'])==upgrade_proof['bytes']
upgrade_db=ROOT/('batch_resume_'+upgrade_proof['sha256'][:16]+'.sqlite')
assert upgrade_db!=upgrade_snapshot and not upgrade_db.exists()
assert shutil.disk_usage(ROOT).free>4*upgrade_proof['bytes']+1024**3
shutil.copyfile(upgrade_snapshot,upgrade_db)
assert file_digest(upgrade_db)==upgrade_proof['sha256']
with sqlite3.connect(upgrade_db.as_uri()+'?mode=ro',uri=True) as upgrade_check:
    assert upgrade_check.execute('PRAGMA quick_check').fetchall()==[('ok',)]
    upgrade_cursors=upgrade_check.execute('SELECT id,role,next_line,complete,expected_rows FROM members ORDER BY id').fetchall()
    assert hashlib.sha256(json.dumps(upgrade_cursors,separators=(',',':')).encode()).hexdigest()==upgrade_parent_state['pause_receipt']['member_cursors_sha256']
    assert upgrade_check.execute('SELECT count(*) FROM samples').fetchone()[0]==upgrade_proof['report_at_snapshot']['unique_chat_rows_processed']
    assert upgrade_check.execute('SELECT count(*) FROM source_versions').fetchone()[0]==upgrade_proof['report_at_snapshot']['source_versions_indexed']
RESTORED_V3_DB=upgrade_db
probe_receipt={'state':'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY','fresh_raw_readback_verified':True,
    'checkpoint_id':EXISTING_V3_ID,'checkpoint_sha256':upgrade_proof['sha256'],
    'checkpoint_bytes':upgrade_proof['bytes'],'audit_identity':recipe['identity'] if 'identity' in recipe else audit_v3.plan_identity(recipe),
    'recipe_sha256':recipe_pointer['sha256'],'local_verified_snapshot_reused':True,
    'source_versions_indexed':upgrade_proof['report_at_snapshot']['source_versions_indexed'],
    'unique_chat_rows_processed':upgrade_proof['report_at_snapshot']['unique_chat_rows_processed'],
    'old_checkpoint_evidence_preserved':True,'training_started':False}
meta_after=upgrade_remote
audit_v3.FullArchiveAudit.consume=_mathvision_original_consume
_mathvision_pause_requested=False
adapter_bytes=base64.b64decode('IiIiT3B0LWluLCBib3VuZGVkIGV4YWN0IHRva2VuaXplciBiYXRjaGluZyBmb3IgdGhlIHVuY2hhbmdlZCB2MyBDUFUgc2Nhbm5lci4KCkFmdGVyIGFuIG9yZGVybHkgc2Nhbm5lciBwYXVzZTo6CgogICAgYWRhcHRlciA9IEJhdGNoVG9rZW5BdWRpdEFkYXB0ZXIoYXVkaXQudG9rZW5pemVyLCBiYXRjaF9zaXplPTEyOCkKICAgIHdpdGggYWRhcHRlci5pbnN0YWxsZWQoKToKICAgICAgICByZXN1bHQgPSBhZGFwdGVyLmNvbnN1bWUoYXVkaXQsIGVudHJ5LCBsb2NhbF9wYXRoLCBtYXhfcm93cz0yMDAwMCkKCk9ubHkgY2FsbHMgbWFkZSBieSBgYGFkYXB0ZXIuY29uc3VtZWBgIGluIGl0cyBjdXJyZW50IHRocmVhZCB1c2UgdGhlIGNhY2hlLgpPdGhlciBjYWxsZXJzIHJldGFpbiB0aGUgb3JpZ2luYWwgZXhhY3QgdG9rZW5pemVyL3BheWxvYWQgZnVuY3Rpb25zLiBObyBkYXRhLApyZWNpcGUsIHNvdXJjZSBsaW5rYWdlLCBzY2FubmVyIGltcGxlbWVudGF0aW9uLCBvciBjaGVja3BvaW50IHNjaGVtYSBjaGFuZ2VzLgpPdmVyc2l6ZWQgaW5kaXZpZHVhbCByb3dzIHVzZSB0aGUgb3JpZ2luYWwgZXhhY3Qgc2luZ2xlLXJvdyBpbXBsZW1lbnRhdGlvbi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29udGV4dGxpYiBpbXBvcnQgY29udGV4dG1hbmFnZXIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGpzb24KaW1wb3J0IHRocmVhZGluZwoKaW1wb3J0IGV4aGF1c3RpdmVfY29sYWJfbG9hZGVyIGFzIEwKaW1wb3J0IG1hdGhfZGF0YV9hcmNoaXZlX2F1ZGl0IGFzIEEKZnJvbSBtYXRoX2RhdGFfY2FuZGlkYXRlX3N0cmVhbSBpbXBvcnQgbWVtYmVyX2tleQoKX09SSUdJTkFMX1RPS0VOSVpFID0gTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seQpfT1JJR0lOQUxfUEFZTE9BRCA9IEEucGF5bG9hZApfTE9DQUwgPSB0aHJlYWRpbmcubG9jYWwoKQpfTE9DSyA9IHRocmVhZGluZy5STG9jaygpCl9PV05FUiA9IE5vbmUKX0FDVElWRSA9IDAKCgpkZWYgX3Jvd19ieXRlcyhyb3cpOgogICAgcmV0dXJuIEwuY2Fub25pY2FsX2pzb24ocm93KS5lbmNvZGUoInV0Zi04IikKCgpjbGFzcyBfUHJlcGFyZWRUb2tlbml6ZXI6CiAgICAiIiJGZWVkcyBleGFjdCBvZmZpY2lhbCBiYXRjaCBvdXRwdXRzIHRocm91Z2ggYWxsIG9yaWdpbmFsIHJvdyBnYXRlcy4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpdGVtLCBmdWxsLCBvZmZzZXRzKToKICAgICAgICBzZWxmLml0ZW0sIHNlbGYuZnVsbCwgc2VsZi5vZmZzZXRzID0gaXRlbSwgZnVsbCwgb2Zmc2V0cwoKICAgIGRlZiBhcHBseV9jaGF0X3RlbXBsYXRlKHNlbGYsIG1lc3NhZ2VzLCAqLCB0b2tlbml6ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5hYmxlX3RoaW5raW5nKToKICAgICAgICBhc3NlcnQgZW5hYmxlX3RoaW5raW5nIGlzIEZhbHNlCiAgICAgICAgaWYgdG9rZW5pemU6CiAgICAgICAgICAgIGFzc2VydCBtZXNzYWdlcyA9PSBzZWxmLml0ZW1bMV1bIm1lc3NhZ2VzIl0gYW5kIG5vdCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQKICAgICAgICAgICAgcmV0dXJuIHNlbGYuZnVsbAogICAgICAgIGlmIGFkZF9nZW5lcmF0aW9uX3Byb21wdDoKICAgICAgICAgICAgYXNzZXJ0IG1lc3NhZ2VzID09IHNlbGYuaXRlbVsxXVsibWVzc2FnZXMiXVs6Ml0KICAgICAgICAgICAgcmV0dXJuIHNlbGYuaXRlbVszXQogICAgICAgIGFzc2VydCBtZXNzYWdlcyA9PSBzZWxmLml0ZW1bMV1bIm1lc3NhZ2VzIl0KICAgICAgICByZXR1cm4gc2VsZi5pdGVtWzRdCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIHJlbmRlcmVkLCAqLCBhZGRfc3BlY2lhbF90b2tlbnMsIHJldHVybl9vZmZzZXRzX21hcHBpbmcpOgogICAgICAgIGFzc2VydCByZW5kZXJlZCA9PSBzZWxmLml0ZW1bNF0KICAgICAgICBhc3NlcnQgYWRkX3NwZWNpYWxfdG9rZW5zIGlzIEZhbHNlIGFuZCByZXR1cm5fb2Zmc2V0c19tYXBwaW5nIGlzIFRydWUKICAgICAgICByZXR1cm4gc2VsZi5vZmZzZXRzCgoKQGRhdGFjbGFzcwpjbGFzcyBfQ29udGV4dDoKICAgIGFkYXB0ZXI6IG9iamVjdAogICAgYXVkaXQ6IG9iamVjdAogICAga2V5OiBzdHIKICAgIHN0YXJ0OiBpbnQKICAgIG1heF9yb3dzOiBpbnQgfCBOb25lCiAgICBjYWNoZTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCmRlZiBfdG9rZW5pemUocm93LCB0b2tlbml6ZXIsIG1heF9zZXF1ZW5jZV90b2tlbnM9NDA5NjApOgogICAgY29udGV4dCA9IGdldGF0dHIoX0xPQ0FMLCAiY29udGV4dCIsIE5vbmUpCiAgICBpZiAoY29udGV4dCBpcyBub3QgTm9uZSBhbmQgdG9rZW5pemVyIGlzIGNvbnRleHQuYWRhcHRlci50b2tlbml6ZXIKICAgICAgICAgICAgYW5kIG1heF9zZXF1ZW5jZV90b2tlbnMgPT0gY29udGV4dC5hdWRpdC5saW1pdCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjYWNoZWQgPSBjb250ZXh0LmNhY2hlLmdldChfcm93X2J5dGVzKHJvdykpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgIyBDYWNoZSBib29ra2VlcGluZyBpcyBvcHRpb25hbC4gVGhlIG9yaWdpbmFsIGV4YWN0IHJvdyBwYXRoCiAgICAgICAgICAgICMgcmVtYWlucyBhdXRob3JpdGF0aXZlIGlmIGEga2V5IGNhbm5vdCBiZSBwcmVwYXJlZC9sb29rZWQgdXAuCiAgICAgICAgICAgIGNvbnRleHQuYWRhcHRlci5zdGF0c1siY2FjaGVfa2V5X2ZhbGxiYWNrcyJdICs9IDEKICAgICAgICAgICAgY2FjaGVkID0gTm9uZQogICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgY29udGV4dC5hZGFwdGVyLnN0YXRzWyJjYWNoZV9oaXRzIl0gKz0gMQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGNhY2hlZCwgRXhjZXB0aW9uKToKICAgICAgICAgICAgICAgICMgQSBiYXRjaC1zcGVjaWZpYyBpbmNvbnNpc3RlbmN5IGlzIG5vdCBldmlkZW5jZSB0aGF0IHRoZQogICAgICAgICAgICAgICAgIyBvcmlnaW5hbCBzY2FsYXIgcm93IGZhaWxzLiBMZXQgdGhlIG9yaWdpbmFsIGdhdGVzIGRlY2lkZSBhdAogICAgICAgICAgICAgICAgIyB0aGlzIGV4YWN0IHJvdywgaW5jbHVkaW5nIGFueSByZWFsIGRlbGltaXRlci9sZW5ndGggZXJyb3IuCiAgICAgICAgICAgICAgICBjb250ZXh0LmFkYXB0ZXIuc3RhdHNbInNpbmdsZV9yb3dfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICAgICAgcmV0dXJuIF9PUklHSU5BTF9UT0tFTklaRShyb3csIHRva2VuaXplciwgbWF4X3NlcXVlbmNlX3Rva2VucykKICAgICAgICAgICAgIyBDYWxsZXIgb3ducyBpdHMgcmVzdWx0OyBldmVuIGFjY2lkZW50YWwgZWRpdHMgY2Fubm90IHBvaXNvbiBjYWNoZS4KICAgICAgICAgICAgcmV0dXJuIHtrOiBsaXN0KHYpIGlmIGlzaW5zdGFuY2UodiwgbGlzdCkgZWxzZSB2IGZvciBrLCB2IGluIGNhY2hlZC5pdGVtcygpfQogICAgICAgIGNvbnRleHQuYWRhcHRlci5zdGF0c1sic2luZ2xlX3Jvd19mYWxsYmFja3MiXSArPSAxCiAgICByZXR1cm4gX09SSUdJTkFMX1RPS0VOSVpFKHJvdywgdG9rZW5pemVyLCBtYXhfc2VxdWVuY2VfdG9rZW5zKQoKCkBjb250ZXh0bWFuYWdlcgpkZWYgX3BheWxvYWQoZW50cnksIHBhdGgpOgogICAgIyBIYXNoLCBtZW1iZXIsIHByb3RlY3RlZC1uYW1lLCBnemlwL0NSQyBhbmQgc3RyZWFtIGxpZmV0aW1lIGNoZWNrcyBzdGF5CiAgICAjIGVudGlyZWx5IGluc2lkZSB0aGUgb3JpZ2luYWwgYXJjaGl2ZSBwYXlsb2FkIGNvbnRleHQgbWFuYWdlci4KICAgIHdpdGggX09SSUdJTkFMX1BBWUxPQUQoZW50cnksIHBhdGgpIGFzIHN0cmVhbToKICAgICAgICBjb250ZXh0ID0gZ2V0YXR0cihfTE9DQUwsICJjb250ZXh0IiwgTm9uZSkKICAgICAgICBpZiAoY29udGV4dCBpcyBub3QgTm9uZSBhbmQgZW50cnlbInJvbGUiXSA9PSAiY2FuZGlkYXRlX2NoYXQiCiAgICAgICAgICAgICAgICBhbmQgbWVtYmVyX2tleShlbnRyeSkgPT0gY29udGV4dC5rZXkpOgogICAgICAgICAgICB5aWVsZCBjb250ZXh0LmFkYXB0ZXIuX2xpbmVzKHN0cmVhbSwgY29udGV4dCkKICAgICAgICBlbHNlOgogICAgICAgICAgICB5aWVsZCBzdHJlYW0KCgpjbGFzcyBCYXRjaFRva2VuQXVkaXRBZGFwdGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRva2VuaXplciwgYmF0Y2hfc2l6ZT0xMjgsIG1heF9yZW5kZXJlZF9jaGFycz0yXzAwMF8wMDAsCiAgICAgICAgICAgICAgICAgbWF4X3Jhd19jaGFycz0yXzAwMF8wMDApOgogICAgICAgIGltcG9ydCB0cmFuc2Zvcm1lcnMKICAgICAgICBpZiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18gIT0gIjUuMTYuMSI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0IGFkYXB0ZXIgcmVxdWlyZXMgdHJhbnNmb3JtZXJzIDUuMTYuMSIpCiAgICAgICAgcmV2aXNpb24gPSB0b2tlbml6ZXIuaW5pdF9rd2FyZ3MuZ2V0KCJfY29tbWl0X2hhc2giKQogICAgICAgIGlmIHJldmlzaW9uICE9IEwuUkVWSVNJT04gYW5kIFBhdGgodG9rZW5pemVyLm5hbWVfb3JfcGF0aCkubmFtZSAhPSBMLlJFVklTSU9OOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQaW5uZWQgUXdlbjMgdG9rZW5pemVyIHJldmlzaW9uIHJlcXVpcmVkIikKICAgICAgICBpZiBub3QgdG9rZW5pemVyLmlzX2Zhc3Qgb3IgdG9rZW5pemVyLmNvbnZlcnRfdG9rZW5zX3RvX2lkcygiPHxpbV9lbmR8PiIpICE9IDE1MTY0NToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2ZmaWNpYWwgZmFzdCBRd2VuMyB0b2tlbml6ZXIgcmVxdWlyZWQiKQogICAgICAgIGlmIGJhdGNoX3NpemUgbm90IGluICgxMjgsIDI1Nik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX3NpemUgbXVzdCBiZSAxMjggb3IgMjU2IikKICAgICAgICBpZiBtYXhfcmVuZGVyZWRfY2hhcnMgPD0gMCBvciBtYXhfcmF3X2NoYXJzIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBvc2l0aXZlIGNoYXJhY3RlciBib3VuZHMgcmVxdWlyZWQiKQogICAgICAgIHNlbGYudG9rZW5pemVyID0gdG9rZW5pemVyCiAgICAgICAgc2VsZi5iYXRjaF9zaXplID0gYmF0Y2hfc2l6ZQogICAgICAgIHNlbGYubWF4X3JlbmRlcmVkX2NoYXJzID0gbWF4X3JlbmRlcmVkX2NoYXJzCiAgICAgICAgc2VsZi5tYXhfcmF3X2NoYXJzID0gbWF4X3Jhd19jaGFycwogICAgICAgIHNlbGYuc3RhdHMgPSBkaWN0KGJhdGNoZXM9MCwgcm93c19iYXRjaGVkPTAsIGNhY2hlX2hpdHM9MCwKICAgICAgICAgICAgICAgICAgICAgICAgICBzaW5nbGVfcm93X2ZhbGxiYWNrcz0wLCBza2lwcGVkX3Jlc3VtZV9saW5lcz0wLAogICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9jYWNoZV9yb3dzPTAsIG1heF9yZW5kZXJlZF9jaGFycz0wLCBtYXhfcmF3X2NoYXJzPTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgYmF0Y2hfcHJlcGFyZV9mYWxsYmFja3M9MCwgcm93X3ByZXBhcmVfZmFsbGJhY2tzPTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgY2FjaGVfa2V5X2ZhbGxiYWNrcz0wLCBzdHJlYW1fZXJyb3JfYnVmZmVyZWRfcm93cz0wKQoKICAgIGRlZiBpbnN0YWxsKHNlbGYpOgogICAgICAgIGdsb2JhbCBfT1dORVIKICAgICAgICB3aXRoIF9MT0NLOgogICAgICAgICAgICBpZiBfT1dORVIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFuIGFkYXB0ZXIgaXMgYWxyZWFkeSBpbnN0YWxsZWQiKQogICAgICAgICAgICBpZiBMLnRva2VuaXplX2Fzc2lzdGFudF9vbmx5IGlzIG5vdCBfT1JJR0lOQUxfVE9LRU5JWkUgb3IgQS5wYXlsb2FkIGlzIG5vdCBfT1JJR0lOQUxfUEFZTE9BRDoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiT3JpZ2luYWwgdG9rZW5pemVyL3BheWxvYWQgaG9va3MgY2hhbmdlZDsgcmVmdXNlIHJlcGxhY2VtZW50IikKICAgICAgICAgICAgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSwgQS5wYXlsb2FkID0gX3Rva2VuaXplLCBfcGF5bG9hZAogICAgICAgICAgICBfT1dORVIgPSBzZWxmCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdW5pbnN0YWxsKHNlbGYpOgogICAgICAgIGdsb2JhbCBfT1dORVIKICAgICAgICB3aXRoIF9MT0NLOgogICAgICAgICAgICBpZiBfT1dORVIgaXMgbm90IHNlbGY6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoaXMgYWRhcHRlciBpcyBub3QgaW5zdGFsbGVkIikKICAgICAgICAgICAgaWYgX0FDVElWRToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiV2FpdCBmb3IgZXZlcnkgYWRhcHRlciBjb25zdW1lIHRvIGZpbmlzaCBiZWZvcmUgdW5pbnN0YWxsIikKICAgICAgICAgICAgaWYgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSBpcyBub3QgX3Rva2VuaXplIG9yIEEucGF5bG9hZCBpcyBub3QgX3BheWxvYWQ6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkluc3RhbGxlZCBob29rcyBjaGFuZ2VkOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgICAgICAgICAgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSwgQS5wYXlsb2FkID0gX09SSUdJTkFMX1RPS0VOSVpFLCBfT1JJR0lOQUxfUEFZTE9BRAogICAgICAgICAgICBfT1dORVIgPSBOb25lCgogICAgQGNvbnRleHRtYW5hZ2VyCiAgICBkZWYgaW5zdGFsbGVkKHNlbGYpOgogICAgICAgIHNlbGYuaW5zdGFsbCgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB5aWVsZCBzZWxmCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgc2VsZi51bmluc3RhbGwoKQoKICAgIGRlZiBjb25zdW1lKHNlbGYsIGF1ZGl0LCBlbnRyeSwgcGF0aCwgbWF4X3Jvd3M9MjAwMDAsIGNvbW1pdF9ldmVyeT0yNTYpOgogICAgICAgIGdsb2JhbCBfQUNUSVZFCiAgICAgICAgd2l0aCBfTE9DSzoKICAgICAgICAgICAgaWYgX09XTkVSIGlzIG5vdCBzZWxmOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFeHBsaWNpdGx5IGluc3RhbGwgYWRhcHRlciBiZWZvcmUgY29uc3VtZSIpCiAgICAgICAgICAgIGlmIGdldGF0dHIoX0xPQ0FMLCAiY29udGV4dCIsIE5vbmUpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOZXN0ZWQgYWRhcHRlciBjb25zdW1lIGlzIHVuc3VwcG9ydGVkIikKICAgICAgICAgICAgaWYgYXVkaXQudG9rZW5pemVyIGlzIG5vdCBzZWxmLnRva2VuaXplcjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkF1ZGl0IHRva2VuaXplciBtdXN0IGJlIHRoZSBleGFjdCBhZGFwdGVyIHRva2VuaXplciBpbnN0YW5jZSIpCiAgICAgICAgICAgIGN1cnNvciA9IGF1ZGl0LmRiLmV4ZWN1dGUoIlNFTEVDVCBuZXh0X2xpbmUgRlJPTSBtZW1iZXJzIFdIRVJFIGlkPT8iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChtZW1iZXJfa2V5KGVudHJ5KSwpKS5mZXRjaG9uZSgpCiAgICAgICAgICAgIGNvbnRleHQgPSBfQ29udGV4dChzZWxmLCBhdWRpdCwgbWVtYmVyX2tleShlbnRyeSksIGN1cnNvclswXSBpZiBjdXJzb3IgZWxzZSAwLCBtYXhfcm93cykKICAgICAgICAgICAgX0xPQ0FMLmNvbnRleHQgPSBjb250ZXh0CiAgICAgICAgICAgIF9BQ1RJVkUgKz0gMQogICAgICAgIHRyeToKICAgICAgICAgICAgIyBBbGwgc291cmNlL2hhc2gvc3BsaXQvbGlua2FnZS9kdXBsaWNhdGUvY29tbWl0L2N1cnNvciBsb2dpYyBpcyB0aGUKICAgICAgICAgICAgIyBhY3R1YWwgb3JpZ2luYWwgc2Nhbm5lciBtZXRob2QsIGluY2x1ZGluZyBpdHMgb3JpZ2luYWwgZXhjZXB0aW9ucy4KICAgICAgICAgICAgcmV0dXJuIGF1ZGl0LmNvbnN1bWUoZW50cnksIHBhdGgsIG1heF9yb3dzLCBjb21taXRfZXZlcnkpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgICAgIHdpdGggX0xPQ0s6CiAgICAgICAgICAgICAgICBkZWwgX0xPQ0FMLmNvbnRleHQKICAgICAgICAgICAgICAgIF9BQ1RJVkUgLT0gMQoKICAgIGRlZiBfaXRlbShzZWxmLCB0ZXh0KToKICAgICAgICAjIEludmFsaWQgc3ludGF4L3NjaGVtYSBpcyBsZWZ0IHVudG91Y2hlZCBmb3IgdGhlIG9yaWdpbmFsIGxpbmUgcmVhZGVyCiAgICAgICAgIyBhbmQgb3JpZ2luYWwgX2NoYXQgdmFsaWRhdGlvbiwgcHJlc2VydmluZyB0aGVpciBlcnJvciBvcmRlci4KICAgICAgICBpZiBsZW4odGV4dCkgPiBzZWxmLm1heF9yYXdfY2hhcnM6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICByb3cgPSBqc29uLmxvYWRzKHRleHQpCiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdywgZGljdCk6CiAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICBMLnZhbGlkYXRlX2NoYXQocm93KQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yLCBLZXlFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHByZWZpeCA9IHNlbGYudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUocm93WyJtZXNzYWdlcyJdWzoyXSwgdG9rZW5pemU9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICByZW5kZXJlZCA9IHNlbGYudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUocm93WyJtZXNzYWdlcyJdLCB0b2tlbml6ZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9RmFsc2UsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICBpZiBsZW4ocHJlZml4KSArIGxlbihyZW5kZXJlZCkgPiBzZWxmLm1heF9yZW5kZXJlZF9jaGFyczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICByZXR1cm4gKHRleHQsIHJvdywgX3Jvd19ieXRlcyhyb3cpLCBwcmVmaXgsIHJlbmRlcmVkKQoKICAgIGRlZiBfcHJlcGFyZShzZWxmLCBpdGVtcywgY29udGV4dCk6CiAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgaWYgbm90IGl0ZW1zOgogICAgICAgICAgICByZXR1cm4KICAgICAgICAjIFR3byBpbmRlcGVuZGVudCBhY3R1YWwgZmFzdC10b2tlbml6ZXIgYmF0Y2ggY2FsbHM6IG9mZmljaWFsIHRlbXBsYXRlCiAgICAgICAgIyBJRHMgdmVyc3VzIHJlbmRlcmVkIHRleHQgKyBjaGFyYWN0ZXIgb2Zmc2V0cy4gTm8gcGFkZGluZy90cnVuY2F0aW9uLgogICAgICAgIGZ1bGwgPSBzZWxmLnRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKFt4WzFdWyJtZXNzYWdlcyJdIGZvciB4IGluIGl0ZW1zXSwKICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9RmFsc2UsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICBpZHMgPSBmdWxsWyJpbnB1dF9pZHMiXSBpZiBoYXNhdHRyKGZ1bGwsICJrZXlzIikgZWxzZSBmdWxsCiAgICAgICAgb2Zmc2V0cyA9IHNlbGYudG9rZW5pemVyKFt4WzRdIGZvciB4IGluIGl0ZW1zXSwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlLAogICAgICAgICAgICAgICAgICAgIHJldHVybl9vZmZzZXRzX21hcHBpbmc9VHJ1ZSwgcGFkZGluZz1GYWxzZSwgdHJ1bmNhdGlvbj1GYWxzZSkKICAgICAgICBpZiBsZW4oaWRzKSAhPSBsZW4oaXRlbXMpIG9yIGxlbihvZmZzZXRzWyJpbnB1dF9pZHMiXSkgIT0gbGVuKGl0ZW1zKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2ZmaWNpYWwgdG9rZW5pemVyIGJhdGNoIGNhcmRpbmFsaXR5IGRpZmZlcnMiKQogICAgICAgIGZvciBpbmRleCwgaXRlbSBpbiBlbnVtZXJhdGUoaXRlbXMpOgogICAgICAgICAgICBwcmVwYXJlZCA9IF9QcmVwYXJlZFRva2VuaXplcihpdGVtLCBpZHNbaW5kZXhdLAogICAgICAgICAgICAgICAgeyJpbnB1dF9pZHMiOiBvZmZzZXRzWyJpbnB1dF9pZHMiXVtpbmRleF0sCiAgICAgICAgICAgICAgICAgIm9mZnNldF9tYXBwaW5nIjogb2Zmc2V0c1sib2Zmc2V0X21hcHBpbmciXVtpbmRleF19KQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBlbmNvZGVkID0gX09SSUdJTkFMX1RPS0VOSVpFKGl0ZW1bMV0sIHByZXBhcmVkLCBjb250ZXh0LmF1ZGl0LmxpbWl0KQogICAgICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEFzc2VydGlvbkVycm9yKSBhcyBlcnJvcjoKICAgICAgICAgICAgICAgICMgRGVmZXIgdGhlIGV4YWN0IG9yaWdpbmFsIGdhdGUgZmFpbHVyZSB1bnRpbCBfY2hhdCByZWFjaGVzCiAgICAgICAgICAgICAgICAjIHRoaXMgcm93OyBwcmlvciByb3dzIGtlZXAgdGhlaXIgb3JpZ2luYWwgY29tbWl0IGJlaGF2aW9yLgogICAgICAgICAgICAgICAgZW5jb2RlZCA9IGVycm9yLndpdGhfdHJhY2ViYWNrKE5vbmUpCiAgICAgICAgICAgIGNvbnRleHQuY2FjaGVbaXRlbVsyXV0gPSBlbmNvZGVkCiAgICAgICAgc2VsZi5zdGF0c1siYmF0Y2hlcyJdICs9IDEKICAgICAgICBzZWxmLnN0YXRzWyJyb3dzX2JhdGNoZWQiXSArPSBsZW4oaXRlbXMpCiAgICAgICAgc2VsZi5zdGF0c1sibWF4X2NhY2hlX3Jvd3MiXSA9IG1heChzZWxmLnN0YXRzWyJtYXhfY2FjaGVfcm93cyJdLCBsZW4oY29udGV4dC5jYWNoZSkpCiAgICAgICAgc2VsZi5zdGF0c1sibWF4X3JlbmRlcmVkX2NoYXJzIl0gPSBtYXgoc2VsZi5zdGF0c1sibWF4X3JlbmRlcmVkX2NoYXJzIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdW0obGVuKHhbM10pICsgbGVuKHhbNF0pIGZvciB4IGluIGl0ZW1zKSkKICAgICAgICBzZWxmLnN0YXRzWyJtYXhfcmF3X2NoYXJzIl0gPSBtYXgoc2VsZi5zdGF0c1sibWF4X3Jhd19jaGFycyJdLCBzdW0obGVuKHhbMF0pIGZvciB4IGluIGl0ZW1zKSkKCiAgICBkZWYgX2xpbmVzKHNlbGYsIHN0cmVhbSwgY29udGV4dCk6CiAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAoKICAgICAgICBkZWYgZmx1c2gocHJlcGFyZT1UcnVlKToKICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgICAgIGlmIHByZXBhcmU6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5fcHJlcGFyZShiYXRjaCwgY29udGV4dCkKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgIyBBIGJhdGNoLW9ubHkgYmFja2VuZC9jYXJkaW5hbGl0eS9pbmRleCBmYWlsdXJlIG11c3Qgbm90CiAgICAgICAgICAgICAgICAgICAgIyBtb3ZlIGEgcm93J3Mgb3JpZ2luYWwgZXJyb3IvY29tbWl0IGJvdW5kYXJ5LiBEaXNjYXJkIGV2ZW4KICAgICAgICAgICAgICAgICAgICAjIGEgcGFydGlhbGx5IHByZXBhcmVkIGNhY2hlIGFuZCBydW4gZXZlcnkgb3JpZ2luYWwgcm93LgogICAgICAgICAgICAgICAgICAgIGNvbnRleHQuY2FjaGUuY2xlYXIoKQogICAgICAgICAgICAgICAgICAgIHNlbGYuc3RhdHNbImJhdGNoX3ByZXBhcmVfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICBmb3IgaXRlbSBpbiBiYXRjaDoKICAgICAgICAgICAgICAgIHlpZWxkIGl0ZW1bMF0KICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCgogICAgICAgIHRyeToKICAgICAgICAgICAgaXRlcmF0b3IsIG51bWJlciA9IGl0ZXIoc3RyZWFtKSwgMAogICAgICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHRleHQgPSBuZXh0KGl0ZXJhdG9yKQogICAgICAgICAgICAgICAgZXhjZXB0IFN0b3BJdGVyYXRpb246CiAgICAgICAgICAgICAgICAgICAgaWYgYmF0Y2g6CiAgICAgICAgICAgICAgICAgICAgICAgIHlpZWxkIGZyb20gZmx1c2goKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgICMgUmVhZGluZyBhaGVhZCBjYW4gaGl0IFVuaWNvZGUvQ1JDL3RyYWlsZXIvSS9PIGZhaWx1cmVzLgogICAgICAgICAgICAgICAgICAgICMgRGVsaXZlciBldmVyeSBwcmV2aW91c2x5IHJlYWQgcGh5c2ljYWwgcm93IHZpYSB0aGUgZXhhY3QKICAgICAgICAgICAgICAgICAgICAjIG9yaWdpbmFsIHBhdGggYmVmb3JlIHN1cmZhY2luZyB0aGlzIHN0cmVhbSBleGNlcHRpb24uCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sic3RyZWFtX2Vycm9yX2J1ZmZlcmVkX3Jvd3MiXSArPSBsZW4oYmF0Y2gpCiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaChwcmVwYXJlPUZhbHNlKQogICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICAgICAgICAgICAgICBudW1iZXIgKz0gMQogICAgICAgICAgICAgICAgaWYgbnVtYmVyIDw9IGNvbnRleHQuc3RhcnQ6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sic2tpcHBlZF9yZXN1bWVfbGluZXMiXSArPSAxCiAgICAgICAgICAgICAgICAgICAgeWllbGQgdGV4dCAgIyBPcmlnaW5hbCBlbnVtZXJhdGUvc2tpcCByZXRhaW5zIHBoeXNpY2FsIGN1cnNvci4KICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgY29udGV4dC5tYXhfcm93cyBpcyBub3QgTm9uZSBhbmQgbnVtYmVyID4gY29udGV4dC5zdGFydCArIGNvbnRleHQubWF4X3Jvd3M6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIk9yaWdpbmFsIGNvbnN1bWUgcmVhZCBiZXlvbmQgaXRzIG93biB3b3JrIGJvdW5kIikKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBpdGVtID0gc2VsZi5faXRlbSh0ZXh0KQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICAjIFJlbmRlcmluZy9jYW5vbmljYWxpemF0aW9uIGFoZWFkIG9mIF9jaGF0IGlzIHNwZWN1bGF0aXZlOwogICAgICAgICAgICAgICAgICAgICMgZmFsbGJhY2sgcHJlc2VydmVzIGR1cGxpY2F0ZSBzaG9ydC1jaXJjdWl0cyBhbmQgbGV0cyB0aGUKICAgICAgICAgICAgICAgICAgICAjIHJlYWwgb3JpZ2luYWwgY2FsbCByYWlzZSBvbmx5IGF0IGl0cyBwaHlzaWNhbCByb3cuCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sicm93X3ByZXBhcmVfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICAgICAgICAgIGl0ZW0gPSBOb25lCiAgICAgICAgICAgICAgICBzaXplID0gbGVuKGl0ZW1bM10pICsgbGVuKGl0ZW1bNF0pIGlmIGl0ZW0gZWxzZSAwCiAgICAgICAgICAgICAgICBpZiAoYmF0Y2ggYW5kIChpdGVtIGlzIE5vbmUgb3IgbGVuKGJhdGNoKSA+PSBzZWxmLmJhdGNoX3NpemUKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3IgcmF3X2NoYXJzICsgbGVuKHRleHQpID4gc2VsZi5tYXhfcmF3X2NoYXJzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9yIHJlbmRlcmVkX2NoYXJzICsgc2l6ZSA+IHNlbGYubWF4X3JlbmRlcmVkX2NoYXJzKSk6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaCgpCiAgICAgICAgICAgICAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAogICAgICAgICAgICAgICAgaWYgaXRlbSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHlpZWxkIHRleHQgICMgRXhhY3Qgb3JpZ2luYWwgZmFsbGJhY2s7IG5ldmVyIGRyb3AgYSBsaW5lLgogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBiYXRjaC5hcHBlbmQoaXRlbSkKICAgICAgICAgICAgICAgICAgICByYXdfY2hhcnMgKz0gbGVuKHRleHQpCiAgICAgICAgICAgICAgICAgICAgcmVuZGVyZWRfY2hhcnMgKz0gc2l6ZQogICAgICAgICAgICAgICAgaWYgKGxlbihiYXRjaCkgPT0gc2VsZi5iYXRjaF9zaXplIG9yCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRleHQubWF4X3Jvd3MgaXMgbm90IE5vbmUgYW5kIG51bWJlciA9PSBjb250ZXh0LnN0YXJ0ICsgY29udGV4dC5tYXhfcm93cyk6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaCgpCiAgICAgICAgICAgICAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGNvbnRleHQuY2FjaGUuY2xlYXIoKQo=')
assert hashlib.sha256(adapter_bytes).hexdigest()=='8d978eceeabebda9103ae23a16c23beab5b8cf5f2248984eda0b584facbedf92'
adapter_path=ROOT/'math_data_batch_token_audit.py'
adapter_path.write_bytes(adapter_bytes)
adapter_spec=importlib.util.spec_from_file_location('math_data_batch_token_audit',adapter_path)
batch_module=importlib.util.module_from_spec(adapter_spec)
sys.modules[adapter_spec.name]=batch_module
adapter_spec.loader.exec_module(batch_module)
batch_adapter=batch_module.BatchTokenAuditAdapter(tokenizer,batch_size=256)

# Attach to the existing, fully raw-verified V3 checkpoint; never create a file.
import threading,time,uuid,shutil
from datetime import datetime,timezone
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
assert os.environ.get('COLAB_GPU','0') in ('0','')
if any(t.name.startswith('MathVisionKidFiniteCPUDataAudit') and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError('A data audit worker is already active; do not duplicate it.')
assert probe_receipt['state']=='EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY'
assert probe_receipt['fresh_raw_readback_verified']
V3_DB=RESTORED_V3_DB
V3_ID=probe_receipt['audit_identity']
recipe_all=recipe;pointer_all=recipe_pointer
assert audit_v3.plan_identity(recipe_all)==V3_ID
assert len(recipe_all['source_members'])==218 and len(recipe_all['chat_members'])==31
def file_digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
assert file_digest(V3_DB)==probe_receipt['checkpoint_sha256']
if shutil.disk_usage(ROOT).free<3*V3_DB.stat().st_size+1024**3:
    raise RuntimeError('Insufficient temporary Colab disk for data checkpoint snapshots.')
fresh=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert fresh==meta_after,'Checkpoint changed after read-only restore; refuse a stale writer.'
from exhaustive_colab_loader import load_qwen_tokenizer
assert tokenizer is batch_adapter.tokenizer  # preserve validated tokenizer instance; no model weights
preclaim=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert preclaim==fresh,'Checkpoint advanced while loading tokenizer; refuse a stale writer.'
writer_session=uuid.uuid4().hex
claim=dict(fresh['appProperties'],writer_session=writer_session)
v3_ref=drive_api.files().update(fileId=EXISTING_V3_ID,body={'appProperties':claim},fields='id,size,appProperties',supportsAllDrives=True).execute()
assert v3_ref['id']==EXISTING_V3_ID
assert v3_ref['appProperties']['data_sha256']==probe_receipt['checkpoint_sha256']
worker_status_path=ROOT/'full_audit_v3_worker_status.json'
full_audit_worker_state={'state':'RESUMING_EXISTING_CHECKPOINT','started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l','audit_identity':V3_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':EXISTING_V3_ID,'writer_session':writer_session,'writer_guard':'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS','restored_parent':probe_receipt,'training_started':False,'gpu_used':False,'model_weights_loaded':False}
full_audit_worker_state['checkpoint']=upgrade_proof
full_audit_worker_state['performance_upgrade']={'adapter_sha256':hashlib.sha256(adapter_bytes).hexdigest(),'validation_tests':21,'checkpoint_interval_seconds':1800,'original_auditor_unchanged':True,'source_rows_changed':False,'thread_local_batch_size':256}

def write_worker_status():
    full_audit_worker_state['batch_adapter_stats']=dict(batch_adapter.stats)
    full_audit_worker_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    tmp=worker_status_path.with_suffix('.writing')
    tmp.write_text(json.dumps(full_audit_worker_state,ensure_ascii=False,indent=2),encoding='utf8')
    tmp.replace(worker_status_path)

def finite_full_data_audit_worker_v3():
    global v3_ref
    auditor=None;checkpoint=dict(v3_ref);last_save=0.0
    def verify_writer():
        remote=work_api.files().get(fileId=checkpoint['id'],fields='id,size,appProperties',supportsAllDrives=True).execute()
        props=remote.get('appProperties',{})
        if props.get('writer_session')!=writer_session or props.get('recipe_identity')!=V3_ID or props.get('data_sha256')!=checkpoint.get('appProperties',{}).get('data_sha256'):
            raise RuntimeError('Checkpoint writer or hash changed; stop without overwriting.')
        return remote
    def save_checkpoint():
        nonlocal checkpoint,last_save
        global v3_ref
        auditor.db.commit();snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
        target_db=sqlite3.connect(snapshot)
        try:auditor.db.backup(target_db)
        finally:target_db.close()
        snapshot_sha=file_digest(snapshot)
        verify_writer()
        props={'recipe_identity':V3_ID,'data_sha256':snapshot_sha,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING','writer_session':writer_session}
        saved=work_api.files().update(fileId=checkpoint['id'],body={'appProperties':props},media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
        # Retain the completed write identity even if the verification read fails.
        # It does not become a verified receipt until all raw bytes match.
        checkpoint=saved;v3_ref=saved
        class HashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):self.hasher.update(data);self.size+=len(data);return len(data)
        sink=HashSink();download=MediaIoBaseDownload(sink,work_api.files().get_media(fileId=saved['id'],supportsAllDrives=True),chunksize=8*1024*1024)
        done=False
        while not done:_,done=download.next_chunk(num_retries=3)
        if sink.hasher.hexdigest()!=snapshot_sha or sink.size!=snapshot.stat().st_size:
            raise RuntimeError('V3 raw checkpoint readback differs.')
        after=verify_writer()
        last_save=time.monotonic()
        proof={'id':saved['id'],'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True,'report_at_snapshot':auditor.report(),'observed_utc':datetime.now(timezone.utc).isoformat()}
        full_audit_worker_state['checkpoint']=proof
        (ROOT/'v3_latest_checkpoint_ref.json').write_text(json.dumps({'metadata':after,'proof':proof},ensure_ascii=False,indent=2),encoding='utf8')
    try:
        work_api=build('drive','v3',credentials=credentials,cache_discovery=False)
        work_reader=GoogleDriveReader(work_api)
        work_cache=LazyDriveCache(work_reader,recipe_all,ROOT/'full_audit_v3_worker_payload_cache')
        auditor=audit_v3.FullArchiveAudit(V3_DB,recipe_all,tokenizer)
        auditor.db.execute('PRAGMA cache_size=-131072')
        full_audit_worker_state['report']=auditor.report();write_worker_status()
        last_save=time.monotonic()
        full_audit_worker_state['state']='RUNNING_DATA_ONLY';write_worker_status()
        while True:
            entry=auditor.next_entry()
            if entry is None:break
            verify_writer()
            full_audit_worker_state['current_member']={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']}
            write_worker_status()
            span=batch_adapter.consume(auditor,entry,work_cache[entry['sha256']],max_rows=50000,commit_every=256)
            full_audit_worker_state.update(last_span=span,report=auditor.report(),current_member={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']})
            if not span.get('processed',0) and not span.get('complete') and not span.get('already_complete'):
                raise RuntimeError('No finite cursor progress; refuse an endless loop.')
            if 'checkpoint' not in full_audit_worker_state or time.monotonic()-last_save>=1800:save_checkpoint()
            write_worker_status()
        save_checkpoint();report=auditor.report()
        full_audit_worker_state.update(report=report,state='AUDIT_PASS_PENDING_CURRICULUM_ACCEPTANCE' if report['full_archive_token_linkage_split_audit_complete'] else 'AUDIT_FINISHED_WITH_FINDINGS')
    except Exception as exc:
        full_audit_worker_state.update(state='STOPPED_REQUIRES_AUDIT_REPAIR',error_type=type(exc).__name__,error=str(exc))
        if auditor is not None:
            full_audit_worker_state['report']=auditor.report()
            try:save_checkpoint()
            except Exception as checkpoint_error:full_audit_worker_state['checkpoint_save_error']=str(checkpoint_error)
    finally:
        if auditor is not None:auditor.close()
        batch_adapter.uninstall()
        write_worker_status()
write_worker_status()
full_audit_worker=threading.Thread(target=finite_full_data_audit_worker_v3,name='MathVisionKidFiniteCPUDataAuditV3',daemon=True)
batch_adapter.install()
full_audit_worker.start()
print(full_audit_worker_state)


ValueError: Pinned Qwen3 tokenizer revision required

In [ ]:
print({'tokenizer_class':type(tokenizer).__name__,'tokenizer_name_or_path':tokenizer.name_or_path,'revision':tokenizer.init_kwargs.get('revision'),'commit_hash':tokenizer.init_kwargs.get('_commit_hash'),'expected_revision':batch_module.L.REVISION,'fast':tokenizer.is_fast,'eot_id':tokenizer.convert_tokens_to_ids('<|im_end|>'),'training_started':False})

{'tokenizer_class': 'Qwen2Tokenizer', 'tokenizer_name_or_path': 'Qwen/Qwen3-1.7B', 'revision': None, 'commit_hash': None, 'expected_revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e', 'fast': True, 'eot_id': 151645, 'training_started': False}


In [ ]:
# Resume from the latest full raw-verified Colab snapshot, never re-download to PC.
import os,json,hashlib,sqlite3,shutil,threading,base64,importlib.util,sys
from pathlib import Path
assert os.environ.get('COLAB_GPU','0') in ('0','')
full_audit_worker.join(timeout=0)
assert not full_audit_worker.is_alive(),'Wait for cooperative checkpoint pause; do not interrupt worker.'
assert full_audit_worker_state['state']=='PAUSED_AT_VERIFIED_CHECKPOINT_FOR_VALIDATED_PERFORMANCE_UPGRADE'
assert full_audit_worker_state['pause_receipt']['committed_members_equal_verified_snapshot']
upgrade_parent_state=json.loads(json.dumps(full_audit_worker_state))
upgrade_proof=upgrade_parent_state['checkpoint']
assert upgrade_proof['fresh_raw_readback_verified']
upgrade_snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
assert upgrade_snapshot.stat().st_size==upgrade_proof['bytes']
assert file_digest(upgrade_snapshot)==upgrade_proof['sha256']
upgrade_remote=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert upgrade_remote['appProperties']['data_sha256']==upgrade_proof['sha256']
assert upgrade_remote['appProperties']['writer_session']==upgrade_parent_state['writer_session']
assert int(upgrade_remote['size'])==upgrade_proof['bytes']
upgrade_db=ROOT/('batch_resume_'+upgrade_proof['sha256'][:16]+'.sqlite')
assert upgrade_db!=upgrade_snapshot
assert shutil.disk_usage(ROOT).free>4*upgrade_proof['bytes']+1024**3
if not upgrade_db.exists():
    shutil.copyfile(upgrade_snapshot,upgrade_db)
assert file_digest(upgrade_db)==upgrade_proof['sha256']
with sqlite3.connect(upgrade_db.as_uri()+'?mode=ro',uri=True) as upgrade_check:
    assert upgrade_check.execute('PRAGMA quick_check').fetchall()==[('ok',)]
    upgrade_cursors=upgrade_check.execute('SELECT id,role,next_line,complete,expected_rows FROM members ORDER BY id').fetchall()
    assert hashlib.sha256(json.dumps(upgrade_cursors,separators=(',',':')).encode()).hexdigest()==upgrade_parent_state['pause_receipt']['member_cursors_sha256']
    assert upgrade_check.execute('SELECT count(*) FROM samples').fetchone()[0]==upgrade_proof['report_at_snapshot']['unique_chat_rows_processed']
    assert upgrade_check.execute('SELECT count(*) FROM source_versions').fetchone()[0]==upgrade_proof['report_at_snapshot']['source_versions_indexed']
RESTORED_V3_DB=upgrade_db
probe_receipt={'state':'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY','fresh_raw_readback_verified':True,
    'checkpoint_id':EXISTING_V3_ID,'checkpoint_sha256':upgrade_proof['sha256'],
    'checkpoint_bytes':upgrade_proof['bytes'],'audit_identity':recipe['identity'] if 'identity' in recipe else audit_v3.plan_identity(recipe),
    'recipe_sha256':recipe_pointer['sha256'],'local_verified_snapshot_reused':True,
    'source_versions_indexed':upgrade_proof['report_at_snapshot']['source_versions_indexed'],
    'unique_chat_rows_processed':upgrade_proof['report_at_snapshot']['unique_chat_rows_processed'],
    'old_checkpoint_evidence_preserved':True,'training_started':False}
meta_after=upgrade_remote
audit_v3.FullArchiveAudit.consume=_mathvision_original_consume
_mathvision_pause_requested=False
adapter_bytes=base64.b64decode('IiIiT3B0LWluLCBib3VuZGVkIGV4YWN0IHRva2VuaXplciBiYXRjaGluZyBmb3IgdGhlIHVuY2hhbmdlZCB2MyBDUFUgc2Nhbm5lci4KCkFmdGVyIGFuIG9yZGVybHkgc2Nhbm5lciBwYXVzZTo6CgogICAgYWRhcHRlciA9IEJhdGNoVG9rZW5BdWRpdEFkYXB0ZXIoYXVkaXQudG9rZW5pemVyLCBiYXRjaF9zaXplPTEyOCkKICAgIHdpdGggYWRhcHRlci5pbnN0YWxsZWQoKToKICAgICAgICByZXN1bHQgPSBhZGFwdGVyLmNvbnN1bWUoYXVkaXQsIGVudHJ5LCBsb2NhbF9wYXRoLCBtYXhfcm93cz0yMDAwMCkKCk9ubHkgY2FsbHMgbWFkZSBieSBgYGFkYXB0ZXIuY29uc3VtZWBgIGluIGl0cyBjdXJyZW50IHRocmVhZCB1c2UgdGhlIGNhY2hlLgpPdGhlciBjYWxsZXJzIHJldGFpbiB0aGUgb3JpZ2luYWwgZXhhY3QgdG9rZW5pemVyL3BheWxvYWQgZnVuY3Rpb25zLiBObyBkYXRhLApyZWNpcGUsIHNvdXJjZSBsaW5rYWdlLCBzY2FubmVyIGltcGxlbWVudGF0aW9uLCBvciBjaGVja3BvaW50IHNjaGVtYSBjaGFuZ2VzLgpPdmVyc2l6ZWQgaW5kaXZpZHVhbCByb3dzIHVzZSB0aGUgb3JpZ2luYWwgZXhhY3Qgc2luZ2xlLXJvdyBpbXBsZW1lbnRhdGlvbi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29udGV4dGxpYiBpbXBvcnQgY29udGV4dG1hbmFnZXIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGpzb24KaW1wb3J0IHRocmVhZGluZwoKaW1wb3J0IGV4aGF1c3RpdmVfY29sYWJfbG9hZGVyIGFzIEwKaW1wb3J0IG1hdGhfZGF0YV9hcmNoaXZlX2F1ZGl0IGFzIEEKZnJvbSBtYXRoX2RhdGFfY2FuZGlkYXRlX3N0cmVhbSBpbXBvcnQgbWVtYmVyX2tleQoKX09SSUdJTkFMX1RPS0VOSVpFID0gTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seQpfT1JJR0lOQUxfUEFZTE9BRCA9IEEucGF5bG9hZApfTE9DQUwgPSB0aHJlYWRpbmcubG9jYWwoKQpfTE9DSyA9IHRocmVhZGluZy5STG9jaygpCl9PV05FUiA9IE5vbmUKX0FDVElWRSA9IDAKCgpkZWYgX3Jvd19ieXRlcyhyb3cpOgogICAgcmV0dXJuIEwuY2Fub25pY2FsX2pzb24ocm93KS5lbmNvZGUoInV0Zi04IikKCgpjbGFzcyBfUHJlcGFyZWRUb2tlbml6ZXI6CiAgICAiIiJGZWVkcyBleGFjdCBvZmZpY2lhbCBiYXRjaCBvdXRwdXRzIHRocm91Z2ggYWxsIG9yaWdpbmFsIHJvdyBnYXRlcy4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpdGVtLCBmdWxsLCBvZmZzZXRzKToKICAgICAgICBzZWxmLml0ZW0sIHNlbGYuZnVsbCwgc2VsZi5vZmZzZXRzID0gaXRlbSwgZnVsbCwgb2Zmc2V0cwoKICAgIGRlZiBhcHBseV9jaGF0X3RlbXBsYXRlKHNlbGYsIG1lc3NhZ2VzLCAqLCB0b2tlbml6ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5hYmxlX3RoaW5raW5nKToKICAgICAgICBhc3NlcnQgZW5hYmxlX3RoaW5raW5nIGlzIEZhbHNlCiAgICAgICAgaWYgdG9rZW5pemU6CiAgICAgICAgICAgIGFzc2VydCBtZXNzYWdlcyA9PSBzZWxmLml0ZW1bMV1bIm1lc3NhZ2VzIl0gYW5kIG5vdCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQKICAgICAgICAgICAgcmV0dXJuIHNlbGYuZnVsbAogICAgICAgIGlmIGFkZF9nZW5lcmF0aW9uX3Byb21wdDoKICAgICAgICAgICAgYXNzZXJ0IG1lc3NhZ2VzID09IHNlbGYuaXRlbVsxXVsibWVzc2FnZXMiXVs6Ml0KICAgICAgICAgICAgcmV0dXJuIHNlbGYuaXRlbVszXQogICAgICAgIGFzc2VydCBtZXNzYWdlcyA9PSBzZWxmLml0ZW1bMV1bIm1lc3NhZ2VzIl0KICAgICAgICByZXR1cm4gc2VsZi5pdGVtWzRdCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIHJlbmRlcmVkLCAqLCBhZGRfc3BlY2lhbF90b2tlbnMsIHJldHVybl9vZmZzZXRzX21hcHBpbmcpOgogICAgICAgIGFzc2VydCByZW5kZXJlZCA9PSBzZWxmLml0ZW1bNF0KICAgICAgICBhc3NlcnQgYWRkX3NwZWNpYWxfdG9rZW5zIGlzIEZhbHNlIGFuZCByZXR1cm5fb2Zmc2V0c19tYXBwaW5nIGlzIFRydWUKICAgICAgICByZXR1cm4gc2VsZi5vZmZzZXRzCgoKQGRhdGFjbGFzcwpjbGFzcyBfQ29udGV4dDoKICAgIGFkYXB0ZXI6IG9iamVjdAogICAgYXVkaXQ6IG9iamVjdAogICAga2V5OiBzdHIKICAgIHN0YXJ0OiBpbnQKICAgIG1heF9yb3dzOiBpbnQgfCBOb25lCiAgICBjYWNoZTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCmRlZiBfdG9rZW5pemUocm93LCB0b2tlbml6ZXIsIG1heF9zZXF1ZW5jZV90b2tlbnM9NDA5NjApOgogICAgY29udGV4dCA9IGdldGF0dHIoX0xPQ0FMLCAiY29udGV4dCIsIE5vbmUpCiAgICBpZiAoY29udGV4dCBpcyBub3QgTm9uZSBhbmQgdG9rZW5pemVyIGlzIGNvbnRleHQuYWRhcHRlci50b2tlbml6ZXIKICAgICAgICAgICAgYW5kIG1heF9zZXF1ZW5jZV90b2tlbnMgPT0gY29udGV4dC5hdWRpdC5saW1pdCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjYWNoZWQgPSBjb250ZXh0LmNhY2hlLmdldChfcm93X2J5dGVzKHJvdykpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgIyBDYWNoZSBib29ra2VlcGluZyBpcyBvcHRpb25hbC4gVGhlIG9yaWdpbmFsIGV4YWN0IHJvdyBwYXRoCiAgICAgICAgICAgICMgcmVtYWlucyBhdXRob3JpdGF0aXZlIGlmIGEga2V5IGNhbm5vdCBiZSBwcmVwYXJlZC9sb29rZWQgdXAuCiAgICAgICAgICAgIGNvbnRleHQuYWRhcHRlci5zdGF0c1siY2FjaGVfa2V5X2ZhbGxiYWNrcyJdICs9IDEKICAgICAgICAgICAgY2FjaGVkID0gTm9uZQogICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgY29udGV4dC5hZGFwdGVyLnN0YXRzWyJjYWNoZV9oaXRzIl0gKz0gMQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGNhY2hlZCwgRXhjZXB0aW9uKToKICAgICAgICAgICAgICAgICMgQSBiYXRjaC1zcGVjaWZpYyBpbmNvbnNpc3RlbmN5IGlzIG5vdCBldmlkZW5jZSB0aGF0IHRoZQogICAgICAgICAgICAgICAgIyBvcmlnaW5hbCBzY2FsYXIgcm93IGZhaWxzLiBMZXQgdGhlIG9yaWdpbmFsIGdhdGVzIGRlY2lkZSBhdAogICAgICAgICAgICAgICAgIyB0aGlzIGV4YWN0IHJvdywgaW5jbHVkaW5nIGFueSByZWFsIGRlbGltaXRlci9sZW5ndGggZXJyb3IuCiAgICAgICAgICAgICAgICBjb250ZXh0LmFkYXB0ZXIuc3RhdHNbInNpbmdsZV9yb3dfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICAgICAgcmV0dXJuIF9PUklHSU5BTF9UT0tFTklaRShyb3csIHRva2VuaXplciwgbWF4X3NlcXVlbmNlX3Rva2VucykKICAgICAgICAgICAgIyBDYWxsZXIgb3ducyBpdHMgcmVzdWx0OyBldmVuIGFjY2lkZW50YWwgZWRpdHMgY2Fubm90IHBvaXNvbiBjYWNoZS4KICAgICAgICAgICAgcmV0dXJuIHtrOiBsaXN0KHYpIGlmIGlzaW5zdGFuY2UodiwgbGlzdCkgZWxzZSB2IGZvciBrLCB2IGluIGNhY2hlZC5pdGVtcygpfQogICAgICAgIGNvbnRleHQuYWRhcHRlci5zdGF0c1sic2luZ2xlX3Jvd19mYWxsYmFja3MiXSArPSAxCiAgICByZXR1cm4gX09SSUdJTkFMX1RPS0VOSVpFKHJvdywgdG9rZW5pemVyLCBtYXhfc2VxdWVuY2VfdG9rZW5zKQoKCkBjb250ZXh0bWFuYWdlcgpkZWYgX3BheWxvYWQoZW50cnksIHBhdGgpOgogICAgIyBIYXNoLCBtZW1iZXIsIHByb3RlY3RlZC1uYW1lLCBnemlwL0NSQyBhbmQgc3RyZWFtIGxpZmV0aW1lIGNoZWNrcyBzdGF5CiAgICAjIGVudGlyZWx5IGluc2lkZSB0aGUgb3JpZ2luYWwgYXJjaGl2ZSBwYXlsb2FkIGNvbnRleHQgbWFuYWdlci4KICAgIHdpdGggX09SSUdJTkFMX1BBWUxPQUQoZW50cnksIHBhdGgpIGFzIHN0cmVhbToKICAgICAgICBjb250ZXh0ID0gZ2V0YXR0cihfTE9DQUwsICJjb250ZXh0IiwgTm9uZSkKICAgICAgICBpZiAoY29udGV4dCBpcyBub3QgTm9uZSBhbmQgZW50cnlbInJvbGUiXSA9PSAiY2FuZGlkYXRlX2NoYXQiCiAgICAgICAgICAgICAgICBhbmQgbWVtYmVyX2tleShlbnRyeSkgPT0gY29udGV4dC5rZXkpOgogICAgICAgICAgICB5aWVsZCBjb250ZXh0LmFkYXB0ZXIuX2xpbmVzKHN0cmVhbSwgY29udGV4dCkKICAgICAgICBlbHNlOgogICAgICAgICAgICB5aWVsZCBzdHJlYW0KCgpjbGFzcyBCYXRjaFRva2VuQXVkaXRBZGFwdGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRva2VuaXplciwgYmF0Y2hfc2l6ZT0xMjgsIG1heF9yZW5kZXJlZF9jaGFycz0yXzAwMF8wMDAsCiAgICAgICAgICAgICAgICAgbWF4X3Jhd19jaGFycz0yXzAwMF8wMDApOgogICAgICAgIGltcG9ydCB0cmFuc2Zvcm1lcnMKICAgICAgICBpZiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18gIT0gIjUuMTYuMSI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0IGFkYXB0ZXIgcmVxdWlyZXMgdHJhbnNmb3JtZXJzIDUuMTYuMSIpCiAgICAgICAgcmV2aXNpb24gPSB0b2tlbml6ZXIuaW5pdF9rd2FyZ3MuZ2V0KCJfY29tbWl0X2hhc2giKQogICAgICAgIGlmIHJldmlzaW9uICE9IEwuUkVWSVNJT04gYW5kIFBhdGgodG9rZW5pemVyLm5hbWVfb3JfcGF0aCkubmFtZSAhPSBMLlJFVklTSU9OOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQaW5uZWQgUXdlbjMgdG9rZW5pemVyIHJldmlzaW9uIHJlcXVpcmVkIikKICAgICAgICBpZiBub3QgdG9rZW5pemVyLmlzX2Zhc3Qgb3IgdG9rZW5pemVyLmNvbnZlcnRfdG9rZW5zX3RvX2lkcygiPHxpbV9lbmR8PiIpICE9IDE1MTY0NToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2ZmaWNpYWwgZmFzdCBRd2VuMyB0b2tlbml6ZXIgcmVxdWlyZWQiKQogICAgICAgIGlmIGJhdGNoX3NpemUgbm90IGluICgxMjgsIDI1Nik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX3NpemUgbXVzdCBiZSAxMjggb3IgMjU2IikKICAgICAgICBpZiBtYXhfcmVuZGVyZWRfY2hhcnMgPD0gMCBvciBtYXhfcmF3X2NoYXJzIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBvc2l0aXZlIGNoYXJhY3RlciBib3VuZHMgcmVxdWlyZWQiKQogICAgICAgIHNlbGYudG9rZW5pemVyID0gdG9rZW5pemVyCiAgICAgICAgc2VsZi5iYXRjaF9zaXplID0gYmF0Y2hfc2l6ZQogICAgICAgIHNlbGYubWF4X3JlbmRlcmVkX2NoYXJzID0gbWF4X3JlbmRlcmVkX2NoYXJzCiAgICAgICAgc2VsZi5tYXhfcmF3X2NoYXJzID0gbWF4X3Jhd19jaGFycwogICAgICAgIHNlbGYuc3RhdHMgPSBkaWN0KGJhdGNoZXM9MCwgcm93c19iYXRjaGVkPTAsIGNhY2hlX2hpdHM9MCwKICAgICAgICAgICAgICAgICAgICAgICAgICBzaW5nbGVfcm93X2ZhbGxiYWNrcz0wLCBza2lwcGVkX3Jlc3VtZV9saW5lcz0wLAogICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9jYWNoZV9yb3dzPTAsIG1heF9yZW5kZXJlZF9jaGFycz0wLCBtYXhfcmF3X2NoYXJzPTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgYmF0Y2hfcHJlcGFyZV9mYWxsYmFja3M9MCwgcm93X3ByZXBhcmVfZmFsbGJhY2tzPTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgY2FjaGVfa2V5X2ZhbGxiYWNrcz0wLCBzdHJlYW1fZXJyb3JfYnVmZmVyZWRfcm93cz0wKQoKICAgIGRlZiBpbnN0YWxsKHNlbGYpOgogICAgICAgIGdsb2JhbCBfT1dORVIKICAgICAgICB3aXRoIF9MT0NLOgogICAgICAgICAgICBpZiBfT1dORVIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFuIGFkYXB0ZXIgaXMgYWxyZWFkeSBpbnN0YWxsZWQiKQogICAgICAgICAgICBpZiBMLnRva2VuaXplX2Fzc2lzdGFudF9vbmx5IGlzIG5vdCBfT1JJR0lOQUxfVE9LRU5JWkUgb3IgQS5wYXlsb2FkIGlzIG5vdCBfT1JJR0lOQUxfUEFZTE9BRDoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiT3JpZ2luYWwgdG9rZW5pemVyL3BheWxvYWQgaG9va3MgY2hhbmdlZDsgcmVmdXNlIHJlcGxhY2VtZW50IikKICAgICAgICAgICAgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSwgQS5wYXlsb2FkID0gX3Rva2VuaXplLCBfcGF5bG9hZAogICAgICAgICAgICBfT1dORVIgPSBzZWxmCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdW5pbnN0YWxsKHNlbGYpOgogICAgICAgIGdsb2JhbCBfT1dORVIKICAgICAgICB3aXRoIF9MT0NLOgogICAgICAgICAgICBpZiBfT1dORVIgaXMgbm90IHNlbGY6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoaXMgYWRhcHRlciBpcyBub3QgaW5zdGFsbGVkIikKICAgICAgICAgICAgaWYgX0FDVElWRToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiV2FpdCBmb3IgZXZlcnkgYWRhcHRlciBjb25zdW1lIHRvIGZpbmlzaCBiZWZvcmUgdW5pbnN0YWxsIikKICAgICAgICAgICAgaWYgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSBpcyBub3QgX3Rva2VuaXplIG9yIEEucGF5bG9hZCBpcyBub3QgX3BheWxvYWQ6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkluc3RhbGxlZCBob29rcyBjaGFuZ2VkOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgICAgICAgICAgTC50b2tlbml6ZV9hc3Npc3RhbnRfb25seSwgQS5wYXlsb2FkID0gX09SSUdJTkFMX1RPS0VOSVpFLCBfT1JJR0lOQUxfUEFZTE9BRAogICAgICAgICAgICBfT1dORVIgPSBOb25lCgogICAgQGNvbnRleHRtYW5hZ2VyCiAgICBkZWYgaW5zdGFsbGVkKHNlbGYpOgogICAgICAgIHNlbGYuaW5zdGFsbCgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB5aWVsZCBzZWxmCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgc2VsZi51bmluc3RhbGwoKQoKICAgIGRlZiBjb25zdW1lKHNlbGYsIGF1ZGl0LCBlbnRyeSwgcGF0aCwgbWF4X3Jvd3M9MjAwMDAsIGNvbW1pdF9ldmVyeT0yNTYpOgogICAgICAgIGdsb2JhbCBfQUNUSVZFCiAgICAgICAgd2l0aCBfTE9DSzoKICAgICAgICAgICAgaWYgX09XTkVSIGlzIG5vdCBzZWxmOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFeHBsaWNpdGx5IGluc3RhbGwgYWRhcHRlciBiZWZvcmUgY29uc3VtZSIpCiAgICAgICAgICAgIGlmIGdldGF0dHIoX0xPQ0FMLCAiY29udGV4dCIsIE5vbmUpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOZXN0ZWQgYWRhcHRlciBjb25zdW1lIGlzIHVuc3VwcG9ydGVkIikKICAgICAgICAgICAgaWYgYXVkaXQudG9rZW5pemVyIGlzIG5vdCBzZWxmLnRva2VuaXplcjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkF1ZGl0IHRva2VuaXplciBtdXN0IGJlIHRoZSBleGFjdCBhZGFwdGVyIHRva2VuaXplciBpbnN0YW5jZSIpCiAgICAgICAgICAgIGN1cnNvciA9IGF1ZGl0LmRiLmV4ZWN1dGUoIlNFTEVDVCBuZXh0X2xpbmUgRlJPTSBtZW1iZXJzIFdIRVJFIGlkPT8iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChtZW1iZXJfa2V5KGVudHJ5KSwpKS5mZXRjaG9uZSgpCiAgICAgICAgICAgIGNvbnRleHQgPSBfQ29udGV4dChzZWxmLCBhdWRpdCwgbWVtYmVyX2tleShlbnRyeSksIGN1cnNvclswXSBpZiBjdXJzb3IgZWxzZSAwLCBtYXhfcm93cykKICAgICAgICAgICAgX0xPQ0FMLmNvbnRleHQgPSBjb250ZXh0CiAgICAgICAgICAgIF9BQ1RJVkUgKz0gMQogICAgICAgIHRyeToKICAgICAgICAgICAgIyBBbGwgc291cmNlL2hhc2gvc3BsaXQvbGlua2FnZS9kdXBsaWNhdGUvY29tbWl0L2N1cnNvciBsb2dpYyBpcyB0aGUKICAgICAgICAgICAgIyBhY3R1YWwgb3JpZ2luYWwgc2Nhbm5lciBtZXRob2QsIGluY2x1ZGluZyBpdHMgb3JpZ2luYWwgZXhjZXB0aW9ucy4KICAgICAgICAgICAgcmV0dXJuIGF1ZGl0LmNvbnN1bWUoZW50cnksIHBhdGgsIG1heF9yb3dzLCBjb21taXRfZXZlcnkpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgICAgIHdpdGggX0xPQ0s6CiAgICAgICAgICAgICAgICBkZWwgX0xPQ0FMLmNvbnRleHQKICAgICAgICAgICAgICAgIF9BQ1RJVkUgLT0gMQoKICAgIGRlZiBfaXRlbShzZWxmLCB0ZXh0KToKICAgICAgICAjIEludmFsaWQgc3ludGF4L3NjaGVtYSBpcyBsZWZ0IHVudG91Y2hlZCBmb3IgdGhlIG9yaWdpbmFsIGxpbmUgcmVhZGVyCiAgICAgICAgIyBhbmQgb3JpZ2luYWwgX2NoYXQgdmFsaWRhdGlvbiwgcHJlc2VydmluZyB0aGVpciBlcnJvciBvcmRlci4KICAgICAgICBpZiBsZW4odGV4dCkgPiBzZWxmLm1heF9yYXdfY2hhcnM6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICByb3cgPSBqc29uLmxvYWRzKHRleHQpCiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdywgZGljdCk6CiAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICBMLnZhbGlkYXRlX2NoYXQocm93KQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yLCBLZXlFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHByZWZpeCA9IHNlbGYudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUocm93WyJtZXNzYWdlcyJdWzoyXSwgdG9rZW5pemU9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICByZW5kZXJlZCA9IHNlbGYudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUocm93WyJtZXNzYWdlcyJdLCB0b2tlbml6ZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9RmFsc2UsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICBpZiBsZW4ocHJlZml4KSArIGxlbihyZW5kZXJlZCkgPiBzZWxmLm1heF9yZW5kZXJlZF9jaGFyczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICByZXR1cm4gKHRleHQsIHJvdywgX3Jvd19ieXRlcyhyb3cpLCBwcmVmaXgsIHJlbmRlcmVkKQoKICAgIGRlZiBfcHJlcGFyZShzZWxmLCBpdGVtcywgY29udGV4dCk6CiAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgaWYgbm90IGl0ZW1zOgogICAgICAgICAgICByZXR1cm4KICAgICAgICAjIFR3byBpbmRlcGVuZGVudCBhY3R1YWwgZmFzdC10b2tlbml6ZXIgYmF0Y2ggY2FsbHM6IG9mZmljaWFsIHRlbXBsYXRlCiAgICAgICAgIyBJRHMgdmVyc3VzIHJlbmRlcmVkIHRleHQgKyBjaGFyYWN0ZXIgb2Zmc2V0cy4gTm8gcGFkZGluZy90cnVuY2F0aW9uLgogICAgICAgIGZ1bGwgPSBzZWxmLnRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKFt4WzFdWyJtZXNzYWdlcyJdIGZvciB4IGluIGl0ZW1zXSwKICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9RmFsc2UsIGVuYWJsZV90aGlua2luZz1GYWxzZSkKICAgICAgICBpZHMgPSBmdWxsWyJpbnB1dF9pZHMiXSBpZiBoYXNhdHRyKGZ1bGwsICJrZXlzIikgZWxzZSBmdWxsCiAgICAgICAgb2Zmc2V0cyA9IHNlbGYudG9rZW5pemVyKFt4WzRdIGZvciB4IGluIGl0ZW1zXSwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlLAogICAgICAgICAgICAgICAgICAgIHJldHVybl9vZmZzZXRzX21hcHBpbmc9VHJ1ZSwgcGFkZGluZz1GYWxzZSwgdHJ1bmNhdGlvbj1GYWxzZSkKICAgICAgICBpZiBsZW4oaWRzKSAhPSBsZW4oaXRlbXMpIG9yIGxlbihvZmZzZXRzWyJpbnB1dF9pZHMiXSkgIT0gbGVuKGl0ZW1zKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2ZmaWNpYWwgdG9rZW5pemVyIGJhdGNoIGNhcmRpbmFsaXR5IGRpZmZlcnMiKQogICAgICAgIGZvciBpbmRleCwgaXRlbSBpbiBlbnVtZXJhdGUoaXRlbXMpOgogICAgICAgICAgICBwcmVwYXJlZCA9IF9QcmVwYXJlZFRva2VuaXplcihpdGVtLCBpZHNbaW5kZXhdLAogICAgICAgICAgICAgICAgeyJpbnB1dF9pZHMiOiBvZmZzZXRzWyJpbnB1dF9pZHMiXVtpbmRleF0sCiAgICAgICAgICAgICAgICAgIm9mZnNldF9tYXBwaW5nIjogb2Zmc2V0c1sib2Zmc2V0X21hcHBpbmciXVtpbmRleF19KQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBlbmNvZGVkID0gX09SSUdJTkFMX1RPS0VOSVpFKGl0ZW1bMV0sIHByZXBhcmVkLCBjb250ZXh0LmF1ZGl0LmxpbWl0KQogICAgICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEFzc2VydGlvbkVycm9yKSBhcyBlcnJvcjoKICAgICAgICAgICAgICAgICMgRGVmZXIgdGhlIGV4YWN0IG9yaWdpbmFsIGdhdGUgZmFpbHVyZSB1bnRpbCBfY2hhdCByZWFjaGVzCiAgICAgICAgICAgICAgICAjIHRoaXMgcm93OyBwcmlvciByb3dzIGtlZXAgdGhlaXIgb3JpZ2luYWwgY29tbWl0IGJlaGF2aW9yLgogICAgICAgICAgICAgICAgZW5jb2RlZCA9IGVycm9yLndpdGhfdHJhY2ViYWNrKE5vbmUpCiAgICAgICAgICAgIGNvbnRleHQuY2FjaGVbaXRlbVsyXV0gPSBlbmNvZGVkCiAgICAgICAgc2VsZi5zdGF0c1siYmF0Y2hlcyJdICs9IDEKICAgICAgICBzZWxmLnN0YXRzWyJyb3dzX2JhdGNoZWQiXSArPSBsZW4oaXRlbXMpCiAgICAgICAgc2VsZi5zdGF0c1sibWF4X2NhY2hlX3Jvd3MiXSA9IG1heChzZWxmLnN0YXRzWyJtYXhfY2FjaGVfcm93cyJdLCBsZW4oY29udGV4dC5jYWNoZSkpCiAgICAgICAgc2VsZi5zdGF0c1sibWF4X3JlbmRlcmVkX2NoYXJzIl0gPSBtYXgoc2VsZi5zdGF0c1sibWF4X3JlbmRlcmVkX2NoYXJzIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdW0obGVuKHhbM10pICsgbGVuKHhbNF0pIGZvciB4IGluIGl0ZW1zKSkKICAgICAgICBzZWxmLnN0YXRzWyJtYXhfcmF3X2NoYXJzIl0gPSBtYXgoc2VsZi5zdGF0c1sibWF4X3Jhd19jaGFycyJdLCBzdW0obGVuKHhbMF0pIGZvciB4IGluIGl0ZW1zKSkKCiAgICBkZWYgX2xpbmVzKHNlbGYsIHN0cmVhbSwgY29udGV4dCk6CiAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAoKICAgICAgICBkZWYgZmx1c2gocHJlcGFyZT1UcnVlKToKICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCiAgICAgICAgICAgIGlmIHByZXBhcmU6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5fcHJlcGFyZShiYXRjaCwgY29udGV4dCkKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgIyBBIGJhdGNoLW9ubHkgYmFja2VuZC9jYXJkaW5hbGl0eS9pbmRleCBmYWlsdXJlIG11c3Qgbm90CiAgICAgICAgICAgICAgICAgICAgIyBtb3ZlIGEgcm93J3Mgb3JpZ2luYWwgZXJyb3IvY29tbWl0IGJvdW5kYXJ5LiBEaXNjYXJkIGV2ZW4KICAgICAgICAgICAgICAgICAgICAjIGEgcGFydGlhbGx5IHByZXBhcmVkIGNhY2hlIGFuZCBydW4gZXZlcnkgb3JpZ2luYWwgcm93LgogICAgICAgICAgICAgICAgICAgIGNvbnRleHQuY2FjaGUuY2xlYXIoKQogICAgICAgICAgICAgICAgICAgIHNlbGYuc3RhdHNbImJhdGNoX3ByZXBhcmVfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICBmb3IgaXRlbSBpbiBiYXRjaDoKICAgICAgICAgICAgICAgIHlpZWxkIGl0ZW1bMF0KICAgICAgICAgICAgY29udGV4dC5jYWNoZS5jbGVhcigpCgogICAgICAgIHRyeToKICAgICAgICAgICAgaXRlcmF0b3IsIG51bWJlciA9IGl0ZXIoc3RyZWFtKSwgMAogICAgICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHRleHQgPSBuZXh0KGl0ZXJhdG9yKQogICAgICAgICAgICAgICAgZXhjZXB0IFN0b3BJdGVyYXRpb246CiAgICAgICAgICAgICAgICAgICAgaWYgYmF0Y2g6CiAgICAgICAgICAgICAgICAgICAgICAgIHlpZWxkIGZyb20gZmx1c2goKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgICMgUmVhZGluZyBhaGVhZCBjYW4gaGl0IFVuaWNvZGUvQ1JDL3RyYWlsZXIvSS9PIGZhaWx1cmVzLgogICAgICAgICAgICAgICAgICAgICMgRGVsaXZlciBldmVyeSBwcmV2aW91c2x5IHJlYWQgcGh5c2ljYWwgcm93IHZpYSB0aGUgZXhhY3QKICAgICAgICAgICAgICAgICAgICAjIG9yaWdpbmFsIHBhdGggYmVmb3JlIHN1cmZhY2luZyB0aGlzIHN0cmVhbSBleGNlcHRpb24uCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sic3RyZWFtX2Vycm9yX2J1ZmZlcmVkX3Jvd3MiXSArPSBsZW4oYmF0Y2gpCiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaChwcmVwYXJlPUZhbHNlKQogICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICAgICAgICAgICAgICBudW1iZXIgKz0gMQogICAgICAgICAgICAgICAgaWYgbnVtYmVyIDw9IGNvbnRleHQuc3RhcnQ6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sic2tpcHBlZF9yZXN1bWVfbGluZXMiXSArPSAxCiAgICAgICAgICAgICAgICAgICAgeWllbGQgdGV4dCAgIyBPcmlnaW5hbCBlbnVtZXJhdGUvc2tpcCByZXRhaW5zIHBoeXNpY2FsIGN1cnNvci4KICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgY29udGV4dC5tYXhfcm93cyBpcyBub3QgTm9uZSBhbmQgbnVtYmVyID4gY29udGV4dC5zdGFydCArIGNvbnRleHQubWF4X3Jvd3M6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIk9yaWdpbmFsIGNvbnN1bWUgcmVhZCBiZXlvbmQgaXRzIG93biB3b3JrIGJvdW5kIikKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBpdGVtID0gc2VsZi5faXRlbSh0ZXh0KQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICAjIFJlbmRlcmluZy9jYW5vbmljYWxpemF0aW9uIGFoZWFkIG9mIF9jaGF0IGlzIHNwZWN1bGF0aXZlOwogICAgICAgICAgICAgICAgICAgICMgZmFsbGJhY2sgcHJlc2VydmVzIGR1cGxpY2F0ZSBzaG9ydC1jaXJjdWl0cyBhbmQgbGV0cyB0aGUKICAgICAgICAgICAgICAgICAgICAjIHJlYWwgb3JpZ2luYWwgY2FsbCByYWlzZSBvbmx5IGF0IGl0cyBwaHlzaWNhbCByb3cuCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0c1sicm93X3ByZXBhcmVfZmFsbGJhY2tzIl0gKz0gMQogICAgICAgICAgICAgICAgICAgIGl0ZW0gPSBOb25lCiAgICAgICAgICAgICAgICBzaXplID0gbGVuKGl0ZW1bM10pICsgbGVuKGl0ZW1bNF0pIGlmIGl0ZW0gZWxzZSAwCiAgICAgICAgICAgICAgICBpZiAoYmF0Y2ggYW5kIChpdGVtIGlzIE5vbmUgb3IgbGVuKGJhdGNoKSA+PSBzZWxmLmJhdGNoX3NpemUKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3IgcmF3X2NoYXJzICsgbGVuKHRleHQpID4gc2VsZi5tYXhfcmF3X2NoYXJzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9yIHJlbmRlcmVkX2NoYXJzICsgc2l6ZSA+IHNlbGYubWF4X3JlbmRlcmVkX2NoYXJzKSk6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaCgpCiAgICAgICAgICAgICAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAogICAgICAgICAgICAgICAgaWYgaXRlbSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHlpZWxkIHRleHQgICMgRXhhY3Qgb3JpZ2luYWwgZmFsbGJhY2s7IG5ldmVyIGRyb3AgYSBsaW5lLgogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBiYXRjaC5hcHBlbmQoaXRlbSkKICAgICAgICAgICAgICAgICAgICByYXdfY2hhcnMgKz0gbGVuKHRleHQpCiAgICAgICAgICAgICAgICAgICAgcmVuZGVyZWRfY2hhcnMgKz0gc2l6ZQogICAgICAgICAgICAgICAgaWYgKGxlbihiYXRjaCkgPT0gc2VsZi5iYXRjaF9zaXplIG9yCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRleHQubWF4X3Jvd3MgaXMgbm90IE5vbmUgYW5kIG51bWJlciA9PSBjb250ZXh0LnN0YXJ0ICsgY29udGV4dC5tYXhfcm93cyk6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgZnJvbSBmbHVzaCgpCiAgICAgICAgICAgICAgICAgICAgYmF0Y2gsIHJhd19jaGFycywgcmVuZGVyZWRfY2hhcnMgPSBbXSwgMCwgMAogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGNvbnRleHQuY2FjaGUuY2xlYXIoKQo=')
assert hashlib.sha256(adapter_bytes).hexdigest()=='8d978eceeabebda9103ae23a16c23beab5b8cf5f2248984eda0b584facbedf92'
adapter_path=ROOT/'math_data_batch_token_audit.py'
adapter_path.write_bytes(adapter_bytes)
adapter_spec=importlib.util.spec_from_file_location('math_data_batch_token_audit',adapter_path)
batch_module=importlib.util.module_from_spec(adapter_spec)
sys.modules[adapter_spec.name]=batch_module
adapter_spec.loader.exec_module(batch_module)
from huggingface_hub import snapshot_download
pinned_tokenizer_path=snapshot_download(repo_id=batch_module.L.MODEL,revision=batch_module.L.REVISION,local_files_only=True,
    allow_patterns=['tokenizer.json','tokenizer_config.json','vocab.json','merges.txt','special_tokens_map.json','added_tokens.json','chat_template.jinja'])
previous_tokenizer=tokenizer
tokenizer=batch_module.L.load_qwen_tokenizer(pinned_tokenizer_path)
assert tokenizer.chat_template==previous_tokenizer.chat_template
assert hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode()).digest()==hashlib.sha256(previous_tokenizer.backend_tokenizer.to_str().encode()).digest()
batch_adapter=batch_module.BatchTokenAuditAdapter(tokenizer,batch_size=256)

# Attach to the existing, fully raw-verified V3 checkpoint; never create a file.
import threading,time,uuid,shutil
from datetime import datetime,timezone
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
assert os.environ.get('COLAB_GPU','0') in ('0','')
if any(t.name.startswith('MathVisionKidFiniteCPUDataAudit') and t.is_alive() for t in threading.enumerate()):
    raise RuntimeError('A data audit worker is already active; do not duplicate it.')
assert probe_receipt['state']=='EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY'
assert probe_receipt['fresh_raw_readback_verified']
V3_DB=RESTORED_V3_DB
V3_ID=probe_receipt['audit_identity']
recipe_all=recipe;pointer_all=recipe_pointer
assert audit_v3.plan_identity(recipe_all)==V3_ID
assert len(recipe_all['source_members'])==218 and len(recipe_all['chat_members'])==31
def file_digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
assert file_digest(V3_DB)==probe_receipt['checkpoint_sha256']
if shutil.disk_usage(ROOT).free<3*V3_DB.stat().st_size+1024**3:
    raise RuntimeError('Insufficient temporary Colab disk for data checkpoint snapshots.')
fresh=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert fresh==meta_after,'Checkpoint changed after read-only restore; refuse a stale writer.'
from exhaustive_colab_loader import load_qwen_tokenizer
assert tokenizer is batch_adapter.tokenizer  # preserve validated tokenizer instance; no model weights
preclaim=drive_api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,modifiedTime,version',supportsAllDrives=True).execute()
assert preclaim==fresh,'Checkpoint advanced while loading tokenizer; refuse a stale writer.'
writer_session=uuid.uuid4().hex
claim=dict(fresh['appProperties'],writer_session=writer_session)
v3_ref=drive_api.files().update(fileId=EXISTING_V3_ID,body={'appProperties':claim},fields='id,size,appProperties',supportsAllDrives=True).execute()
assert v3_ref['id']==EXISTING_V3_ID
assert v3_ref['appProperties']['data_sha256']==probe_receipt['checkpoint_sha256']
worker_status_path=ROOT/'full_audit_v3_worker_status.json'
full_audit_worker_state={'state':'RESUMING_EXISTING_CHECKPOINT','started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l','audit_identity':V3_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':EXISTING_V3_ID,'writer_session':writer_session,'writer_guard':'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS','restored_parent':probe_receipt,'training_started':False,'gpu_used':False,'model_weights_loaded':False}
full_audit_worker_state['checkpoint']=upgrade_proof
full_audit_worker_state['performance_upgrade']={'adapter_sha256':hashlib.sha256(adapter_bytes).hexdigest(),'validation_tests':21,'checkpoint_interval_seconds':1800,'original_auditor_unchanged':True,'source_rows_changed':False,'thread_local_batch_size':256}

def write_worker_status():
    full_audit_worker_state['batch_adapter_stats']=dict(batch_adapter.stats)
    full_audit_worker_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    tmp=worker_status_path.with_suffix('.writing')
    tmp.write_text(json.dumps(full_audit_worker_state,ensure_ascii=False,indent=2),encoding='utf8')
    tmp.replace(worker_status_path)

def finite_full_data_audit_worker_v3():
    global v3_ref
    auditor=None;checkpoint=dict(v3_ref);last_save=0.0
    def verify_writer():
        remote=work_api.files().get(fileId=checkpoint['id'],fields='id,size,appProperties',supportsAllDrives=True).execute()
        props=remote.get('appProperties',{})
        if props.get('writer_session')!=writer_session or props.get('recipe_identity')!=V3_ID or props.get('data_sha256')!=checkpoint.get('appProperties',{}).get('data_sha256'):
            raise RuntimeError('Checkpoint writer or hash changed; stop without overwriting.')
        return remote
    def save_checkpoint():
        nonlocal checkpoint,last_save
        global v3_ref
        auditor.db.commit();snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
        target_db=sqlite3.connect(snapshot)
        try:auditor.db.backup(target_db)
        finally:target_db.close()
        snapshot_sha=file_digest(snapshot)
        verify_writer()
        props={'recipe_identity':V3_ID,'data_sha256':snapshot_sha,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING','writer_session':writer_session}
        saved=work_api.files().update(fileId=checkpoint['id'],body={'appProperties':props},media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
        # Retain the completed write identity even if the verification read fails.
        # It does not become a verified receipt until all raw bytes match.
        checkpoint=saved;v3_ref=saved
        class HashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):self.hasher.update(data);self.size+=len(data);return len(data)
        sink=HashSink();download=MediaIoBaseDownload(sink,work_api.files().get_media(fileId=saved['id'],supportsAllDrives=True),chunksize=8*1024*1024)
        done=False
        while not done:_,done=download.next_chunk(num_retries=3)
        if sink.hasher.hexdigest()!=snapshot_sha or sink.size!=snapshot.stat().st_size:
            raise RuntimeError('V3 raw checkpoint readback differs.')
        after=verify_writer()
        last_save=time.monotonic()
        proof={'id':saved['id'],'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True,'report_at_snapshot':auditor.report(),'observed_utc':datetime.now(timezone.utc).isoformat()}
        full_audit_worker_state['checkpoint']=proof
        (ROOT/'v3_latest_checkpoint_ref.json').write_text(json.dumps({'metadata':after,'proof':proof},ensure_ascii=False,indent=2),encoding='utf8')
    try:
        work_api=build('drive','v3',credentials=credentials,cache_discovery=False)
        work_reader=GoogleDriveReader(work_api)
        work_cache=LazyDriveCache(work_reader,recipe_all,ROOT/'full_audit_v3_worker_payload_cache')
        auditor=audit_v3.FullArchiveAudit(V3_DB,recipe_all,tokenizer)
        auditor.db.execute('PRAGMA cache_size=-131072')
        full_audit_worker_state['report']=auditor.report();write_worker_status()
        last_save=time.monotonic()
        full_audit_worker_state['state']='RUNNING_DATA_ONLY';write_worker_status()
        while True:
            entry=auditor.next_entry()
            if entry is None:break
            verify_writer()
            full_audit_worker_state['current_member']={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']}
            write_worker_status()
            span=batch_adapter.consume(auditor,entry,work_cache[entry['sha256']],max_rows=50000,commit_every=256)
            full_audit_worker_state.update(last_span=span,report=auditor.report(),current_member={'role':entry['role'],'member':entry.get('member'),'payload_sha256':entry['sha256']})
            if not span.get('processed',0) and not span.get('complete') and not span.get('already_complete'):
                raise RuntimeError('No finite cursor progress; refuse an endless loop.')
            if 'checkpoint' not in full_audit_worker_state or time.monotonic()-last_save>=1800:save_checkpoint()
            write_worker_status()
        save_checkpoint();report=auditor.report()
        full_audit_worker_state.update(report=report,state='AUDIT_PASS_PENDING_CURRICULUM_ACCEPTANCE' if report['full_archive_token_linkage_split_audit_complete'] else 'AUDIT_FINISHED_WITH_FINDINGS')
    except Exception as exc:
        full_audit_worker_state.update(state='STOPPED_REQUIRES_AUDIT_REPAIR',error_type=type(exc).__name__,error=str(exc))
        if auditor is not None:
            full_audit_worker_state['report']=auditor.report()
            try:save_checkpoint()
            except Exception as checkpoint_error:full_audit_worker_state['checkpoint_save_error']=str(checkpoint_error)
    finally:
        if auditor is not None:auditor.close()
        batch_adapter.uninstall()
        write_worker_status()
write_worker_status()
full_audit_worker=threading.Thread(target=finite_full_data_audit_worker_v3,name='MathVisionKidFiniteCPUDataAuditV3',daemon=True)
batch_adapter.install()
full_audit_worker.start()
print(full_audit_worker_state)


{'state': 'RESUMING_EXISTING_CHECKPOINT', 'started_utc': '2026-10-22T11:45:30.023309+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8', 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'writer_session': '5013a972d1ac47cdb1d8f6eff79ce3a6', 'writer_guard': 'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS', 'restored_parent': {'state': 'EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY', 'fresh_raw_readback_verified': True, 'checkpoint_id': '16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI', 'checkpoint_sha256': '809e20f201991f9de327967da0b5cca43ff8d1c4d483ab14e25b4ef6de9281e0', 'checkpoint_bytes': 2655571968, 'audit_identity': '06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2', 'recipe_sha256': 'e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8', 'local_verified_snapshot_r

In [ ]:
import threading,json,sqlite3
from datetime import datetime,timezone
phase_root=ROOT/'audit_finish_20261022'; phase_root.mkdir(exist_ok=True)
phase_observed={'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':full_audit_worker.is_alive(),'stored_status':json.loads(json.dumps(full_audit_worker_state)),'live_batch_stats':dict(batch_adapter.stats),'training_started':False}
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=3) as phase_db:
    phase_observed['unique_chat_rows_committed']=phase_db.execute('SELECT count(*) FROM samples').fetchone()[0]
    phase_observed['issues_committed']=dict(phase_db.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
(phase_root/'runtime_observed.json').write_text(json.dumps(phase_observed,ensure_ascii=False,indent=2),encoding='utf8')
print({k:v for k,v in phase_observed.items() if k!='stored_status'})

{'observed_utc': '2026-10-22T13:54:05.352454+00:00', 'worker_alive': False, 'live_batch_stats': {'batches': 3443, 'rows_batched': 878352, 'cache_hits': 878352, 'single_row_fallbacks': 0, 'skipped_resume_lines': 9350000, 'max_cache_rows': 256, 'max_rendered_chars': 288558, 'max_raw_chars': 454766, 'batch_prepare_fallbacks': 0, 'row_prepare_fallbacks': 0, 'cache_key_fallbacks': 0, 'stream_error_buffered_rows': 0}, 'training_started': False, 'unique_chat_rows_committed': 1078096, 'issues_committed': {'SOURCE_SPLIT_MISSING_OR_UNKNOWN': 1}}


In [ ]:
import json,sqlite3,threading
from datetime import datetime,timezone
resume_probe={'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':full_audit_worker.is_alive(),'status':full_audit_worker_state,'runtime_database':str(V3_DB),'training_started':False}
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=5) as probe_db:
    resume_probe['database_config']=probe_db.execute('SELECT identity,sequence_limit FROM config').fetchall()
    resume_probe['counts']={t:probe_db.execute('SELECT count(*) FROM '+t).fetchone()[0] for t in ['source_versions','samples','occurrences','issues','members']}
    resume_probe['members_done']=probe_db.execute('SELECT role,sum(complete),count(*) FROM members GROUP BY role').fetchall()
    resume_probe['issues']=dict(probe_db.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
    resume_probe['database_journal_mode']=probe_db.execute('PRAGMA journal_mode').fetchone()[0]
resume_probe['other_audit_workers']=[t.name for t in threading.enumerate() if t.name.startswith('MathVisionKidFiniteCPUDataAudit') and t.is_alive()]
(ROOT/'resume_after_g2_probe.json').write_text(json.dumps(resume_probe,ensure_ascii=False,indent=2))
print(json.dumps(resume_probe,ensure_ascii=False))

{"observed_utc": "2026-10-22T13:54:51.269534+00:00", "worker_alive": false, "status": {"state": "STOPPED_REQUIRES_AUDIT_REPAIR", "started_utc": "2026-10-22T11:45:30.023309+00:00", "notebook_id": "1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l", "audit_identity": "06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2", "recipe_sha256": "e4197a8e797a6c4d22b938779b663d0e306536dc8af75a5f81fb3c98f7e1a7a8", "checkpoint_id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "writer_session": "5013a972d1ac47cdb1d8f6eff79ce3a6", "writer_guard": "SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS", "restored_parent": {"state": "EXISTING_V3_CHECKPOINT_RAW_VERIFIED_READONLY", "fresh_raw_readback_verified": true, "checkpoint_id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "checkpoint_sha256": "809e20f201991f9de327967da0b5cca43ff8d1c4d483ab14e25b4ef6de9281e0", "checkpoint_bytes": 2655571968, "audit_identity": "06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2", "recipe_sha256": "e4197a

In [ ]:
RESUME_PAYLOAD_B64='eJy1WG1z27gR/ius+4FSy+j4JomSh53R2U7OMznHJ9tpr2kGAwJLi2eKZAlStpLxf+8uSFqUbDe5zlWesURgsVjsPvvsgl+PihI2SV4rlkjIqqTaHs2NI3vieTPuu5HN/SiQrggETCPH9hzuO3wiuBOLmTeR0ps4Uvr2eAzONA6ms0k8jgJwjyzjKIOHak9pbHueK+OZY7s2F2DHwg9kEPgzJ4bZ1Jv40g4Cd+KPA/whpk7EfRATe+z64I6DifektASRFMDUirvjCWl2omgyE97MF1PgHPjU9nzHHYtJ5KGBAFEUgS2nU7CnzpgH/tj3PCeOYCoAvDgODjUXeZJVUKLqr0dcVDVPWVXyJGN5lm5Zmd+zKq9wcANlEicgUTDmqQJUI1a8YmtYR1AqHPZcGqvLEt3ACpD8Nr/dMtjgYlSjmIKCl7wCFPUDFJUgUl6CZIJnMpE4w7RGEkaZydh2Pd9HwbhOU8ZLsUo2wHgtk4qJfF2koHV11qwSVeVlInA7VaQoo+1WaAlqz27Zcy24uCprWtscgtzLMYS3Jd9gHBk6ht8Bc213gnF0fyjqKE3UiqVoqqpYnSV51rpx9JvKM3JtwbdpziUOx4COEKA9Q8ctyvy2BEXg03E8HS83v4zffPROq1//Gb97WKfrzeYy/fhldv7m6gH+Qdr+2LCrvC4F9CLmOrPH/9GypzU7E70giDhmylROPJsHM8+LI+5Np8KZoeWe40fjqYxm45h7+k94k9kMuCs4pofHp6S2i67sQ952fc+RHE9kT2w+GU/x5NN4xgNwfDmzBY/iGIKxxFxyxrbwIpfjdlI448iPwIm1YskLhPmh6kDOpgGgj3gEkeSYrx4H1+PORLhehKOY4iIex67rB7PAR1Db0TjwYy4iwPR2m5BrxN/n5R1uENeZqBAYpF1CbMRJliCyNfoQ47wFcCu98QbD+b8yAz+3aR7x1Nh4hJ1mSIvmZXiRZ3AsViDudLaGMhHVoBEcHqccsaj4BkJ7ZDfraF+dr1t2X+Lu5dMm9ClhnVcQkgWMF8koTlJQg+HoFqoB/T6X4W6vT2Yizc8WZn4qVYgPlkq+gMWL4rLMCyirBJRpqboo8rJSizQ9LTG/VHiNiTUcwQOIuoLBcLc7AqdQYWOD3tI80PX1sSedxM2CRrI5DBKJUuhhc/incH/EyMu+eMtwHS+T/EePnZ8eiOmoNJAgkd3ZXzPvhWU992oX80SBsaxx3zWclWVeDsyTJ71GYzWZseJqZSDrZbcgjw0ksMK4T6pVXldGjhEkQeSukTnsh6+qy6yN4i7ehAC2s30v4hlyeY7EaOzmrSfY7MReQGAPhSMZjZB21wnqPlYZL9Qqr8Llhw/XP5gNt2pg4+pucqT+jTwM5k5VxUv0HZNR2Ex5qDHLANHcrekdtCq3897eERd3dTF4UtGTxBzjabqdP82NRJqrPdh1+iliIaGcyeQWWfyljQ9S5xC7X58Ba65hZe1BYt7f0TKLuizQJHNuvr15/56dLq4XbHFzen7NPly8/5VdfGDXy8X5xfnFO9M6xPl8//mxdyqMn3yeyXVB1fTVZI5yucVT7AN7rg/3aK1BJpxpkZ/p51tUclNQTRuoqty5y1ojtKttASEpSrHuEuv9sMlkG3bPtLBA1GsepdDQwR/IIn82llBhl2JUKzC6ZkA2iWV0cTFgAxlRCAk1/UtjJSYPl9g4JKka9XWeV4bMQWG+VEYEqBYMbnSNDy4SkBSVQVmdGgg4TPN7I9piM2CseSVWPV09stYxOm6SqnnoiWEWKuMnJIGrJLs7IBHKasaofDA2UJDGwzn9HxFnQBnSV5pEowZvlJM0SX4N7eeKtGe0FotQuqeqw4ue2Kn5a5hC1g62pPM00IMgGh52J0ArZH6fEVoa9JznP3IFp+3YgIStlwoP07DrEKu91IL1FVBYYlVnd/q0wV+wO/P1v55dEmtm+JZ6w93Y/Qr16+jS7JxZWqizeKTbYq13kNVrjFdVIi5Db78e0Rk6x63goaURLBz9hCdu14Jk4W5qpCqOsiOiXpz4jqrx0dMo2+FJY5e40JAJtj2l2isOPEaiCF8lsF2vQFuM1jn6Is8ScUByeYz0gN6f9wNhaqTjWHcsy3yZ6mJM+xVDq1ln6tPlwZxT8CwkUIopw2a/W2o+UX0zNxhaZh4pKNEAVlfCnBNEtdVZfj+gH18weCOcGo4Slcd5uSbX9sixV5Tabou8D5/MnTPNz6E+7m7RoClomK5tl78TpgTWfb45HGnXsgoBM6CRkazXhRp8NZESOSUIHocigbxP6jW55vGjBZmqS7x9KJEkDTitJCO2Ct0hTmJzitU+NOsqDrqoUhXsIbjNnTCqk1RiE0IpgRVjg2wr8C5FxIdqw95vS3A8A5Y7Jail2Db7Dg90UqgQOe/y/DYFnWhLPTLodjxcoLWG7/mXrZY+ocdBT5XVFknkSeulJqENSXdf0ur2kNz2vt2C0VtcvmiubwsaG2DRPf2xv02V30GGwCyfq6EGoisj5uVy8e7nhdH4RVPIG8dz7Knb3/9V9DT4ROQcAva4AUVPula/N/le3VV/4abm8uaCuoSmg6DewfzWvg3rUeId8A0CBOHQHUOznx7qL25JT48biTLoKjKPMMR3+zKvUs5/z8T2bUFzJcXjYW+Vp9gl6f0+NQ9IPe18M9y03+0Q0kQHoY6N2rXt4+fHfVu+4Sz6qIJnYUQVnbXXRmpTsZWBQessS+9h7RLh08Gmn601f9BvMsKxjR+r6ZwZ6AxEke/zUFeZG+iQWfTPahD3DIDWvjv/L858Dg0qp2RVowmJTmCbilRv2UODZ/JgvuvVzJcmeUrcsWU7oQPE0uelKnmRt/dsA12gmuudfj1xjOUyrlGeZ4hhif2GMtI8L/aKZnuQfl3QhmF7+UpcqLwfJvGbp/z+W+gEtj1/diP7XUB8tvr4lah/kzw6ELXrmy9LT4Vmcwe5XFxdscuzi1NilpOb5fL85Ob9zc9scXJydnm9uDg5M8lDzdJP5t6rNM26LMWugN9C++pt/yWd+dkALDlGu9lbvOZc/XR2yv5+fv0TPdGuV11A4EEANtln+ot6da5obP7dx2wPdnX94fISN1me/XJzvjy7am9by7PLxfnStICAw/T9hf4NcIvhiLGMr4GxZjak+w6N7/d/rf+JDAkkmhC/k+5eLR3766nkfwM8L3mp16lo++ff0QBpvDbiaBYd+FBLu3F3xf6mK7qjPbuB7xNqnSUZWpOme5XqpZQ4evwPJvP8RQ=='
# Execute only in the already connected, authorized DATA_ONLY Colab kernel.
# Runtime-only working database; same Drive checkpoint ID, no training.
import io,json,sqlite3,hashlib,threading,time,uuid,shutil,base64,zlib,zipfile
from pathlib import Path
from datetime import datetime,timezone
from googleapiclient.http import MediaFileUpload,MediaIoBaseDownload
import httplib2,google_auth_httplib2

assert not full_audit_worker.is_alive(), 'Existing audit worker must stop first'
assert not any(t.is_alive() and t.name.startswith('MathVisionKidFiniteCPUDataAudit') for t in threading.enumerate())
assert not globals().get('g2_resume_thread') or not g2_resume_thread.is_alive()
resume_payload=json.loads(zlib.decompress(base64.b64decode(RESUME_PAYLOAD_B64)))
assert audit_v3.plan_identity(recipe_all)==resume_payload['previous_identity']
assert hashlib.sha256(Path(audit_v3.__file__).read_bytes()).hexdigest()==resume_payload['audit_code_sha256']
assert hashlib.sha256(Path(batch_module.__file__).read_bytes()).hexdigest()==resume_payload['adapter_code_sha256']
assert full_audit_worker_state['checkpoint']['fresh_raw_readback_verified']
resume_old_state=json.loads(json.dumps(full_audit_worker_state))
resume_closed_root_connections=[]
for resume_connection_name in ['g2db','confirmation_db','phase_db','probe_db','status_db']:
    resume_connection=globals().get(resume_connection_name)
    if isinstance(resume_connection,sqlite3.Connection):
        resume_connection.close()
        resume_closed_root_connections.append(resume_connection_name)
resume_directory=ROOT/'resume_after_g2_20261022';resume_directory.mkdir(exist_ok=True)
(resume_directory/'parent_worker_state.json').write_text(json.dumps(resume_old_state,ensure_ascii=False,indent=2))
g2_resume_state={'stage':'PREPARING_EXISTING_CHECKPOINT_APPEND','started_utc':datetime.now(timezone.utc).isoformat(),
    'previous_identity':resume_payload['previous_identity'],'next_identity':resume_payload['next_identity'],
    'checkpoint_id':EXISTING_V3_ID,'training_started':False,'gpu_used':False,'model_weights_loaded':False}
def g2_resume_status():
    g2_resume_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    (resume_directory/'migration_status.json').write_text(json.dumps(g2_resume_state,ensure_ascii=False,indent=2))
def g2_connected_api():
    auth_http=google_auth_httplib2.AuthorizedHttp(credentials,http=httplib2.Http(timeout=120))
    return build('drive','v3',http=auth_http,cache_discovery=False)

def g2_migrate_resume():
    global recipe,recipe_all,recipe_pointer,pointer_all,V3_ID,v3_ref,writer_session,batch_adapter
    global full_audit_worker_state,full_audit_worker,finite_full_data_audit_worker_v3
    a=None
    try:
        api=g2_connected_api()
        # Fetch the published recipe directly on Colab, not through the PC.
        sink_recipe=io.BytesIO()
        dl_recipe=MediaIoBaseDownload(sink_recipe,api.files().get_media(fileId=resume_payload['progress_id'],supportsAllDrives=True),chunksize=64*1024*1024)
        done=False
        while not done:_,done=dl_recipe.next_chunk(num_retries=3)
        progress_bytes=sink_recipe.getvalue()
        assert hashlib.sha256(progress_bytes).hexdigest()==resume_payload['progress_sha256']
        with zipfile.ZipFile(io.BytesIO(progress_bytes)) as progress_zip:
            resume_recipe_bytes=progress_zip.read(resume_payload['next_recipe_pointer']['member'])
        assert hashlib.sha256(resume_recipe_bytes).hexdigest()==resume_payload['next_recipe_sha256']
        resume_next_recipe=json.loads(resume_recipe_bytes)
        assert audit_v3.plan_identity(resume_next_recipe)==resume_payload['next_identity']
        (resume_directory/'next_recipe.json').write_bytes(resume_recipe_bytes)
        metadata_fields='id,name,size,appProperties,modifiedTime,version'
        parent_meta=api.files().get(fileId=EXISTING_V3_ID,fields=metadata_fields,supportsAllDrives=True).execute()
        parent_proof=resume_old_state['checkpoint']
        props=parent_meta.get('appProperties',{})
        assert props.get('recipe_identity')==resume_payload['previous_identity']
        assert props.get('data_sha256')==parent_proof['sha256']
        assert props.get('writer_session')==resume_old_state['writer_session']
        snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
        assert shutil.disk_usage(ROOT).free>V3_DB.stat().st_size+1024**3, 'Runtime disk lacks snapshot space'
        assert snapshot.is_file() and snapshot.stat().st_size==parent_proof['bytes']
        assert file_digest(snapshot)==parent_proof['sha256']
        parent_kept=ROOT/('parent_checkpoint_'+parent_proof['sha256']+'.sqlite')
        if parent_kept.exists():assert file_digest(parent_kept)==parent_proof['sha256']
        else:snapshot.rename(parent_kept)
        g2_resume_state['parent_verified_snapshot_preserved']=dict(parent_proof,runtime_path=str(parent_kept))
        # Root-created readers closed on their creating kernel thread above.
        # The stopped worker's connection was closed by its finally block.
        db=sqlite3.connect(V3_DB,timeout=120)
        try:
            db.execute('PRAGMA busy_timeout=120000')
            before_counts={t:db.execute('SELECT count(*) FROM '+t).fetchone()[0] for t in ['source_versions','samples','occurrences','issues']}
            before_members=list(db.execute('SELECT * FROM members ORDER BY id'))
            before_issues=list(db.execute('SELECT * FROM issues ORDER BY 1,2,3,4'))
            assert before_counts['samples']==1078096 and before_counts['source_versions']==1629842
            assert db.execute('PRAGMA journal_mode=WAL').fetchone()[0]=='wal'
        finally:db.close()
        g2_resume_state.update(stage='APPENDING_EXACT_RECIPE_WITHOUT_RESET',before_counts=before_counts)
        g2_resume_state['closed_root_read_connections']=resume_closed_root_connections
        g2_resume_status()
        migration=audit_v3.FullArchiveAudit.extend_recipe(V3_DB,recipe_all,resume_next_recipe)
        g2_resume_state['recipe_migration']=migration
        a=audit_v3.FullArchiveAudit(V3_DB,resume_next_recipe,tokenizer)
        a.db.execute('PRAGMA busy_timeout=120000');a.db.execute('PRAGMA cache_size=-131072')
        assert a.db.execute('PRAGMA journal_mode').fetchone()[0]=='wal'
        source_delta=resume_next_recipe['source_members'][len(recipe_all['source_members']):]
        chat_delta=resume_next_recipe['chat_members'][len(recipe_all['chat_members']):]
        assert len(source_delta)==len(chat_delta)==1
        assert source_delta[0]['rows']==64 and chat_delta[0]['rows']==256
        reader_now=GoogleDriveReader(api)
        cache_now=LazyDriveCache(reader_now,resume_next_recipe,ROOT/'full_audit_v3_worker_payload_cache')
        for e in source_delta+chat_delta:
            a.consume(e,cache_now[e['sha256']],max_rows=None,commit_every=64)
        after_counts={t:a.db.execute('SELECT count(*) FROM '+t).fetchone()[0] for t in before_counts}
        assert after_counts['source_versions']==before_counts['source_versions']+64
        assert after_counts['samples']==before_counts['samples']+256
        assert after_counts['occurrences']==before_counts['occurrences']+256
        assert list(a.db.execute('SELECT * FROM issues ORDER BY 1,2,3,4'))==before_issues
        after_members={x[0]:x for x in a.db.execute('SELECT * FROM members')}
        assert all(after_members[x[0]]==x for x in before_members)
        g2_resume_state.update(stage='NEW64_INDEXED_AND_256_CHAT_VERIFIED',after_counts=after_counts,
            prior_member_cursors_unchanged=True,prior_issues_preserved=True,
            bounded_new256_linkage_token_check_passed=True,new_report=a.report())
        g2_resume_status()
        a.db.commit()
        target=sqlite3.connect(snapshot)
        try:
            a.db.backup(target)
            assert target.execute('PRAGMA journal_mode=DELETE').fetchone()[0]=='delete'
        finally:target.close()
        snapshot_sha=file_digest(snapshot)
        assert api.files().get(fileId=EXISTING_V3_ID,fields=metadata_fields,supportsAllDrives=True).execute()==parent_meta
        writer_session=uuid.uuid4().hex
        new_props={'recipe_identity':resume_payload['next_identity'],'data_sha256':snapshot_sha,
            'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING','writer_session':writer_session}
        g2_resume_state.update(stage='UPLOADING_APPENDED_CHECKPOINT_SAME_ID',new_snapshot_sha256=snapshot_sha,
            new_snapshot_bytes=snapshot.stat().st_size)
        g2_resume_status()
        saved=api.files().update(fileId=EXISTING_V3_ID,
            body={'appProperties':new_props,'name':'MathVisionKid_FULL_DATA_AUDIT_V3_'+resume_payload['next_identity'][:16]+'.sqlite'},
            media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True,chunksize=64*1024*1024),
            fields='id,size,appProperties',supportsAllDrives=True).execute()
        assert saved['id']==EXISTING_V3_ID
        g2_resume_state['uploaded_checkpoint_metadata']=saved
        class ResumeHashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):
                self.hasher.update(data);self.size+=len(data)
                g2_resume_state['fresh_readback_bytes']=self.size
                return len(data)
        sink=ResumeHashSink();download=MediaIoBaseDownload(sink,api.files().get_media(fileId=EXISTING_V3_ID,supportsAllDrives=True),chunksize=64*1024*1024)
        g2_resume_state['stage']='FRESH_RAW_APPENDED_CHECKPOINT_READBACK';g2_resume_status()
        done=False
        while not done:
            _,done=download.next_chunk(num_retries=3)
            g2_resume_status()
        assert sink.size==snapshot.stat().st_size and sink.hasher.hexdigest()==snapshot_sha
        after_meta=api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties',supportsAllDrives=True).execute()
        assert after_meta==saved
        report=a.report();a.close();a=None
        proof={'id':EXISTING_V3_ID,'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True,
            'report_at_snapshot':report,'observed_utc':datetime.now(timezone.utc).isoformat()}
        recipe=recipe_all=resume_next_recipe
        recipe_pointer=pointer_all=resume_payload['next_recipe_pointer']
        V3_ID=resume_payload['next_identity'];v3_ref=saved
        batch_adapter=batch_module.BatchTokenAuditAdapter(tokenizer,batch_size=256)
        full_audit_worker_state={'state':'RESUMING_AFTER_LOCK_REPAIR_AND_EXACT_APPEND',
            'started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l',
            'audit_identity':V3_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':EXISTING_V3_ID,
            'writer_session':writer_session,'checkpoint':proof,'report':report,
            'writer_guard':'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS',
            'training_started':False,'gpu_used':False,'model_weights_loaded':False,
            'restored_parent':resume_old_state['checkpoint'],
            'lock_repair':{'journal_mode':'wal','busy_timeout_ms':120000,'root_reader_connections_closed':True},
            'performance_upgrade':dict(resume_old_state['performance_upgrade'],checkpoint_readback_chunk_bytes=64*1024*1024)}
        (ROOT/'v3_latest_checkpoint_ref.json').write_text(json.dumps({'metadata':after_meta,'proof':proof},ensure_ascii=False,indent=2))
        # The frozen auditor and adapter are untouched. Only worker IO/locking
        # settings change; all token/source/split/duplicate gates remain active.
        worker_source=resume_payload['parent_worker_function']
        assert worker_source.count("auditor.db.execute('PRAGMA cache_size=-131072')")==1
        worker_source=worker_source.replace("auditor.db.execute('PRAGMA cache_size=-131072')",
            "auditor.db.execute('PRAGMA cache_size=-131072')\n        auditor.db.execute('PRAGMA busy_timeout=120000')\n        assert auditor.db.execute('PRAGMA journal_mode').fetchone()[0]=='wal'")
        assert worker_source.count("chunksize=8*1024*1024")==1
        worker_source=worker_source.replace('chunksize=8*1024*1024','chunksize=64*1024*1024')
        assert worker_source.count('try:auditor.db.backup(target_db)')==1
        worker_source=worker_source.replace('try:auditor.db.backup(target_db)',
            "try:\n            auditor.db.backup(target_db)\n            assert target_db.execute('PRAGMA journal_mode=DELETE').fetchone()[0]=='delete'")
        assert worker_source.count("work_api=build('drive','v3',credentials=credentials,cache_discovery=False)")==1
        worker_source=worker_source.replace("work_api=build('drive','v3',credentials=credentials,cache_discovery=False)",'work_api=g2_connected_api()')
        (resume_directory/'resumed_worker_function.py').write_text(worker_source)
        exec(compile(worker_source,'resumed_worker_function.py','exec'),globals())
        assert not any(t.is_alive() and t.name.startswith('MathVisionKidFiniteCPUDataAudit') for t in threading.enumerate())
        batch_adapter.install()
        full_audit_worker=threading.Thread(target=finite_full_data_audit_worker_v3,name='MathVisionKidFiniteCPUDataAuditV3',daemon=True)
        full_audit_worker.start()
        g2_resume_state.update(stage='APPEND_CHECKPOINT_RAW_VERIFIED_WORKER_RESUMED',checkpoint=proof,
            new_worker_alive=full_audit_worker.is_alive(),checkpoint_id_unchanged=True,parent_snapshot_retained=True)
        g2_resume_status()
    except Exception as e:
        g2_resume_state.update(stage='STOPPED_REQUIRES_RESUME_REPAIR',error_type=type(e).__name__,error=str(e))
        g2_resume_status()
    finally:
        if a is not None:a.close()

g2_resume_status()
g2_resume_thread=threading.Thread(target=g2_migrate_resume,name='MathVisionKidG2AppendCheckpointMigration',daemon=True)
g2_resume_thread.start()
print(json.dumps({'migration_started':True,'checkpoint_id':EXISTING_V3_ID,
    'existing_chat_cursor':1078096,'new_cases':64,'new_chats':256,'training_started':False}))


{"migration_started": true, "checkpoint_id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "existing_chat_cursor": 1078096, "new_cases": 64, "new_chats": 256, "training_started": false}


In [ ]:
import json,sqlite3
from datetime import datetime,timezone
resume_observed={'observed_utc':datetime.now(timezone.utc).isoformat(),'migration_thread_alive':g2_resume_thread.is_alive(),'migration':json.loads(json.dumps(g2_resume_state)),'audit_worker_alive':full_audit_worker.is_alive(),'training_started':False}
if g2_resume_state['stage']=='APPEND_CHECKPOINT_RAW_VERIFIED_WORKER_RESUMED':
    with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as status_read:
        resume_observed['actual_chat_rows_committed']=status_read.execute('SELECT count(*) FROM samples').fetchone()[0]
        resume_observed['actual_source_versions']=status_read.execute('SELECT count(*) FROM source_versions').fetchone()[0]
        resume_observed['issues']=dict(status_read.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
        resume_observed['journal_mode']=status_read.execute('PRAGMA journal_mode').fetchone()[0]
    status_read.close()
(ROOT/'resume_after_g2_20261022/latest_observation.json').write_text(json.dumps(resume_observed,ensure_ascii=False,indent=2))
print(json.dumps(resume_observed,ensure_ascii=False))

{"observed_utc": "2026-10-22T15:14:41.210359+00:00", "migration_thread_alive": false, "migration": {"stage": "APPEND_CHECKPOINT_RAW_VERIFIED_WORKER_RESUMED", "started_utc": "2026-10-22T14:03:40.548497+00:00", "previous_identity": "06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2", "next_identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "checkpoint_id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "training_started": false, "gpu_used": false, "model_weights_loaded": false, "updated_utc": "2026-10-22T14:12:47.399095+00:00", "parent_verified_snapshot_preserved": {"id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "bytes": 3414695936, "sha256": "6edaf07aa5e133ea5cf3cb4c04d30278fd9deb90ce62d0c4f95a32ac92ccad87", "fresh_raw_readback_verified": true, "report_at_snapshot": {"schema": "MathVisionKid.full_current_archive_audit.v3", "identity": "06339a42b0a4b8d2c8ce7b1031a41a6ca1fc936dd361dd4055e17f8796f5b8e2", "derivation_provenance": {"parent_file": "math_data_full_

In [ ]:
import json,sqlite3,sys,traceback
from datetime import datetime,timezone
connections=[]
for name,value in list(globals().items()):
    if isinstance(value,sqlite3.Connection):
        try:connections.append({'name':name,'in_transaction':value.in_transaction,'databases':value.execute('PRAGMA database_list').fetchall()})
        except Exception as error:connections.append({'name':name,'closed_or_error':str(error)})
frame=sys._current_frames().get(g2_resume_thread.ident)
stack=[{'function':x.name,'file':x.filename,'line':x.lineno} for x in traceback.extract_stack(frame)[-8:]] if frame else []
print(json.dumps({'stage':g2_resume_state['stage'],'migration_alive':g2_resume_thread.is_alive(),'connections':connections,'migration_stack':stack,'training_started':False},ensure_ascii=False))

{"stage": "NEW64_INDEXED_AND_256_CHAT_VERIFIED", "migration_alive": true, "connections": [{"name": "con", "closed_or_error": "Cannot operate on a closed database."}, {"name": "con_probe", "closed_or_error": "Cannot operate on a closed database."}, {"name": "diagnostic_db", "closed_or_error": "Cannot operate on a closed database."}, {"name": "upgrade_check", "in_transaction": false, "databases": [[0, "main", "/content/mathvisionkid_data_only/batch_resume_809e20f201991f9d.sqlite"], [1, "temp", ""]]}, {"name": "phase_db", "closed_or_error": "Cannot operate on a closed database."}, {"name": "confirmation_db", "closed_or_error": "Cannot operate on a closed database."}, {"name": "g2db", "closed_or_error": "Cannot operate on a closed database."}, {"name": "probe_db", "closed_or_error": "Cannot operate on a closed database."}], "migration_stack": [{"function": "_bootstrap", "file": "/usr/lib/python3.13/threading.py", "line": 1015}, {"function": "_bootstrap_inner", "file": "/usr/lib/python3.13/

In [ ]:
def g2_finish_verified_resume(a,snapshot_sha,sink,saved,after_meta,resume_next_recipe):
    global recipe,recipe_all,recipe_pointer,pointer_all,V3_ID,v3_ref,writer_session,batch_adapter,full_audit_worker_state,full_audit_worker,finite_full_data_audit_worker_v3
    report=a.report();a.close();a=None
    proof={'id':EXISTING_V3_ID,'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True,
        'report_at_snapshot':report,'observed_utc':datetime.now(timezone.utc).isoformat()}
    recipe=recipe_all=resume_next_recipe
    recipe_pointer=pointer_all=resume_payload['next_recipe_pointer']
    V3_ID=resume_payload['next_identity'];v3_ref=saved
    batch_adapter=batch_module.BatchTokenAuditAdapter(tokenizer,batch_size=256)
    full_audit_worker_state={'state':'RESUMING_AFTER_LOCK_REPAIR_AND_EXACT_APPEND',
        'started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l',
        'audit_identity':V3_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':EXISTING_V3_ID,
        'writer_session':writer_session,'checkpoint':proof,'report':report,
        'writer_guard':'SINGLE_OBSERVED_KERNEL_PLUS_OPTIMISTIC_HASH_NOT_DISTRIBUTED_ATOMIC_CAS',
        'training_started':False,'gpu_used':False,'model_weights_loaded':False,
        'restored_parent':resume_old_state['checkpoint'],
        'lock_repair':{'journal_mode':'wal','busy_timeout_ms':120000,'root_reader_connections_closed':True},
        'performance_upgrade':dict(resume_old_state['performance_upgrade'],checkpoint_readback_chunk_bytes=64*1024*1024)}
    (ROOT/'v3_latest_checkpoint_ref.json').write_text(json.dumps({'metadata':after_meta,'proof':proof},ensure_ascii=False,indent=2))
    # The frozen auditor and adapter are untouched. Only worker IO/locking
    # settings change; all token/source/split/duplicate gates remain active.
    worker_source=resume_payload['parent_worker_function']
    assert worker_source.count("auditor.db.execute('PRAGMA cache_size=-131072')")==1
    worker_source=worker_source.replace("auditor.db.execute('PRAGMA cache_size=-131072')",
        "auditor.db.execute('PRAGMA cache_size=-131072')\n        auditor.db.execute('PRAGMA busy_timeout=120000')\n        assert auditor.db.execute('PRAGMA journal_mode').fetchone()[0]=='wal'")
    assert worker_source.count("chunksize=8*1024*1024")==1
    worker_source=worker_source.replace('chunksize=8*1024*1024','chunksize=64*1024*1024')
    assert worker_source.count('try:auditor.db.backup(target_db)')==1
    worker_source=worker_source.replace('try:auditor.db.backup(target_db)',
        "try:\n            auditor.db.backup(target_db)\n            assert target_db.execute('PRAGMA journal_mode=DELETE').fetchone()[0]=='delete'")
    assert worker_source.count("work_api=build('drive','v3',credentials=credentials,cache_discovery=False)")==1
    worker_source=worker_source.replace("work_api=build('drive','v3',credentials=credentials,cache_discovery=False)",'work_api=g2_connected_api()')
    (resume_directory/'resumed_worker_function.py').write_text(worker_source)
    exec(compile(worker_source,'resumed_worker_function.py','exec'),globals())
    assert not any(t.is_alive() and t.name.startswith('MathVisionKidFiniteCPUDataAudit') for t in threading.enumerate())
    batch_adapter.install()
    full_audit_worker=threading.Thread(target=finite_full_data_audit_worker_v3,name='MathVisionKidFiniteCPUDataAuditV3',daemon=True)
    full_audit_worker.start()
    g2_resume_state.update(stage='APPEND_CHECKPOINT_RAW_VERIFIED_WORKER_RESUMED',checkpoint=proof,
        new_worker_alive=full_audit_worker.is_alive(),checkpoint_id_unchanged=True,parent_snapshot_retained=True)
    g2_resume_status()

# Continue the already committed append; never reinsert/reset/re-migrate it.
from googleapiclient.http import build_http
def g2_connected_api():
    # Official client config leaves HTTP308 for resumable-upload handling.
    h=build_http();h.timeout=120
    auth_http=google_auth_httplib2.AuthorizedHttp(credentials,http=h)
    return build('drive','v3',http=auth_http,cache_discovery=False)

assert not full_audit_worker.is_alive() and not g2_resume_thread.is_alive()
assert g2_resume_state['recipe_migration']['next_identity']==resume_payload['next_identity']
assert g2_resume_state['after_counts']['samples']==1078352
g2_retry_parent=json.loads(json.dumps(g2_resume_state))
(resume_directory/'publication_redirect_failure_preserved.json').write_text(json.dumps(g2_retry_parent,ensure_ascii=False,indent=2))
if isinstance(globals().get('upgrade_check'),sqlite3.Connection):upgrade_check.close()
def g2_retry_checkpoint_publication():
    global writer_session
    a=None
    try:
        api=g2_connected_api()
        checkpoint_path=ROOT/'full_audit_v3_snapshot.sqlite'
        expected_sha=g2_retry_parent['new_snapshot_sha256']
        assert checkpoint_path.stat().st_size==g2_retry_parent['new_snapshot_bytes']
        assert file_digest(checkpoint_path)==expected_sha
        next_recipe_bytes=(resume_directory/'next_recipe.json').read_bytes()
        assert hashlib.sha256(next_recipe_bytes).hexdigest()==resume_payload['next_recipe_sha256']
        next_recipe=json.loads(next_recipe_bytes)
        a=audit_v3.FullArchiveAudit(V3_DB,next_recipe,tokenizer)
        a.db.execute('PRAGMA busy_timeout=120000')
        assert a.db.execute('PRAGMA journal_mode').fetchone()[0]=='wal'
        assert a.db.execute('SELECT count(*) FROM samples').fetchone()[0]==1078352
        assert a.db.execute('SELECT count(*) FROM source_versions').fetchone()[0]==1629906
        before=api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties,version,modifiedTime',supportsAllDrives=True).execute()
        props=before.get('appProperties',{})
        assert props.get('data_sha256') in [resume_old_state['checkpoint']['sha256'],expected_sha]
        assert props.get('writer_session') in [resume_old_state['writer_session'],writer_session]
        writer_session=uuid.uuid4().hex
        g2_resume_state.update(stage='RETRY_RESUMABLE_UPLOAD_OFFICIAL_HTTP308_HANDLING',
            redirect_failure_preserved=True,upload_error=str(g2_retry_parent.get('error')))
        g2_resume_status()
        newprops={'recipe_identity':resume_payload['next_identity'],'data_sha256':expected_sha,
            'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING','writer_session':writer_session}
        saved=api.files().update(fileId=EXISTING_V3_ID,
            body={'appProperties':newprops,'name':'MathVisionKid_FULL_DATA_AUDIT_V3_'+resume_payload['next_identity'][:16]+'.sqlite'},
            media_body=MediaFileUpload(str(checkpoint_path),mimetype='application/vnd.sqlite3',resumable=True,chunksize=64*1024*1024),
            fields='id,size,appProperties',supportsAllDrives=True).execute()
        g2_resume_state['uploaded_checkpoint_metadata']=saved
        class RetrySink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):
                self.hasher.update(data);self.size+=len(data)
                g2_resume_state['fresh_readback_bytes']=self.size
                return len(data)
        sink=RetrySink();dl=MediaIoBaseDownload(sink,api.files().get_media(fileId=EXISTING_V3_ID,supportsAllDrives=True),chunksize=64*1024*1024)
        g2_resume_state['stage']='FRESH_RAW_APPENDED_CHECKPOINT_READBACK';g2_resume_status()
        done=False
        while not done:_,done=dl.next_chunk(num_retries=3);g2_resume_status()
        assert sink.size==checkpoint_path.stat().st_size and sink.hasher.hexdigest()==expected_sha
        after=api.files().get(fileId=EXISTING_V3_ID,fields='id,size,appProperties',supportsAllDrives=True).execute()
        assert after==saved
        g2_finish_verified_resume(a,expected_sha,sink,saved,after,next_recipe)
        a=None
    except Exception as e:
        g2_resume_state.update(stage='STOPPED_REQUIRES_RESUME_REPAIR',retry_error_type=type(e).__name__,retry_error=str(e))
        g2_resume_status()
    finally:
        if a is not None:a.close()
g2_resume_thread=threading.Thread(target=g2_retry_checkpoint_publication,name='MathVisionKidG2AppendCheckpointMigration',daemon=True)
g2_resume_thread.start()
print({'publication_retry_started':True,'already_checked_chat_rows':1078352,'no_data_reinserted':True,'training_started':False})


{'publication_retry_started': True, 'already_checked_chat_rows': 1078352, 'no_data_reinserted': True, 'training_started': False}


In [ ]:
import json,sqlite3,hashlib
from datetime import datetime,timezone
# Read-only metadata/evidence; no benchmark question or model weights.
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as qa_db:
    qa_db.execute('PRAGMA query_only=ON')
    qa_counts={name:qa_db.execute('SELECT count(*) FROM '+name).fetchone()[0] for name in ['source_versions','samples','occurrences','issues']}
    qa_split=dict(qa_db.execute('SELECT split,count(*) FROM samples GROUP BY split'))
    qa_labels=dict(qa_db.execute("SELECT coalesce(source_split,'<missing>'),count(*) FROM source_versions GROUP BY source_split"))
    qa_findings=qa_db.execute('SELECT member,line,kind,detail FROM issues ORDER BY 1,2,3,4').fetchall()
    qa_reg_sources=qa_db.execute("SELECT id,raw_sha,canonical_sha,source_split FROM source_versions WHERE source_split='regression_only'").fetchall()
    qa_reg_links=[dict(sample_id=row[0],raw_sha=row[1],canonical_sha=row[2],source_label=row[3],linked_chat_rows=qa_db.execute('SELECT count(*) FROM samples WHERE case_id=? AND source_raw_sha=?',(row[0],row[1])).fetchone()[0]) for row in qa_reg_sources]
    qa_members=qa_db.execute('SELECT role,count(*),sum(complete),sum(next_line) FROM members GROUP BY role').fetchall()
    qa_links=dict(linkage_unverified=qa_db.execute('SELECT count(*) FROM samples WHERE linkage=0').fetchone()[0],incomplete_four_mode_cases=qa_db.execute('SELECT count(*) FROM (SELECT case_id FROM samples GROUP BY case_id HAVING count(*)!=4)').fetchone()[0])
qa_db.close()
qa_live=dict(observed_utc=datetime.now(timezone.utc).isoformat(),worker_alive=full_audit_worker.is_alive(),
    audit_identity=V3_ID,counts=qa_counts,actual_chat_splits=qa_split,source_labels=qa_labels,issues=qa_findings,
    regression_only_source_evidence=qa_reg_links,member_progress=qa_members,linkage=qa_links,
    latest_verified_checkpoint=full_audit_worker_state.get('checkpoint'),worker_state=full_audit_worker_state.get('state'),
    executing_recipe_sha256=pointer_all['sha256'],full_archive_complete=False,training_started=False,
    scope='All indexed source labels; current committed chats only. Final exclusion zero-links proof waits until every member finishes.')
assert qa_live['worker_alive']
assert len(qa_reg_links)==1 and qa_reg_links[0]['raw_sha']=='1729ed713dfedcc0205b130c6573313dcc5784c7eaf5618d0cf32fdc728cf26b'
assert qa_reg_links[0]['linked_chat_rows']==0 and qa_links['linkage_unverified']==0
qa_dir=ROOT/'continued_archive_check_20261022';qa_dir.mkdir(exist_ok=True)
(qa_dir/'source_split_linkage_evidence.json').write_text(json.dumps(qa_live,ensure_ascii=False,indent=2))
print(json.dumps(qa_live,ensure_ascii=False))

{"observed_utc": "2026-10-22T15:07:55.700327+00:00", "worker_alive": true, "audit_identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "counts": {"source_versions": 1629906, "samples": 2105008, "occurrences": 2105008, "issues": 1}, "actual_chat_splits": {"train": 2105264}, "source_labels": {"regression_only": 1, "test": 6, "train": 1381, "train_candidate": 1628512, "validation": 6}, "issues": [["79765b8281ff64675b4e3826c07b003237875a98d67e898f22f2a4c1f687b0d1", 1, "SOURCE_SPLIT_MISSING_OR_UNKNOWN", "exhaustive_v1_div_1563_43"]], "regression_only_source_evidence": [{"sample_id": "exhaustive_v1_div_1563_43", "raw_sha": "1729ed713dfedcc0205b130c6573313dcc5784c7eaf5618d0cf32fdc728cf26b", "canonical_sha": "1729ed713dfedcc0205b130c6573313dcc5784c7eaf5618d0cf32fdc728cf26b", "source_label": "regression_only", "linked_chat_rows": 0}], "member_progress": [["candidate_chat", 4, 3, 2107312], ["candidate_source", 219, 219, 1629906]], "linkage": {"linkage_unverified": 0, "i

In [ ]:
import os,json,time,sys,traceback
from datetime import datetime,timezone
cpu_obs={'utc':datetime.now(timezone.utc).isoformat(),'cpu_count':os.cpu_count(),'affinity':sorted(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else None,'batch_stats':dict(batch_adapter.stats),'worker_alive':full_audit_worker.is_alive(),'state':full_audit_worker_state.get('state'),'current_member':full_audit_worker_state.get('current_member'),'checkpoint':full_audit_worker_state.get('checkpoint'),'runtime_environment':{k:os.environ.get(k) for k in ['TOKENIZERS_PARALLELISM','RAYON_NUM_THREADS','OMP_NUM_THREADS']}}
f=sys._current_frames().get(full_audit_worker.ident)
cpu_obs['stack']=[{'function':x.name,'file':x.filename,'line':x.lineno} for x in traceback.extract_stack(f)[-10:]] if f else []
print(json.dumps(cpu_obs,ensure_ascii=False))

{"utc": "2026-10-22T15:09:07.705632+00:00", "cpu_count": 2, "affinity": [0, 1], "batch_stats": {"batches": 4125, "rows_batched": 1052336, "cache_hits": 1052336, "single_row_fallbacks": 0, "skipped_resume_lines": 8583472, "max_cache_rows": 256, "max_rendered_chars": 288558, "max_raw_chars": 454766, "batch_prepare_fallbacks": 0, "row_prepare_fallbacks": 0, "cache_key_fallbacks": 0, "stream_error_buffered_rows": 0}, "worker_alive": true, "state": "RUNNING_DATA_ONLY", "current_member": {"role": "candidate_chat", "member": null, "payload_sha256": "8161fe5fbb6fd353983e7d2187fc6c502429316c27d5454fe9bbe6e6b19dd613"}, "checkpoint": {"id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "bytes": 3964362752, "sha256": "4f80cf9cde0787856bbecd03690506c880f86642f33cb58f4e260eed380832fa", "fresh_raw_readback_verified": true, "report_at_snapshot": {"schema": "MathVisionKid.full_current_archive_audit.v3", "identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "derivation_provenance": {"pa

In [ ]:
import json,time
from datetime import datetime,timezone
span_before={'utc':datetime.now(timezone.utc).isoformat(),'batch_stats':dict(batch_adapter.stats),'worker_alive':full_audit_worker.is_alive(),'current_member':full_audit_worker_state.get('current_member'),'last_span':full_audit_worker_state.get('last_span')}
print(json.dumps(span_before,ensure_ascii=False))

In [ ]:
import json,sqlite3
from datetime import datetime,timezone
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as count_db:
    count_db.execute('PRAGMA query_only=ON')
    count_reconcile={'utc':datetime.now(timezone.utc).isoformat(),'versions_and_distinct_ids':count_db.execute('SELECT count(*),count(DISTINCT id) FROM source_versions').fetchone(),
      'labels':count_db.execute('SELECT source_split,count(*),count(DISTINCT id) FROM source_versions GROUP BY source_split').fetchall(),
      'multiple_versions_id_count':count_db.execute('SELECT count(*) FROM (SELECT id FROM source_versions GROUP BY id HAVING count(*)>1)').fetchone()[0],
      'version_multiplicities':count_db.execute('SELECT n,count(*) FROM (SELECT count(*) n FROM source_versions GROUP BY id) GROUP BY n').fetchall(),
      'scope':'Metadata-only index reconciliation; no held-out question content read. Current chats incomplete.',
      'train_or_weights_started':False}
count_db.close()
(qa_dir/'source_id_inventory_reconciliation.json').write_text(json.dumps(count_reconcile,ensure_ascii=False,indent=2))
print(json.dumps(count_reconcile,ensure_ascii=False))

{"utc": "2026-10-22T15:11:23.462510+00:00", "versions_and_distinct_ids": [1629906, 1629906], "labels": [["regression_only", 1, 1], ["test", 6, 6], ["train", 1381, 1381], ["train_candidate", 1628512, 1628512], ["validation", 6, 6]], "multiple_versions_id_count": 0, "version_multiplicities": [[1, 1629906]], "scope": "Metadata-only index reconciliation; no held-out question content read. Current chats incomplete.", "train_or_weights_started": false}


In [ ]:
# Orchestration-only work budget; original scanner, adapter and commits intact.
import json,hashlib,time
from datetime import datetime,timezone
assert full_audit_worker.is_alive()
assert not globals().get('_larger_span_runtime_applied',False)
assert hashlib.sha256(Path(batch_module.__file__).read_bytes()).hexdigest()=='8d978eceeabebda9103ae23a16c23beab5b8cf5f2248984eda0b584facbedf92'
assert hashlib.sha256(Path(audit_v3.__file__).read_bytes()).hexdigest()=='02431da433060a6574127f9a8e14d90cabffe85d8d0150c3b2a331dc15b4be1f'
assert batch_adapter.__class__ is batch_module.BatchTokenAuditAdapter
assert audit_v3.plan_identity(recipe_all)=='f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863'
span_original_consume=batch_adapter.consume
span_runtime_evidence={'applied_utc':datetime.now(timezone.utc).isoformat(),
    'old_span_rows':50000,'new_span_rows':200000,'commit_every':256,'token_batch_size':256,
    'scanner_or_adapter_files_changed':False,'checkpoint_id_unchanged':True,
    'already_started_calls_keep_captured_method':True,'first_new_budget_call_observed':False,
    'remote_checkpoint_tradeoff':'Checkpoint interval is checked at span boundaries; local SQLite still commits every256 rows.',
    'original_stats':dict(batch_adapter.stats),'training_started':False,'calls':[]}
def bounded_larger_span_consume(audit,entry,path,max_rows=20000,commit_every=256):
    # Only the current worker's explicit50K budget is enlarged, not other callers.
    chosen=200000 if max_rows==50000 and commit_every==256 else max_rows
    span_runtime_evidence['first_new_budget_call_observed']=True
    span_runtime_evidence['last_effective_rows']=chosen
    started=time.monotonic();previous=dict(batch_adapter.stats)
    result=span_original_consume(audit,entry,path,max_rows=chosen,commit_every=commit_every)
    span_runtime_evidence['calls'].append({'ended_utc':datetime.now(timezone.utc).isoformat(),
        'seconds':time.monotonic()-started,'requested_rows':max_rows,'effective_rows':chosen,
        'processed':result.get('processed',0),'complete':result.get('complete',False),
        'new_batched_rows':batch_adapter.stats['rows_batched']-previous['rows_batched'],
        'skipped_prefix_rows':batch_adapter.stats['skipped_resume_lines']-previous['skipped_resume_lines']})
    (qa_dir/'span_runtime_evidence.json').write_text(json.dumps(span_runtime_evidence,ensure_ascii=False,indent=2))
    return result
batch_adapter.consume=bounded_larger_span_consume
_larger_span_runtime_applied=True
full_audit_worker_state['performance_upgrade']['outer_span_rows']=200000
full_audit_worker_state['performance_upgrade']['checkpoint_boundary_tradeoff']='up to one200K span between elapsed-time checks; local commits256'
(qa_dir/'span_runtime_evidence.json').write_text(json.dumps(span_runtime_evidence,ensure_ascii=False,indent=2))
write_worker_status()
print(json.dumps({'larger_bounded_span_applied':True,'worker_still_alive':full_audit_worker.is_alive(),
    'frozen_scanner_adapter_unchanged':True,'commit_every':256,'training_started':False}))



{"larger_bounded_span_applied": true, "worker_still_alive": true, "frozen_scanner_adapter_unchanged": true, "commit_every": 256, "training_started": false}


In [ ]:
import json,sqlite3,time
from datetime import datetime,timezone
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as light_db:
    light_n=light_db.execute('SELECT count(*) FROM samples').fetchone()[0]
    light_done=light_db.execute('SELECT count(*) FROM members WHERE complete=1').fetchone()[0]
    light_issues=dict(light_db.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
light_db.close()
light_progress={'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':full_audit_worker.is_alive(),
    'committed_chat_rows':light_n,'members_complete':light_done,'members_expected':251,'issues':light_issues,
    'last_verified_checkpoint':{k:full_audit_worker_state.get('checkpoint',{}).get(k) for k in ['id','bytes','sha256','observed_utc']},
    'worker_state':full_audit_worker_state.get('state'),'error':full_audit_worker_state.get('error'),
    'span_stats':globals().get('span_runtime_evidence',{}).get('calls',[])[-2:],'batch_stats':dict(batch_adapter.stats),
    'full_archive_complete':False,'training_started':False}
(qa_dir/'latest_compact_progress.json').write_text(json.dumps(light_progress,ensure_ascii=False,indent=2))
print(json.dumps(light_progress,ensure_ascii=False))

{"observed_utc": "2026-10-22T15:27:26.682835+00:00", "worker_alive": true, "committed_chat_rows": 2464560, "members_complete": 222, "members_expected": 251, "issues": {"SOURCE_SPLIT_MISSING_OR_UNKNOWN": 1}, "last_verified_checkpoint": {"id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "bytes": 3964362752, "sha256": "4f80cf9cde0787856bbecd03690506c880f86642f33cb58f4e260eed380832fa", "observed_utc": "2026-10-22T14:48:55.123235+00:00"}, "worker_state": "RUNNING_DATA_ONLY", "error": null, "span_stats": [{"ended_utc": "2026-10-22T15:23:47.231662+00:00", "seconds": 508.0966846859992, "requested_rows": 50000, "effective_rows": 200000, "processed": 200000, "complete": false, "new_batched_rows": 200000, "skipped_prefix_rows": 400000}], "batch_stats": {"batches": 5432, "rows_batched": 1386208, "cache_hits": 1386208, "single_row_fallbacks": 0, "skipped_resume_lines": 9633472, "max_cache_rows": 256, "max_rendered_chars": 288558, "max_raw_chars": 454766, "batch_prepare_fallbacks": 0, "row_prepare_fallback

In [ ]:
import os,time,threading,json,psutil,sys,traceback
p=psutil.Process();p.cpu_percent(None);start=time.monotonic();time.sleep(2)
obs={'utc':datetime.now(timezone.utc).isoformat(),'cpu_percent_process':p.cpu_percent(None),'thread_count':p.num_threads(),'cpu_affinity':p.cpu_affinity(),'rss_mb':p.memory_info().rss/1024**2,'tokenizers_parallelism':os.environ.get('TOKENIZERS_PARALLELISM'),'rayon_threads':os.environ.get('RAYON_NUM_THREADS'),'worker_alive':full_audit_worker.is_alive()}
fr=sys._current_frames().get(full_audit_worker.ident)
obs['stack']=[{'function':x.name,'line':x.lineno,'file':x.filename.rsplit('/',1)[-1]} for x in traceback.extract_stack(fr)] if fr else []
print(json.dumps(obs))

{"utc": "2026-10-22T17:24:51.638809+00:00", "cpu_percent_process": 50.4, "thread_count": 26, "cpu_affinity": [0, 1], "rss_mb": 1046.54296875, "tokenizers_parallelism": null, "rayon_threads": null, "worker_alive": true, "stack": [{"function": "_bootstrap", "line": 1015, "file": "threading.py"}, {"function": "_bootstrap_inner", "line": 1044, "file": "threading.py"}, {"function": "run", "line": 995, "file": "threading.py"}, {"function": "finite_full_data_audit_worker_v3", "line": 68, "file": "resumed_worker_function.py"}, {"function": "save_checkpoint", "line": 31, "file": "resumed_worker_function.py"}, {"function": "positional_wrapper", "line": 130, "file": "_helpers.py"}, {"function": "next_chunk", "line": 821, "file": "http.py"}, {"function": "_retry_request", "line": 191, "file": "http.py"}, {"function": "request", "line": 214, "file": "google_auth_httplib2.py"}, {"function": "request", "line": 1755, "file": "__init__.py"}, {"function": "_request", "line": 1475, "file": "__init__.py"}

In [ ]:
import json
print(json.dumps({'small_index_files':[{'relative_path':str(p.relative_to(ROOT)),'bytes':p.stat().st_size} for p in ROOT.rglob('*.json') if p.is_file() and ('protect' in p.name.lower() or 'holdout' in p.name.lower())][:60],'working_db':str(V3_DB),'worker_state':full_audit_worker_state.get('state'),'writer_checkpoint_status':{k:full_audit_worker_state.get('checkpoint',{}).get(k) for k in ['bytes','observed_utc','sha256']}}))

{"small_index_files": [], "working_db": "/content/mathvisionkid_data_only/batch_resume_809e20f201991f9d.sqlite", "worker_state": "RUNNING_DATA_ONLY", "writer_checkpoint_status": {"bytes": 4574961664, "observed_utc": "2026-10-22T15:31:10.528976+00:00", "sha256": "aab811924a15f8f6b900fc40fbb1493dd3de333ad12116b878d473a2a0a863ad"}}


In [ ]:
import base64,hashlib,json,importlib.util,sys
(qa_dir/"finalize_current_candidate_metadata.py").write_bytes(base64.b64decode('IiIiT25lIGZpbml0ZSBtZXRhZGF0YS1vbmx5IGZpbmFsaXphdGlvbiBhZnRlciB0aGUgZXhpc3RpbmcgQ1BVIHNjYW5uZXIgZXhpdHMuCgpFeGVjdXRlIGluc2lkZSB0aGUgYXV0aGVudGljYXRlZCBEQVRBX09OTFkgQ29sYWIgcnVudGltZSwgbm90IG9uIHRoZSB1c2VyJ3MgUEMuClRoaXMgY3JlYXRlcyBubyBkYXRhIGNhc2VzLCB0cmFpbmluZyBwcm9jZXNzLCBEcml2ZSBmaWxlLCBvciBhY2NlcHRhbmNlIHNpZ25vZmYuCkl0IHByZXNlcnZlcyB0aGUgcmF3IFYzIGZpbmRpbmcgYW5kIHdyaXRlcyBzZXBhcmF0ZSwgbmFycm93ZXIgY29uY2x1c2lvbnMuCiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW1wb3J0bGliLnV0aWwKaW1wb3J0IGpzb24KaW1wb3J0IHNxbGl0ZTMKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGhyZWFkaW5nCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgcmVnaXN0ZXJfZmluYWxfbWV0YWRhdGFfY2hlY2sobmFtZXNwYWNlKToKICAgIHJvb3QgPSBuYW1lc3BhY2VbJ1JPT1QnXQogICAgZGlyZWN0b3J5ID0gbmFtZXNwYWNlWydxYV9kaXInXQogICAgd29ya2VyID0gbmFtZXNwYWNlWydmdWxsX2F1ZGl0X3dvcmtlciddCiAgICBkYXRhYmFzZSA9IG5hbWVzcGFjZVsnVjNfREInXQogICAgZXhwZWN0ZWRfaWRlbnRpdHkgPSAnZjAzMzJkZjkxMDIwYWNlMGZjNDhkODg0OTFmZTk3MzY0ZDA4ODI2NDU4ZDA4YzcxYmE0ZWM2MDUyNGUyNTg2MycKICAgIGV4cGVjdGVkX3JlY2lwZV9zaGEgPSAnMWJiNjljMzk0YzdlYWFlYTcwMzQxMjVjNmIzMzYxZWViYmJlMGQ3N2UwNzE1YTg0NTQzMzFmYmU3Y2VlM2ZmOCcKICAgIGlmIG5hbWVzcGFjZVsnVjNfSUQnXSAhPSBleHBlY3RlZF9pZGVudGl0eSBvciBuYW1lc3BhY2VbJ3BvaW50ZXJfYWxsJ11bJ3NoYTI1NiddICE9IGV4cGVjdGVkX3JlY2lwZV9zaGE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignT25seSB0aGUgZXhhY3QgY3VycmVudCByZWNpcGUgbWF5IGJlIGZpbmFsaXplZCcpCiAgICBpZiBhbnkodC5uYW1lID09ICdNYXRoVmlzaW9uS2lkRmluYWxNZXRhZGF0YU9ubHknIGFuZCB0LmlzX2FsaXZlKCkgZm9yIHQgaW4gdGhyZWFkaW5nLmVudW1lcmF0ZSgpKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0EgZmluYWwgbWV0YWRhdGEgY2hlY2sgYWxyZWFkeSB3YWl0cyBmb3IgdGhpcyB3b3JrZXInKQogICAgZGlyZWN0b3J5Lm1rZGlyKGV4aXN0X29rPVRydWUsIHBhcmVudHM9VHJ1ZSkKCiAgICBkZWYgc2F2ZShuYW1lLCB2YWx1ZSk6CiAgICAgICAgcGF0aCA9IGRpcmVjdG9yeSAvIG5hbWUKICAgICAgICBkYXRhID0gKGpzb24uZHVtcHModmFsdWUsIGVuc3VyZV9hc2NpaT1GYWxzZSwgc29ydF9rZXlzPVRydWUsIGluZGVudD0yKSArICdcbicpLmVuY29kZSgpCiAgICAgICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICcud3JpdGluZycpCiAgICAgICAgdGVtcG9yYXJ5LndyaXRlX2J5dGVzKGRhdGEpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAgICAgICByZXR1cm4geyduYW1lJzogbmFtZSwgJ2J5dGVzJzogbGVuKGRhdGEpLCAnc2hhMjU2JzogaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCl9CgogICAgc3RhdHVzID0geydzdGF0ZSc6ICdXQUlUSU5HX0ZPUl9FWElTVElOR19GVUxMX1NDQU4nLAogICAgICAgICAgICAgICdyZWdpc3RlcmVkX3V0Yyc6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICAgICAgICdyZWNpcGVfaWRlbnRpdHknOiBleHBlY3RlZF9pZGVudGl0eSwgJ3JlY2lwZV9zaGEyNTYnOiBleHBlY3RlZF9yZWNpcGVfc2hhLAogICAgICAgICAgICAgICd0cmFpbmluZ19zdGFydGVkJzogRmFsc2UsICduZXdfY2FzZXMnOiAwLCAnbmV3X2NoYXRfcm93cyc6IDB9CiAgICBzYXZlKCdmaW5hbF9tZXRhZGF0YV9zdGF0dXMuanNvbicsIHN0YXR1cykKCiAgICBkZWYgZmluaXNoX29uY2UoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgICMgVGhpcyBpcyBhIGJhY2tncm91bmQgd2FpdCwgbm90IGEgc2Vjb25kIHNjYW5uZXIgb3IgYSBwb2xsaW5nIGxvb3AuCiAgICAgICAgICAgIHdvcmtlci5qb2luKCkKICAgICAgICAgICAgc3RhdGUgPSBuYW1lc3BhY2VbJ2Z1bGxfYXVkaXRfd29ya2VyX3N0YXRlJ10KICAgICAgICAgICAgaWYgc3RhdGUuZ2V0KCdzdGF0ZScpICE9ICdBVURJVF9GSU5JU0hFRF9XSVRIX0ZJTkRJTkdTJzoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1dvcmtlciBkaWQgbm90IGZpbmlzaCB3aXRoIG9ubHkgaXRzIGtub3duIHByZXNlcnZlZCBmaW5kaW5nOiAnICsgc3RyKHN0YXRlLmdldCgnc3RhdGUnKSkpCiAgICAgICAgICAgIHJhd19yZXBvcnQgPSBzdGF0ZVsncmVwb3J0J10KICAgICAgICAgICAgcHJvb2YgPSBzdGF0ZVsnY2hlY2twb2ludCddCiAgICAgICAgICAgIGlmIG5vdCBwcm9vZi5nZXQoJ2ZyZXNoX3Jhd19yZWFkYmFja192ZXJpZmllZCcpIG9yIHByb29mWydpZCddICE9ICcxNlE5ZnQ5eGhjaW9jb25rLTV6cHhtZFJ2MVpETnhxRkknOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRmluYWwgY2hlY2twb2ludCBsYWNrcyBleGFjdC1JRCBmcmVzaCByYXcgcmVhZGJhY2snKQogICAgICAgICAgICBpZiBwcm9vZlsncmVwb3J0X2F0X3NuYXBzaG90J10gIT0gcmF3X3JlcG9ydDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ZpbmFsIHJlcG9ydCBkaWZmZXJzIGZyb20gdGhlIHZlcmlmaWVkIGNoZWNrcG9pbnQgcmVwb3J0JykKICAgICAgICAgICAgc25hcHNob3QgPSByb290IC8gJ2Z1bGxfYXVkaXRfdjNfc25hcHNob3Quc3FsaXRlJwogICAgICAgICAgICBpZiBzbmFwc2hvdC5zdGF0KCkuc3Rfc2l6ZSAhPSBwcm9vZlsnYnl0ZXMnXSBvciBuYW1lc3BhY2VbJ2ZpbGVfZGlnZXN0J10oc25hcHNob3QpICE9IHByb29mWydzaGEyNTYnXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ZpbmFsIHBvcnRhYmxlIHNuYXBzaG90IGJ5dGVzIG5vIGxvbmdlciBtYXRjaCB2ZXJpZmllZCBwcm9vZicpCgogICAgICAgICAgICByZWNpcGUgPSBuYW1lc3BhY2VbJ3JlY2lwZV9hbGwnXQogICAgICAgICAgICBtZW1iZXJzID0gW3snaWQnOiBuYW1lc3BhY2VbJ2JhdGNoX21vZHVsZSddLm1lbWJlcl9rZXkoZSksCiAgICAgICAgICAgICAgICAgICAgICAgICdyb2xlJzogZVsncm9sZSddLCAncm93cyc6IGVbJ3Jvd3MnXX0KICAgICAgICAgICAgICAgICAgICAgICBmb3IgZSBpbiByZWNpcGVbJ3NvdXJjZV9tZW1iZXJzJ10gKyByZWNpcGVbJ2NoYXRfbWVtYmVycyddXQogICAgICAgICAgICB3aXRoIHNxbGl0ZTMuY29ubmVjdChkYXRhYmFzZS5hc191cmkoKSArICc/bW9kZT1ybycsIHVyaT1UcnVlLCB0aW1lb3V0PTMwKSBhcyBkYjoKICAgICAgICAgICAgICAgIGRiLmV4ZWN1dGUoJ1BSQUdNQSBxdWVyeV9vbmx5PU9OJykKICAgICAgICAgICAgICAgIGlmIGRiLmV4ZWN1dGUoJ1NFTEVDVCBpZGVudGl0eSxzZXF1ZW5jZV9saW1pdCBGUk9NIGNvbmZpZycpLmZldGNob25lKCkgIT0gKGV4cGVjdGVkX2lkZW50aXR5LCA0MDk2MCk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRmluYWwgd29ya2luZyBkYXRhYmFzZSBpZGVudGl0eSBvciBzZXF1ZW5jZSBsaW1pdCBjaGFuZ2VkJykKICAgICAgICAgICAgICAgIGFjdHVhbCA9IHtyWzBdOiB0dXBsZShyWzE6XSkgZm9yIHIgaW4gZGIuZXhlY3V0ZSgKICAgICAgICAgICAgICAgICAgICAnU0VMRUNUIGlkLHJvbGUsbmV4dF9saW5lLGNvbXBsZXRlLGV4cGVjdGVkX3Jvd3MgRlJPTSBtZW1iZXJzJyl9CiAgICAgICAgICAgICAgICBleHBlY3RlZCA9IHttWydpZCddOiAobVsncm9sZSddLCBtWydyb3dzJ10sIDEsIG1bJ3Jvd3MnXSkgZm9yIG0gaW4gbWVtYmVyc30KICAgICAgICAgICAgICAgIGlmIGFjdHVhbCAhPSBleHBlY3RlZDoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdFeGFjdCBmaW5hbCBtZW1iZXIgY3Vyc29ycy9yb2xlcy9yb3cgY291bnRzIGRpZmZlcicpCiAgICAgICAgICAgICAgICBpc3N1ZV9yb3dzID0gW2RpY3QoemlwKCgnbWVtYmVyJywnbGluZScsJ2tpbmQnLCdkZXRhaWwnKSwgcikpIGZvciByIGluIGRiLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAgICAgJ1NFTEVDVCBtZW1iZXIsbGluZSxraW5kLGRldGFpbCBGUk9NIGlzc3VlcyBPUkRFUiBCWSBtZW1iZXIsbGluZSxraW5kLGRldGFpbCcpXQogICAgICAgICAgICAgICAgc291cmNlID0gZGIuZXhlY3V0ZSgnU0VMRUNUIGlkLHNvdXJjZV9zcGxpdCxyYXdfc2hhLGNhbm9uaWNhbF9zaGEgRlJPTSBzb3VyY2VfdmVyc2lvbnMgV0hFUkUgaWQ9PycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICgnZXhoYXVzdGl2ZV92MV9kaXZfMTU2M180MycsKSkuZmV0Y2hhbGwoKQogICAgICAgICAgICAgICAgaWYgbGVuKHNvdXJjZSkgIT0gMToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdEaWFnbm9zdGljIHNvdXJjZSBpZGVudGl0eSBpcyBhbWJpZ3VvdXMgb3IgYWJzZW50JykKICAgICAgICAgICAgICAgIGRpYWdub3N0aWMgPSBkaWN0KHppcCgoJ3NhbXBsZV9pZCcsJ3NwbGl0JywncmF3X3NoYTI1NicsJ2Nhbm9uaWNhbF9zaGEyNTYnKSwgc291cmNlWzBdKSkKICAgICAgICAgICAgICAgIGxpbmtzID0gZGIuZXhlY3V0ZSgnU0VMRUNUIGNvdW50KCopIEZST00gc2FtcGxlcyBXSEVSRSBjYXNlX2lkPT8nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICgnZXhoYXVzdGl2ZV92MV9kaXZfMTU2M180MycsKSkuZmV0Y2hvbmUoKVswXQogICAgICAgICAgICAgICAgaW52YWxpZF90b2tlbnMgPSBkYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSBzYW1wbGVzIFdIRVJFIHNlcXVlbmNlX3Rva2Vuczw9MCBPUiBzZXF1ZW5jZV90b2tlbnM+NDA5NjAgT1IgY29tcGxldGlvbl90b2tlbnM8PTAgT1IgY29tcGxldGlvbl90b2tlbnM+c2VxdWVuY2VfdG9rZW5zJykuZmV0Y2hvbmUoKVswXQogICAgICAgICAgICAgICAgaWYgaW52YWxpZF90b2tlbnM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBwZXItcm93IHNlcXVlbmNlL2NvbXBsZXRpb24gbGVuZ3RocyByZW1haW4nKQogICAgICAgICAgICAgICAgY2FzZV9zcGxpdHMgPSBkaWN0KGRiLmV4ZWN1dGUoJ1NFTEVDVCBzcGxpdCxjb3VudChESVNUSU5DVCBjYXNlX2lkKSBGUk9NIHNhbXBsZXMgR1JPVVAgQlkgc3BsaXQnKSkKICAgICAgICAgICAgICAgIGlmIGNhc2Vfc3BsaXRzICE9IHsndHJhaW4nOjE2MjU1ODYsICd2YWxpZGF0aW9uJzo2LCAndGVzdCc6Nn06CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQWN0dWFsIHVuaXF1ZSBjYXNlIHNwbGl0IGNvdW50cyBkaWZmZXInKQogICAgICAgICAgICBkYi5jbG9zZSgpCgogICAgICAgICAgICBzcGVjID0gaW1wb3J0bGliLnV0aWwuc3BlY19mcm9tX2ZpbGVfbG9jYXRpb24oJ212a19maW5hbF9hZGp1ZGljYXRpb24nLCBkaXJlY3RvcnkvJ2ZpbmFsX3RlY2huaWNhbF9hZGp1ZGljYXRpb24ucHknKQogICAgICAgICAgICBhZGp1ZGljYXRvciA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKICAgICAgICAgICAgc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUoYWRqdWRpY2F0b3IpCiAgICAgICAgICAgIHJlc3VsdCA9IGFkanVkaWNhdG9yLmFkanVkaWNhdGVfZmluYWxfdjNfbWV0YWRhdGEocmF3X3JlcG9ydCwgaXNzdWVfcm93cywgZGlhZ25vc3RpYywKICAgICAgICAgICAgICAgIHsnZXhoYXVzdGl2ZV92MV9kaXZfMTU2M180Myc6IHsnbGlua2VkX2NoYXRfcm93cyc6IGxpbmtzfX0sCiAgICAgICAgICAgICAgICB7J21lbWJlcnMnOm1lbWJlcnMsICd1bmlxdWVfY2hhdF9yb3dzJzo2NTAyMzkyLAogICAgICAgICAgICAgICAgICd1bmlxdWVfY2hhdF9yb3dzX2J5X3NwbGl0Jzogeyd0cmFpbic6NjUwMjM0NCwndmFsaWRhdGlvbic6MjQsJ3Rlc3QnOjI0fSwKICAgICAgICAgICAgICAgICAnbW9kZXNfcGVyX2Nhc2UnOjR9KQogICAgICAgICAgICByYXdfcmVmID0gc2F2ZSgnZmluYWxfcmF3X3YzX3JlcG9ydF9wcmVzZXJ2ZWQuanNvbicsIHJhd19yZXBvcnQpCiAgICAgICAgICAgIHJlc3VsdC51cGRhdGUob2JzZXJ2ZWRfdXRjPWRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgIGNoZWNrcG9pbnQ9cHJvb2YsIGV4YWN0X21lbWJlcl9jdXJzb3JzX3ZlcmlmaWVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3Jvd190b2tlbl9sZW5ndGhzX3ZlcmlmaWVkPVRydWUsIGFjdHVhbF9jYXNlc19ieV9zcGxpdD1jYXNlX3NwbGl0cywKICAgICAgICAgICAgICAgICAgICAgICAgICByYXdfcmVwb3J0X2FydGlmYWN0PXJhd19yZWYsIHRva2VuaXplcl9yZXZpc2lvbj1uYW1lc3BhY2VbJ2JhdGNoX21vZHVsZSddLkwuUkVWSVNJT04pCiAgICAgICAgICAgIHJlc3VsdF9yZWYgPSBzYXZlKCdmaW5hbF9jYW5kaWRhdGVfY2hhdF90ZWNobmljYWxfYWRqdWRpY2F0aW9uLmpzb24nLCByZXN1bHQpCgogICAgICAgICAgICBjb21tYW5kID0gW3N5cy5leGVjdXRhYmxlLCBzdHIoZGlyZWN0b3J5LydmaW5hbF9zb3VyY2VfY2hhdF9hbnRpam9pbi5weScpLAogICAgICAgICAgICAgICAgICAgICAgIHN0cihkYXRhYmFzZSksIHN0cihkaXJlY3RvcnkvJ2ZpbmFsX3NvdXJjZV9jaGF0X2FudGlqb2luLmpzb24nKSwKICAgICAgICAgICAgICAgICAgICAgICAnLS1wcm90ZWN0ZWQtaW5kZXgnLCBzdHIoZGlyZWN0b3J5Lydwcm90ZWN0ZWRfbnVtZXJpY19zaWduYXR1cmVzLmpzb24nKSwKICAgICAgICAgICAgICAgICAgICAgICAnLS1ob2xkb3V0LWluZGV4Jywgc3RyKGRpcmVjdG9yeS8naG9sZG91dF9zaWduYXR1cmVzLmpzb24nKV0KICAgICAgICAgICAgYW50aSA9IHN1YnByb2Nlc3MucnVuKGNvbW1hbmQsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD05MDApCiAgICAgICAgICAgIHNhdmUoJ2ZpbmFsX2FudGlqb2luX2V4ZWN1dGlvbi5qc29uJywgeydyZXR1cm5jb2RlJzphbnRpLnJldHVybmNvZGUsCiAgICAgICAgICAgICAgICAgJ3N0ZG91dCc6YW50aS5zdGRvdXRbLTQwMDA6XSwgJ3N0ZGVycic6YW50aS5zdGRlcnJbLTQwMDA6XSwKICAgICAgICAgICAgICAgICAncGF5bG9hZF9vcl9xdWVzdGlvbl9jb250ZW50X3JlYWQnOkZhbHNlfSkKICAgICAgICAgICAgaWYgYW50aS5yZXR1cm5jb2RlOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU291cmNlLW9ubHkgcmVjb25jaWxpYXRpb24gZmFpbGVkOyBwcmVzZXJ2ZSBhbGwgZXZpZGVuY2UnKQogICAgICAgICAgICBjbGFzc2lmaWNhdGlvbiA9IGpzb24ubG9hZHMoKGRpcmVjdG9yeS8nZmluYWxfc291cmNlX2NoYXRfYW50aWpvaW4uanNvbicpLnJlYWRfYnl0ZXMoKSkKICAgICAgICAgICAgaWYgY2xhc3NpZmljYXRpb25bJ2NvdW50cyddWydjbGFzc2lmaWNhdGlvbnMnXS5nZXQoJ1VOUkVTT0xWRURfUVVBUkFOVElORScsIDApOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5yZXNvbHZlZCBzb3VyY2Utb25seSByb3dzIHJlbWFpbiBxdWFyYW50aW5lZCcpCiAgICAgICAgICAgIG1hbmlmZXN0ID0gewogICAgICAgICAgICAgICAgJ3NjaGVtYSc6J01hdGhWaXNpb25LaWQuY3VycmVudF90ZXh0X2NhbmRpZGF0ZV92aWV3X2ZpbmFsX21ldGFkYXRhLnYxJywKICAgICAgICAgICAgICAgICdzdGF0dXMnOidDVVJSRU5UX1RFWFRfQ0FORElEQVRFX1ZJRVdfVEVDSE5JQ0FMTFlfVkVSSUZJRUQnLAogICAgICAgICAgICAgICAgJ3JlY2lwZV9pZGVudGl0eSc6ZXhwZWN0ZWRfaWRlbnRpdHksICdyZWNpcGVfc2hhMjU2JzpleHBlY3RlZF9yZWNpcGVfc2hhLAogICAgICAgICAgICAgICAgJ2NhbmRpZGF0ZV9jaGF0X3Jvd3MnOjY1MDIzNDQsICdjYW5kaWRhdGVfY2FzZV9pZHMnOjE2MjU1ODYsCiAgICAgICAgICAgICAgICAndmFsaWRhdGlvbl9jaGF0X3Jvd3MnOjI0LCAndGVzdF9jaGF0X3Jvd3MnOjI0LAogICAgICAgICAgICAgICAgJ3Jhd19jaGF0X3Jvd3NfYWxsX2N1cnJlbnRfc3BsaXRzJzo2NTAyMzkyLAogICAgICAgICAgICAgICAgJ3NvdXJjZV92ZXJzaW9uc19yZXRhaW5lZCc6MTYyOTkwNiwKICAgICAgICAgICAgICAgICdzb3VyY2Vfb25seV9hdWRpdF9yb3dzX3JldGFpbmVkJzpjbGFzc2lmaWNhdGlvblsnY291bnRzJ11bJ3VubGlua2VkX3NvdXJjZV92ZXJzaW9ucyddLAogICAgICAgICAgICAgICAgJ3NlbGVjdGlvbic6J0FsbCBhbmQgb25seSBjdXJyZW50IGFjY2VwdGVkIGNhbmRpZGF0ZSByb3dzIHdpdGggZXhwbGljaXQgc3BsaXQ9dHJhaW47IHByZXNlcnZlIGV2YWx1YXRpb24gc3BsaXRzLicsCiAgICAgICAgICAgICAgICAnY29yZV9jYXNlcyc6NDk1LCAnY29yZV9ub2Rlcyc6MTY1LCAnY29yZV9jaGF0X3Jvd3MnOjE5ODAsCiAgICAgICAgICAgICAgICAndGVjaG5pY2FsX2FkanVkaWNhdGlvbic6cmVzdWx0X3JlZiwgJ2NoZWNrcG9pbnQnOnByb29mLAogICAgICAgICAgICAgICAgJ3BoeXNpY2FsX29ic2VydmF0aW9uX3Rhc2tzX3BlbmRpbmcnOjUsICdhc3NldF9tYXN0ZXJzX3BlbmRpbmcnOjEwLAogICAgICAgICAgICAgICAgJ2czX3NvdXJjZV9sb2NhdG9yX3BlbmRpbmcnOjEsICd3aG9sZV9jdXJyaWN1bHVtX2NlcnRpZmllZCc6RmFsc2UsCiAgICAgICAgICAgICAgICAnYWxsX3Jvd3Nfc2VtYW50aWNhbGx5X2h1bWFuX3Jldmlld2VkJzpGYWxzZSwKICAgICAgICAgICAgICAgICdUUkFJTklOR19EQVRBU0VUX1JFQURZJzpGYWxzZSwgJ3RyYWluaW5nX3N0YXJ0ZWQnOkZhbHNlLAogICAgICAgICAgICAgICAgJ3Njb3BlX2xpbWl0JzonVGVjaG5pY2FsIHZlcmlmaWNhdGlvbiBvZiB0aGUgY3VycmVudCBmdWxsIHRleHQgY2FuZGlkYXRlIHZpZXc7IHJlbWFpbmluZyBwaHlzaWNhbC9pbWFnZS9zb3VyY2UgY3VycmljdWx1bSBhY2NlcHRhbmNlIGlzIHNlcGFyYXRlLiBObyB0cmFpbmluZyBsYXVuY2hlZC4nLAogICAgICAgICAgICB9CiAgICAgICAgICAgIG1hbmlmZXN0X3JlZiA9IHNhdmUoJ2ZpbmFsX2N1cnJlbnRfdGV4dF9jYW5kaWRhdGVfbWFuaWZlc3QuanNvbicsIG1hbmlmZXN0KQogICAgICAgICAgICBzdGF0dXMudXBkYXRlKHN0YXRlPSdGSU5BTF9URVhUX1RFQ0hOSUNBTF9FVklERU5DRV9DT01QTEVURV9QRU5ESU5HX0NPTlRST0xfUFVCTElDQVRJT04nLAogICAgICAgICAgICAgICAgICAgICAgICAgIGNvbXBsZXRlZF91dGM9ZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgbWFuaWZlc3Q9bWFuaWZlc3RfcmVmLCBjaGVja3BvaW50PXByb29mLAogICAgICAgICAgICAgICAgICAgICAgICAgIHNvdXJjZV9vbmx5X2NsYXNzaWZpY2F0aW9ucz1jbGFzc2lmaWNhdGlvblsnY291bnRzJ11bJ2NsYXNzaWZpY2F0aW9ucyddKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6CiAgICAgICAgICAgIHN0YXR1cy51cGRhdGUoc3RhdGU9J0ZJTkFMSVpBVElPTl9TVE9QUEVEX1dJVEhfRVZJREVOQ0UnLCBlcnJvcl90eXBlPXR5cGUoZXJyb3IpLl9fbmFtZV9fLAogICAgICAgICAgICAgICAgICAgICAgICAgIGVycm9yPXN0cihlcnJvciksIHN0b3BwZWRfdXRjPWRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHNhdmUoJ2ZpbmFsX21ldGFkYXRhX3N0YXR1cy5qc29uJywgc3RhdHVzKQoKICAgIHRocmVhZCA9IHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PWZpbmlzaF9vbmNlLCBuYW1lPSdNYXRoVmlzaW9uS2lkRmluYWxNZXRhZGF0YU9ubHknLCBkYWVtb249VHJ1ZSkKICAgIHRocmVhZC5zdGFydCgpCiAgICByZXR1cm4gdGhyZWFkLCBkaWN0KHN0YXR1cykK'))
(qa_dir/"final_technical_adjudication.py").write_bytes(base64.b64decode('IiIiRmFpbC1jbG9zZWQgZmluYWwgVjMgbWV0YWRhdGEgYWRqdWRpY2F0aW9uOyBubyBjb3JwdXMgcm93cyBhcmUgb3BlbmVkLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCmZyb20gY29weSBpbXBvcnQgZGVlcGNvcHkKaW1wb3J0IHVuaXR0ZXN0CmltcG9ydCBoYXNobGliLCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUEFTU19MQUJFTCA9ICJDVVJSRU5UX0NBTkRJREFURV9DSEFUX1RFQ0hOSUNBTF9QQVNTX1dJVEhfSEFTSF9CT1VORF9ESUFHTk9TVElDX0VYQ0xVU0lPTiIKRVhQRUNURURfSURFTlRJVFkgPSAiZjAzMzJkZjkxMDIwYWNlMGZjNDhkODg0OTFmZTk3MzY0ZDA4ODI2NDU4ZDA4YzcxYmE0ZWM2MDUyNGUyNTg2MyIKUkVHUkVTU0lPTl9TQU1QTEVfSUQgPSAiZXhoYXVzdGl2ZV92MV9kaXZfMTU2M180MyIKUkVHUkVTU0lPTl9TSEEyNTYgPSAiMTcyOWVkNzEzZGZlZGNjMDIwNWIxMzBjNjU3MzMxM2RjYzU3ODRjN2VhZjU2MThkMGNmMzJmZGM3MjhjZjI2YiIKUkVHUkVTU0lPTl9NRU1CRVJfS0VZID0gIjc5NzY1YjgyODFmZjY0Njc1YjRlMzgyNmMwN2IwMDMyMzc4NzVhOThkNjdlODk4ZjIyZjJhNGMxZjY4N2IwZDEiClJFR1JFU1NJT05fSVNTVUUgPSB7Im1lbWJlciI6IFJFR1JFU1NJT05fTUVNQkVSX0tFWSwgImxpbmUiOiAxLAogICAgICAgICAgICAgICAgICAgICJraW5kIjogIlNPVVJDRV9TUExJVF9NSVNTSU5HX09SX1VOS05PV04iLCAiZGV0YWlsIjogUkVHUkVTU0lPTl9TQU1QTEVfSUR9CkVYUEVDVEVEX1NQTElUUyA9IHsidHJhaW4iOiA2XzUwMl8zNDQsICJ2YWxpZGF0aW9uIjogMjQsICJ0ZXN0IjogMjR9CkVYUEVDVEVEX01FTUJFUlNfU0hBMjU2ID0gImIzMWJlZGU0ZGQ4ODMyNGEzYmJjZTMwYTdmMTQ3OTdiYjljNGRmMWRmNGRjOWQ3MTkwMTlmYzY3NjQ0NzgxNTgiCgoKZGVmIGFkanVkaWNhdGVfZmluYWxfdjNfbWV0YWRhdGEocmF3X3JlcG9ydDogZGljdCwgZXhhY3RfaXNzdWVfcm93czogbGlzdFtkaWN0XSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVncmVzc2lvbl9zb3VyY2VfbWV0YTogZGljdCwgemVyb19saW5rY291bnRzOiBkaWN0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZWNpcGVfZXhwZWN0ZWRfcm93Y291bnRzOiBkaWN0KSAtPiBkaWN0OgogICAgIiIiRGVyaXZlIHRlY2huaWNhbCBwYXNzIHdoaWxlIHJldGFpbmluZyBWMydzIHJhdyBmYWxzZSBjb21wbGV0aW9uIGZsYWcuCgogICAgUmVxdWlyZWQgcmVjaXBlIG1ldGFkYXRhIGNvbnRhaW5zIGFsbCAyNTEgZXhhY3QgbWVtYmVyIGlkcywgcm9sZXMsIGFuZCByb3cKICAgIGNvdW50cywgcGx1cyBleHBlY3RlZCB1bmlxdWVfY2hhdF9yb3dzX2J5X3NwbGl0IGFuZCBtb2Rlc19wZXJfY2FzZT00LgogICAgTm8gcXVlc3Rpb24gdGV4dCBpcyBhY2NlcHRlZCBvciBuZWVkZWQuCiAgICAiIiIKICAgIGlmIG5vdCBhbGwoaXNpbnN0YW5jZSh4LCBkaWN0KSBmb3IgeCBpbiAocmF3X3JlcG9ydCwgcmVncmVzc2lvbl9zb3VyY2VfbWV0YSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB6ZXJvX2xpbmtjb3VudHMsIHJlY2lwZV9leHBlY3RlZF9yb3djb3VudHMpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtZXRhZGF0YSBpbnB1dHMgbXVzdCBiZSBvYmplY3RzIikKICAgIG1lbWJlcnMgPSByZWNpcGVfZXhwZWN0ZWRfcm93Y291bnRzLmdldCgibWVtYmVycyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShtZW1iZXJzLCBsaXN0KSBvciBsZW4obWVtYmVycykgIT0gMjUxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2lwZSBtdXN0IGRlY2xhcmUgZXhhY3RseSAyNTEgbWVtYmVycyIpCiAgICBpZiBhbnkobm90IGlzaW5zdGFuY2UobSwgZGljdCkgb3Igbm90IGlzaW5zdGFuY2UobS5nZXQoImlkIiksIHN0cikgb3Igbm90IG1bImlkIl0KICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZShtLmdldCgicm93cyIpLCBpbnQpIG9yIGlzaW5zdGFuY2UobS5nZXQoInJvd3MiKSwgYm9vbCkgb3IgbVsicm93cyJdIDwgMAogICAgICAgICAgIG9yIG0uZ2V0KCJyb2xlIikgbm90IGluICgiY2FuZGlkYXRlX3NvdXJjZSIsICJjYW5kaWRhdGVfY2hhdCIpIGZvciBtIGluIG1lbWJlcnMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2lwZSBtZW1iZXIgaWRzLCByb2xlcywgb3Igcm93IGNvdW50cyBhcmUgbWFsZm9ybWVkIikKICAgIGlmIGxlbih7bVsiaWQiXSBmb3IgbSBpbiBtZW1iZXJzfSkgIT0gMjUxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2lwZSBtZW1iZXIgaWRzIGFyZSBub3QgdW5pcXVlIikKICAgIGlmIHN1bShtWyJyb2xlIl0gPT0gImNhbmRpZGF0ZV9zb3VyY2UiIGZvciBtIGluIG1lbWJlcnMpICE9IDIxOToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyZWNpcGUgbXVzdCBjb250YWluIGV4YWN0bHkgMjE5IHNvdXJjZSBtZW1iZXJzIikKICAgIGlmIHN1bShtWyJyb2xlIl0gPT0gImNhbmRpZGF0ZV9jaGF0IiBmb3IgbSBpbiBtZW1iZXJzKSAhPSAzMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyZWNpcGUgbXVzdCBjb250YWluIGV4YWN0bHkgMzIgY2hhdCBtZW1iZXJzIikKICAgIG1lbWJlcl9ieXRlcyA9IGpzb24uZHVtcHMobWVtYmVycywgZW5zdXJlX2FzY2lpPUZhbHNlLCBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKS5lbmNvZGUoKQogICAgaWYgaGFzaGxpYi5zaGEyNTYobWVtYmVyX2J5dGVzKS5oZXhkaWdlc3QoKSAhPSBFWFBFQ1RFRF9NRU1CRVJTX1NIQTI1NjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJleGFjdCBwaW5uZWQgbWVtYmVyIG1ldGFkYXRhIGRpZmZlcnMiKQoKICAgIHNvdXJjZV9yb3dzID0gc3VtKG1bInJvd3MiXSBmb3IgbSBpbiBtZW1iZXJzIGlmIG1bInJvbGUiXSA9PSAiY2FuZGlkYXRlX3NvdXJjZSIpCiAgICBwaHlzaWNhbF9jaGF0X3Jvd3MgPSBzdW0obVsicm93cyJdIGZvciBtIGluIG1lbWJlcnMgaWYgbVsicm9sZSJdID09ICJjYW5kaWRhdGVfY2hhdCIpCiAgICBleHBlY3RlZF9zcGxpdHMgPSByZWNpcGVfZXhwZWN0ZWRfcm93Y291bnRzLmdldCgidW5pcXVlX2NoYXRfcm93c19ieV9zcGxpdCIpCiAgICBpZiBleHBlY3RlZF9zcGxpdHMgIT0gRVhQRUNURURfU1BMSVRTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2lwZSBzcGxpdCBleHBlY3RhdGlvbnMgZGlmZmVyIGZyb20gY3VycmVudCByZWNpcGUiKQogICAgaWYgc291cmNlX3Jvd3MgIT0gMV82MjlfOTA2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2lwZSBzb3VyY2Ugcm93IHRvdGFsIGRpZmZlcnMgZnJvbSAxLDYyOSw5MDYiKQogICAgaWYgcGh5c2ljYWxfY2hhdF9yb3dzICE9IDZfNTAyXzM5MiBvciBzdW0oZXhwZWN0ZWRfc3BsaXRzLnZhbHVlcygpKSAhPSBwaHlzaWNhbF9jaGF0X3Jvd3M6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVjaXBlIHBoeXNpY2FsIGNoYXQgcm93cyBkaWZmZXIgZnJvbSBleHBlY3RlZCBzcGxpdCB0b3RhbHMiKQogICAgaWYgcmVjaXBlX2V4cGVjdGVkX3Jvd2NvdW50cy5nZXQoInVuaXF1ZV9jaGF0X3Jvd3MiKSAhPSA2XzUwMl8zOTI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXhwZWN0ZWQgdW5pcXVlIGNoYXQgcm93cyBtdXN0IGVxdWFsIDYsNTAyLDM5MiIpCiAgICBpZiByZWNpcGVfZXhwZWN0ZWRfcm93Y291bnRzLmdldCgibW9kZXNfcGVyX2Nhc2UiKSAhPSA0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFsbCBmb3VyIG1vZGVzIG11c3QgYmUgZXhwZWN0ZWQiKQoKICAgIGlmIHJhd19yZXBvcnQuZ2V0KCJzY2hlbWEiKSAhPSAiTWF0aFZpc2lvbktpZC5mdWxsX2N1cnJlbnRfYXJjaGl2ZV9hdWRpdC52MyI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidW5leHBlY3RlZCByYXcgcmVwb3J0IHNjaGVtYSIpCiAgICBpZiByYXdfcmVwb3J0LmdldCgiaWRlbnRpdHkiKSAhPSBFWFBFQ1RFRF9JREVOVElUWToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyYXcgcmVwb3J0IHJlY2lwZSBpZGVudGl0eSBkaWZmZXJzIikKICAgICMgVjMncyByYXcgZmxhZyBpcyBmYWxzZSBieSBkZXNpZ24gd2hlbmV2ZXIgYW55IGlzc3VlIGV4aXN0cy4gUHJlc2VydmUgdGhhdAogICAgIyByYXcgb3V0Y29tZSBhbmQgZGVyaXZlIHRoaXMgbmFycm93ZXIgc3RhdHVzIGFmdGVyIHZhbGlkYXRpbmcgdGhlIG9ubHkKICAgICMgcGVybWl0dGVkLCB6ZXJvLWxpbmsgcmVncmVzc2lvbl9vbmx5IGRpYWdub3N0aWMgYmVsb3cuCiAgICBpZiByYXdfcmVwb3J0LmdldCgiZnVsbF9hcmNoaXZlX3Rva2VuX2xpbmthZ2Vfc3BsaXRfYXVkaXRfY29tcGxldGUiKSBpcyBub3QgRmFsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmF3IFYzIGNvbXBsZXRpb24gZmxhZyBtdXN0IHJlbWFpbiBmYWxzZSB3aXRoIHRoZSBwcmVzZXJ2ZWQgaXNzdWUiKQogICAgcmVxdWlyZWQgPSB7CiAgICAgICAgIm1lbWJlcnNfZXhwZWN0ZWQiOiAyNTEsCiAgICAgICAgIm1lbWJlcnNfY29tcGxldGUiOiAyNTEsCiAgICAgICAgImFsbF9kZWNsYXJlZF9tZW1iZXJzX3NjYW5uZWQiOiBUcnVlLAogICAgICAgICJzb3VyY2VfdmVyc2lvbnNfaW5kZXhlZCI6IDFfNjI5XzkwNiwKICAgICAgICAidW5pcXVlX2NoYXRfcm93c19wcm9jZXNzZWQiOiA2XzUwMl8zOTIsCiAgICAgICAgInBoeXNpY2FsX2NoYXRfb2NjdXJyZW5jZXNfcHJvY2Vzc2VkIjogNl81MDJfMzkyLAogICAgICAgICJleGFjdF9kdXBsaWNhdGVfcmVmZXJlbmNlcyI6IDAsCiAgICAgICAgInNvdXJjZV9saW5rYWdlX3VudmVyaWZpZWRfcm93cyI6IDAsCiAgICAgICAgImluY29tcGxldGVfZm91cl9tb2RlX2Nhc2VzIjogMCwKICAgICAgICAibWF4X3NlcXVlbmNlX3Rva2VucyI6ICgiYXRfbW9zdCIsIDQwXzk2MCksCiAgICB9CiAgICBmb3IgZmllbGQsIGV4cGVjdGVkIGluIHJlcXVpcmVkLml0ZW1zKCk6CiAgICAgICAgYWN0dWFsID0gcmF3X3JlcG9ydC5nZXQoZmllbGQpCiAgICAgICAgaWYgaXNpbnN0YW5jZShleHBlY3RlZCwgdHVwbGUpOgogICAgICAgICAgICBpZiB0eXBlKGFjdHVhbCkgaXMgbm90IGludCBvciBub3QgMCA8IGFjdHVhbCA8PSBleHBlY3RlZFsxXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJyYXcgcmVwb3J0IHtmaWVsZH0gbXVzdCBiZSA8PSB7ZXhwZWN0ZWRbMV19IikKICAgICAgICBlbGlmIGFjdHVhbCAhPSBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInJhdyByZXBvcnQge2ZpZWxkfSBleHBlY3RlZCB7ZXhwZWN0ZWQhcn0sIGdvdCB7YWN0dWFsIXJ9IikKICAgIGlmIHR5cGUocmF3X3JlcG9ydC5nZXQoIm1heF9jb21wbGV0aW9uX3Rva2VucyIpKSBpcyBub3QgaW50IG9yIG5vdCAwIDwgcmF3X3JlcG9ydFsibWF4X2NvbXBsZXRpb25fdG9rZW5zIl0gPD0gcmF3X3JlcG9ydFsibWF4X3NlcXVlbmNlX3Rva2VucyJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJhdyByZXBvcnQgbXVzdCBjb250YWluIGEgcG9zaXRpdmUgbWF4X2NvbXBsZXRpb25fdG9rZW5zIikKICAgIGlmIHJhd19yZXBvcnQuZ2V0KCJhY3R1YWxfcm93c19ieV9zcGxpdCIpICE9IEVYUEVDVEVEX1NQTElUUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyYXcgcmVwb3J0IHNwbGl0IGNvdW50cyBkaWZmZXIgZnJvbSB0cmFpbi92YWxpZGF0aW9uL3Rlc3QgZXhwZWN0YXRpb25zIikKICAgIGlmIHJhd19yZXBvcnQuZ2V0KCJpc3N1ZXMiKSAhPSB7UkVHUkVTU0lPTl9JU1NVRVsia2luZCJdOiAxfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyYXcgaXNzdWUgYWdncmVnYXRlIG11c3QgY29udGFpbiBleGFjdGx5IG9uZSBwaW5uZWQgZGlhZ25vc3RpYyIpCiAgICBpZiBleGFjdF9pc3N1ZV9yb3dzICE9IFtSRUdSRVNTSU9OX0lTU1VFXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpc3N1ZSByb3dzIGRpZmZlciBmcm9tIHRoZSBvbmUgcGVybWl0dGVkIGV4YWN0IGRpYWdub3N0aWMiKQogICAgaWYgcmVncmVzc2lvbl9zb3VyY2VfbWV0YSAhPSB7CiAgICAgICAgInNhbXBsZV9pZCI6IFJFR1JFU1NJT05fU0FNUExFX0lELAogICAgICAgICJzcGxpdCI6ICJyZWdyZXNzaW9uX29ubHkiLAogICAgICAgICJyYXdfc2hhMjU2IjogUkVHUkVTU0lPTl9TSEEyNTYsCiAgICAgICAgImNhbm9uaWNhbF9zaGEyNTYiOiBSRUdSRVNTSU9OX1NIQTI1NiwKICAgIH06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVncmVzc2lvbiBzb3VyY2UgaWRlbnRpdHkgb3IgaGFzaGVzIGRpZmZlciBmcm9tIHRoZSBwaW5uZWQgZGlhZ25vc3RpYyIpCiAgICBpZiB6ZXJvX2xpbmtjb3VudHMgIT0ge1JFR1JFU1NJT05fU0FNUExFX0lEOiB7ImxpbmtlZF9jaGF0X3Jvd3MiOiAwfX06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGlhZ25vc3RpYyBzb3VyY2UgbXVzdCBoYXZlIGV4YWN0bHkgemVybyBsaW5rZWQgY2hhdHMiKQogICAgaWYgcmF3X3JlcG9ydC5nZXQoIlRSQUlOSU5HX0RBVEFTRVRfUkVBRFkiKSBpcyBub3QgRmFsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVFJBSU5JTkdfREFUQVNFVF9SRUFEWSBtdXN0IHJlbWFpbiBmYWxzZSIpCiAgICBpZiByYXdfcmVwb3J0LmdldCgiYWxsX3Jvd3Nfc2VtYW50aWNhbGx5X2h1bWFuX3Jldmlld2VkIikgaXMgbm90IEZhbHNlIG9yIHJhd19yZXBvcnQuZ2V0KCJ3aG9sZV9jdXJyaWN1bHVtX2NlcnRpZmllZCIpIGlzIG5vdCBGYWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJodW1hbi1yZXZpZXcgYW5kIHdob2xlLWN1cnJpY3VsdW0gZmxhZ3MgbXVzdCByZW1haW4gZmFsc2UiKQogICAgaWYgcmF3X3JlcG9ydC5nZXQoInRyYWluaW5nX3N0YXJ0ZWQiKSBpcyBub3QgRmFsc2Ugb3IgcmF3X3JlcG9ydC5nZXQoImdwdV91c2VkIikgaXMgbm90IEZhbHNlIG9yIHJhd19yZXBvcnQuZ2V0KCJtb2RlbF93ZWlnaHRzX2xvYWRlZCIpIGlzIG5vdCBGYWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmV4cGVjdGVkIHRyYWluaW5nL3dlaWdodHMgc3RhdGUgaW4gcmF3IHJlcG9ydCIpCgogICAgcmV0dXJuIHsKICAgICAgICAic3RhdHVzIjogUEFTU19MQUJFTCwKICAgICAgICAicmF3X3JlcG9ydF9wcmVzZXJ2ZWQiOiBkZWVwY29weShyYXdfcmVwb3J0KSwKICAgICAgICAiZXhhY3RfaXNzdWVfcm93c19wcmVzZXJ2ZWQiOiBkZWVwY29weShleGFjdF9pc3N1ZV9yb3dzKSwKICAgICAgICAiZGlhZ25vc3RpY19leGNsdXNpb24iOiB7InNhbXBsZV9pZCI6IFJFR1JFU1NJT05fU0FNUExFX0lELAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicmF3X3NoYTI1NiI6IFJFR1JFU1NJT05fU0hBMjU2LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY2Fub25pY2FsX3NoYTI1NiI6IFJFR1JFU1NJT05fU0hBMjU2LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibGlua2VkX2NoYXRfcm93cyI6IDB9LAogICAgICAgICJUUkFJTklOR19EQVRBU0VUX1JFQURZIjogRmFsc2UsCiAgICAgICAgInBoeXNpY2FsX2N1cnJpY3VsdW1fcGVuZGluZyI6IFRydWUsCiAgICB9CgoKY2xhc3MgQWRqdWRpY2F0aW9uVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHNldFVwKHNlbGYpOgogICAgICAgICMgQWN0dWFsIHBpbm5lZCByZWNpcGUgbWV0YWRhdGEgb25seTsgbm8gcXVlc3Rpb25zIG9yIGNvcnB1cyByb3dzLgogICAgICAgIHNlbGYucmVjaXBlID0gewogICAgICAgICAgICAibWVtYmVycyI6IGpzb24ubG9hZHMoUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJmaW5hbF9yZWNpcGVfbWVtYmVyX21ldGFkYXRhLmpzb24iKS5yZWFkX2J5dGVzKCkpLAogICAgICAgICAgICAidW5pcXVlX2NoYXRfcm93cyI6IDZfNTAyXzM5MiwKICAgICAgICAgICAgInVuaXF1ZV9jaGF0X3Jvd3NfYnlfc3BsaXQiOiBkaWN0KEVYUEVDVEVEX1NQTElUUyksCiAgICAgICAgICAgICJtb2Rlc19wZXJfY2FzZSI6IDQsCiAgICAgICAgfQogICAgICAgICMgMjE4Kjc0NDIgKyA3NTUwID0gMSw2MjksOTA2OyAzMSoyMDMxOTkgKyAyMDMyMjMgPSA2LDUwMiwzOTIuCiAgICAgICAgc2VsZi5yZXBvcnQgPSB7CiAgICAgICAgICAgICJzY2hlbWEiOiAiTWF0aFZpc2lvbktpZC5mdWxsX2N1cnJlbnRfYXJjaGl2ZV9hdWRpdC52MyIsCiAgICAgICAgICAgICJpZGVudGl0eSI6IEVYUEVDVEVEX0lERU5USVRZLAogICAgICAgICAgICAibWVtYmVyc19leHBlY3RlZCI6IDI1MSwgIm1lbWJlcnNfY29tcGxldGUiOiAyNTEsCiAgICAgICAgICAgICJhbGxfZGVjbGFyZWRfbWVtYmVyc19zY2FubmVkIjogVHJ1ZSwKICAgICAgICAgICAgInNvdXJjZV92ZXJzaW9uc19pbmRleGVkIjogMV82MjlfOTA2LAogICAgICAgICAgICAidW5pcXVlX2NoYXRfcm93c19wcm9jZXNzZWQiOiA2XzUwMl8zOTIsCiAgICAgICAgICAgICJhY3R1YWxfcm93c19ieV9zcGxpdCI6IGRpY3QoRVhQRUNURURfU1BMSVRTKSwKICAgICAgICAgICAgInBoeXNpY2FsX2NoYXRfb2NjdXJyZW5jZXNfcHJvY2Vzc2VkIjogNl81MDJfMzkyLAogICAgICAgICAgICAiZXhhY3RfZHVwbGljYXRlX3JlZmVyZW5jZXMiOiAwLAogICAgICAgICAgICAibWF4X3NlcXVlbmNlX3Rva2VucyI6IDQwXzk2MCwKICAgICAgICAgICAgIm1heF9jb21wbGV0aW9uX3Rva2VucyI6IDFfMjEzLAogICAgICAgICAgICAic291cmNlX2xpbmthZ2VfdW52ZXJpZmllZF9yb3dzIjogMCwKICAgICAgICAgICAgImluY29tcGxldGVfZm91cl9tb2RlX2Nhc2VzIjogMCwKICAgICAgICAgICAgImlzc3VlcyI6IHsiU09VUkNFX1NQTElUX01JU1NJTkdfT1JfVU5LTk9XTiI6IDF9LAogICAgICAgICAgICAiZnVsbF9hcmNoaXZlX3Rva2VuX2xpbmthZ2Vfc3BsaXRfYXVkaXRfY29tcGxldGUiOiBGYWxzZSwKICAgICAgICAgICAgImFsbF9yb3dzX3NlbWFudGljYWxseV9odW1hbl9yZXZpZXdlZCI6IEZhbHNlLAogICAgICAgICAgICAid2hvbGVfY3VycmljdWx1bV9jZXJ0aWZpZWQiOiBGYWxzZSwKICAgICAgICAgICAgIlRSQUlOSU5HX0RBVEFTRVRfUkVBRFkiOiBGYWxzZSwKICAgICAgICAgICAgInRyYWluaW5nX3N0YXJ0ZWQiOiBGYWxzZSwgImdwdV91c2VkIjogRmFsc2UsICJtb2RlbF93ZWlnaHRzX2xvYWRlZCI6IEZhbHNlLAogICAgICAgIH0KICAgICAgICBzZWxmLmlzc3VlcyA9IFtkaWN0KFJFR1JFU1NJT05fSVNTVUUpXQogICAgICAgIHNlbGYuc291cmNlID0geyJzYW1wbGVfaWQiOiBSRUdSRVNTSU9OX1NBTVBMRV9JRCwgInNwbGl0IjogInJlZ3Jlc3Npb25fb25seSIsCiAgICAgICAgICAgICAgICAgICAgICAgInJhd19zaGEyNTYiOiBSRUdSRVNTSU9OX1NIQTI1NiwgImNhbm9uaWNhbF9zaGEyNTYiOiBSRUdSRVNTSU9OX1NIQTI1Nn0KICAgICAgICBzZWxmLnplcm9fbGlua3MgPSB7UkVHUkVTU0lPTl9TQU1QTEVfSUQ6IHsibGlua2VkX2NoYXRfcm93cyI6IDB9fQoKICAgIGRlZiBjYWxsKHNlbGYsICoqb3ZlcnJpZGVzKToKICAgICAgICBhcmdzID0gZGljdChyYXdfcmVwb3J0PXNlbGYucmVwb3J0LCBleGFjdF9pc3N1ZV9yb3dzPXNlbGYuaXNzdWVzLAogICAgICAgICAgICAgICAgICAgIHJlZ3Jlc3Npb25fc291cmNlX21ldGE9c2VsZi5zb3VyY2UsIHplcm9fbGlua2NvdW50cz1zZWxmLnplcm9fbGlua3MsCiAgICAgICAgICAgICAgICAgICAgcmVjaXBlX2V4cGVjdGVkX3Jvd2NvdW50cz1zZWxmLnJlY2lwZSkKICAgICAgICBhcmdzLnVwZGF0ZShvdmVycmlkZXMpCiAgICAgICAgcmV0dXJuIGFkanVkaWNhdGVfZmluYWxfdjNfbWV0YWRhdGEoKiphcmdzKQoKICAgIGRlZiB0ZXN0X2FjY2VwdHNfZXhhY3RfcHJlc2VydmVkX2lzc3VlX2FuZF9yYXdfZmFsc2VfZmxhZyhzZWxmKToKICAgICAgICBiZWZvcmUgPSBkZWVwY29weSgoc2VsZi5yZXBvcnQsIHNlbGYuaXNzdWVzKSkKICAgICAgICByZXN1bHQgPSBzZWxmLmNhbGwoKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJzdGF0dXMiXSwgUEFTU19MQUJFTCkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHJlc3VsdFsicmF3X3JlcG9ydF9wcmVzZXJ2ZWQiXVsiZnVsbF9hcmNoaXZlX3Rva2VuX2xpbmthZ2Vfc3BsaXRfYXVkaXRfY29tcGxldGUiXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFsiZXhhY3RfaXNzdWVfcm93c19wcmVzZXJ2ZWQiXSwgc2VsZi5pc3N1ZXMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgoc2VsZi5yZXBvcnQsIHNlbGYuaXNzdWVzKSwgYmVmb3JlKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UocmVzdWx0WyJUUkFJTklOR19EQVRBU0VUX1JFQURZIl0pCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHJlc3VsdFsicGh5c2ljYWxfY3VycmljdWx1bV9wZW5kaW5nIl0pCgogICAgZGVmIHRlc3RfcmVqZWN0c19wYXJ0aWFsX2NvbXBsZXRpb24oc2VsZik6CiAgICAgICAgYmFkID0gZGVlcGNvcHkoc2VsZi5yZXBvcnQpOyBiYWRbIm1lbWJlcnNfY29tcGxldGUiXSA9IDI1MAogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChWYWx1ZUVycm9yLCAibWVtYmVyc19jb21wbGV0ZSIpOgogICAgICAgICAgICBzZWxmLmNhbGwocmF3X3JlcG9ydD1iYWQpCgogICAgZGVmIHRlc3RfcmVqZWN0c193cm9uZ19pZGVudGl0eV9hbmRfcmF3X3RydWVfZmxhZyhzZWxmKToKICAgICAgICBiYWQgPSBkZWVwY29weShzZWxmLnJlcG9ydCk7IGJhZFsiaWRlbnRpdHkiXSA9ICIwIiAqIDY0CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJpZGVudGl0eSIpOgogICAgICAgICAgICBzZWxmLmNhbGwocmF3X3JlcG9ydD1iYWQpCiAgICAgICAgYmFkID0gZGVlcGNvcHkoc2VsZi5yZXBvcnQpOyBiYWRbImZ1bGxfYXJjaGl2ZV90b2tlbl9saW5rYWdlX3NwbGl0X2F1ZGl0X2NvbXBsZXRlIl0gPSBUcnVlCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJyYXcgVjMgY29tcGxldGlvbiBmbGFnIik6CiAgICAgICAgICAgIHNlbGYuY2FsbChyYXdfcmVwb3J0PWJhZCkKCiAgICBkZWYgdGVzdF9yZWplY3RzX3dyb25nX3JvbGVzKHNlbGYpOgogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVjaXBlKTsgYmFkWyJtZW1iZXJzIl1bMF1bInJvbGUiXSA9ICJjYW5kaWRhdGVfY2hhdCIKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzUmVnZXgoVmFsdWVFcnJvciwgIjIxOSBzb3VyY2UgbWVtYmVycyIpOgogICAgICAgICAgICBzZWxmLmNhbGwocmVjaXBlX2V4cGVjdGVkX3Jvd2NvdW50cz1iYWQpCgogICAgZGVmIHRlc3RfcmVqZWN0c191bnBpbm5lZF9tZW1iZXJfb3JfY291bnRlcmJhbGFuY2VkX3Jvd3Moc2VsZik6CiAgICAgICAgYmFkID0gZGVlcGNvcHkoc2VsZi5yZWNpcGUpOyBiYWRbIm1lbWJlcnMiXVswXVsiaWQiXSA9ICJpbnZlbnRlZCIKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzUmVnZXgoVmFsdWVFcnJvciwgInBpbm5lZCBtZW1iZXIiKToKICAgICAgICAgICAgc2VsZi5jYWxsKHJlY2lwZV9leHBlY3RlZF9yb3djb3VudHM9YmFkKQogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVjaXBlKQogICAgICAgIGJhZFsibWVtYmVycyJdWzBdWyJyb3dzIl0gKz0gMTsgYmFkWyJtZW1iZXJzIl1bMV1bInJvd3MiXSAtPSAxCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJwaW5uZWQgbWVtYmVyIik6CiAgICAgICAgICAgIHNlbGYuY2FsbChyZWNpcGVfZXhwZWN0ZWRfcm93Y291bnRzPWJhZCkKCiAgICBkZWYgdGVzdF9yZWplY3RzX2Jvb2xfb3JfZmFsc2VfaHVtYW5fY2xhaW0oc2VsZik6CiAgICAgICAgZm9yIGZpZWxkIGluICgibWF4X3NlcXVlbmNlX3Rva2VucyIsICJtYXhfY29tcGxldGlvbl90b2tlbnMiKToKICAgICAgICAgICAgYmFkID0gZGVlcGNvcHkoc2VsZi5yZXBvcnQpOyBiYWRbZmllbGRdID0gVHJ1ZQogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOiBzZWxmLmNhbGwocmF3X3JlcG9ydD1iYWQpCiAgICAgICAgZm9yIGZpZWxkIGluICgiYWxsX3Jvd3Nfc2VtYW50aWNhbGx5X2h1bWFuX3Jldmlld2VkIiwgIndob2xlX2N1cnJpY3VsdW1fY2VydGlmaWVkIik6CiAgICAgICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVwb3J0KTsgYmFkW2ZpZWxkXSA9IFRydWUKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTogc2VsZi5jYWxsKHJhd19yZXBvcnQ9YmFkKQoKICAgIGRlZiB0ZXN0X3JlamVjdHNfd3Jvbmdfc3BsaXRfY291bnRzKHNlbGYpOgogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVwb3J0KTsgYmFkWyJhY3R1YWxfcm93c19ieV9zcGxpdCJdWyJ2YWxpZGF0aW9uIl0gKz0gMQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChWYWx1ZUVycm9yLCAic3BsaXQgY291bnRzIik6CiAgICAgICAgICAgIHNlbGYuY2FsbChyYXdfcmVwb3J0PWJhZCkKCiAgICBkZWYgdGVzdF9yZWplY3RzX3dyb25nX3NvdXJjZV9oYXNoKHNlbGYpOgogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYuc291cmNlKTsgYmFkWyJyYXdfc2hhMjU2Il0gPSAiMCIgKiA2NAogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChWYWx1ZUVycm9yLCAicmVncmVzc2lvbiBzb3VyY2UgaWRlbnRpdHkiKToKICAgICAgICAgICAgc2VsZi5jYWxsKHJlZ3Jlc3Npb25fc291cmNlX21ldGE9YmFkKQoKICAgIGRlZiB0ZXN0X3JlamVjdHNfbGlua2VkX2RpYWdub3N0aWMoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJ6ZXJvIGxpbmtlZCBjaGF0cyIpOgogICAgICAgICAgICBzZWxmLmNhbGwoemVyb19saW5rY291bnRzPXtSRUdSRVNTSU9OX1NBTVBMRV9JRDogeyJsaW5rZWRfY2hhdF9yb3dzIjogMX19KQoKICAgIGRlZiB0ZXN0X3JlamVjdHNfZXh0cmFfaXNzdWUoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJpc3N1ZSByb3dzIGRpZmZlciIpOgogICAgICAgICAgICBzZWxmLmNhbGwoZXhhY3RfaXNzdWVfcm93cz1zZWxmLmlzc3VlcyArIFt7Im1lbWJlciI6IngiLCJsaW5lIjoyLCJraW5kIjoiTkVXIiwiZGV0YWlsIjoieCJ9XSkKCiAgICBkZWYgdGVzdF9yZWplY3RzX21pc3NpbmdfbW9kZXNfYW5kX2NvdW50X21pc21hdGNoKHNlbGYpOgogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVwb3J0KTsgYmFkWyJpbmNvbXBsZXRlX2ZvdXJfbW9kZV9jYXNlcyJdID0gMQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChWYWx1ZUVycm9yLCAiaW5jb21wbGV0ZV9mb3VyX21vZGVfY2FzZXMiKToKICAgICAgICAgICAgc2VsZi5jYWxsKHJhd19yZXBvcnQ9YmFkKQogICAgICAgIGJhZCA9IGRlZXBjb3B5KHNlbGYucmVwb3J0KTsgYmFkWyJwaHlzaWNhbF9jaGF0X29jY3VycmVuY2VzX3Byb2Nlc3NlZCJdIC09IDEKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzUmVnZXgoVmFsdWVFcnJvciwgInBoeXNpY2FsX2NoYXRfb2NjdXJyZW5jZXNfcHJvY2Vzc2VkIik6CiAgICAgICAgICAgIHNlbGYuY2FsbChyYXdfcmVwb3J0PWJhZCkKCiAgICBkZWYgdGVzdF9yZWplY3RzX3dyb25nX21heF9zZXF1ZW5jZV9hbmRfbm9ucG9zaXRpdmVfY29tcGxldGlvbihzZWxmKToKICAgICAgICBiYWQgPSBkZWVwY29weShzZWxmLnJlcG9ydCk7IGJhZFsibWF4X3NlcXVlbmNlX3Rva2VucyJdID0gNDBfOTYxCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJtYXhfc2VxdWVuY2VfdG9rZW5zIik6CiAgICAgICAgICAgIHNlbGYuY2FsbChyYXdfcmVwb3J0PWJhZCkKICAgICAgICBiYWQgPSBkZWVwY29weShzZWxmLnJlcG9ydCk7IGJhZFsibWF4X2NvbXBsZXRpb25fdG9rZW5zIl0gPSAwCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlc1JlZ2V4KFZhbHVlRXJyb3IsICJtYXhfY29tcGxldGlvbl90b2tlbnMiKToKICAgICAgICAgICAgc2VsZi5jYWxsKHJhd19yZXBvcnQ9YmFkKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByZXN1bHQgPSB1bml0dGVzdC5UZXh0VGVzdFJ1bm5lcih2ZXJib3NpdHk9MikucnVuKAogICAgICAgIHVuaXR0ZXN0LmRlZmF1bHRUZXN0TG9hZGVyLmxvYWRUZXN0c0Zyb21UZXN0Q2FzZShBZGp1ZGljYXRpb25UZXN0cykpCiAgICByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgcmVzdWx0Lndhc1N1Y2Nlc3NmdWwoKSBlbHNlIDEpDQo='))
(qa_dir/"final_source_chat_antijoin.py").write_bytes(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWFkLW9ubHksIGJvdW5kZWQgbWV0YWRhdGEgcmVjb25jaWxpYXRpb24gaGVscGVyOyBpbnRlbnRpb25hbGx5IG5vdCBydW4gaGVyZS4KClJlYWRzIG9ubHkgVjMgc291cmNlX3ZlcnNpb25zKGlkLHJhd19zaGEsY2Fub25pY2FsX3NoYSxzaWduYXR1cmVzLHNvdXJjZV9zcGxpdCkKYW5kIHNhbXBsZXMoY2FzZV9pZCxtb2RlLHNwbGl0KSwgcGx1cyB0d28gU0hBLTI1Ni12ZXJpZmllZCBzaWduYXR1cmUgaW5kZXhlcy4KSXQgbmV2ZXIgcmVhZHMgcHJvamVjdGlvbiwgcHJvbXB0cywgYW5zd2VycywgYXJjaGl2ZSBwYXlsb2Fkcywgb3IgZXZhbHVhdGlvbiB0ZXh0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgYXJncGFyc2UsIGhhc2hsaWIsIGpzb24sIHNxbGl0ZTMKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClBST1RFQ1RFRF9TSEEyNTYgPSAiNTZhNTY5MWMxYjNiNTY2ZTM3MzU1OWI5Yzg4M2Q1ZWFjZGZmMzUzNDczY2NhNjY4NjhmMjk4OGFiM2UzNjhiOCIKSE9MRE9VVF9TSEEyNTYgPSAiYTVjMTU0MjBhOTYwNmYyN2JkY2ZhZGQwMGY0NGI1ZWVhMGIyM2EyYjUyODQ1OGU3M2E0YTFjNDhkNjMwNDI2MyIKIyBUaGVzZSBhcmUgdGhlIHBhdGhzIGltcGxpZWQgYnkgZXhoYXVzdGl2ZV9jbG91ZF9yZXN1bWUucHkncyBNRVRBIGFuZCBSVU4uCiMgU3VwcGx5IGV4YWN0IGZyZXNoIGluZGV4IGZpbGVzIGZyb20gdGhlIHZlcmlmaWVkIHJ1bnRpbWU7IGxvY2FsIGZpbGVzIGFyZSBhYnNlbnQuClBST1RFQ1RFRF9QQVRIID0gUGF0aCgidGhpbmtpbmcvY2xvdWRfcHVibGljYXRpb24vcmVzdW1lXzIwMjYxMDE4L21ldGFkYXRhL3RoaW5raW5nL2RhdGEvZXhoYXVzdGl2ZV9tYXRoX3YxL3Byb3RlY3RlZF9udW1lcmljX3NpZ25hdHVyZXMuanNvbiIpCkhPTERPVVRfUEFUSCA9IFBhdGgoInRoaW5raW5nL2Nsb3VkX3B1YmxpY2F0aW9uL3Jlc3VtZV8yMDI2MTAxOC9ob2xkb3V0X3NpZ25hdHVyZXMuanNvbiIpCk1BWF9VTkxJTktFRCA9IDUwMDAKRVhQRUNURURfVU5MSU5LRUQgPSA0MzA4ICAjIGN1cnJlbnQgb2JzZXJ2ZWQgZ2FwOiA0MzA3IHRyYWluX2NhbmRpZGF0ZSArIDEgcmVncmVzc2lvbl9vbmx5CgpkZWYgc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBmOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKZGVmIGxvYWRfc2lnbmF0dXJlcyhwYXRoOiBQYXRoLCBleHBlY3RlZDogc3RyKSAtPiBzZXRbc3RyXToKICAgIGFjdHVhbCA9IHNoYTI1NihwYXRoKQogICAgaWYgYWN0dWFsICE9IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJpbmRleCBoYXNoIG1pc21hdGNoIGZvciB7cGF0aH06IHthY3R1YWx9IikKICAgIG9iaiA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoInV0Zi04IikpCiAgICB2YWx1ZXMgPSBvYmouZ2V0KCJzaWduYXR1cmVzIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlcywgbGlzdCkgb3IgYW55KG5vdCBpc2luc3RhbmNlKHgsIHN0cikgZm9yIHggaW4gdmFsdWVzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5leHBlY3RlZCBzaWduYXR1cmUgaW5kZXggc2NoZW1hOiB7cGF0aH0iKQogICAgcmV0dXJuIHNldCh2YWx1ZXMpCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiZGF0YWJhc2UiLCB0eXBlPVBhdGgsIGhlbHA9ImZyZXNobHkgdmVyaWZpZWQgVjMgY2hlY2twb2ludCBjb3B5IikKICAgIGFwLmFkZF9hcmd1bWVudCgib3V0cHV0IiwgdHlwZT1QYXRoLCBoZWxwPSJuZXcgYm91bmRlZCBKU09OIHJlcG9ydCBwYXRoIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcm90ZWN0ZWQtaW5kZXgiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UFJPVEVDVEVEX1BBVEgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taG9sZG91dC1pbmRleCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1IT0xET1VUX1BBVEgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXhwZWN0ZWQtdW5saW5rZWQiLCB0eXBlPWludCwgZGVmYXVsdD1FWFBFQ1RFRF9VTkxJTktFRCkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIHByb3RlY3RlZCA9IGxvYWRfc2lnbmF0dXJlcyhhcmdzLnByb3RlY3RlZF9pbmRleCwgUFJPVEVDVEVEX1NIQTI1NikKICAgIGhvbGRvdXQgPSBsb2FkX3NpZ25hdHVyZXMoYXJncy5ob2xkb3V0X2luZGV4LCBIT0xET1VUX1NIQTI1NikKICAgIGRiID0gc3FsaXRlMy5jb25uZWN0KGYiZmlsZTp7YXJncy5kYXRhYmFzZS5yZXNvbHZlKCkuYXNfcG9zaXgoKX0/bW9kZT1ybyIsIHVyaT1UcnVlKQogICAgZGIuZXhlY3V0ZSgiUFJBR01BIHF1ZXJ5X29ubHk9T04iKQogICAgdHJ5OgogICAgICAgIGlkZW50aXR5ID0gZGIuZXhlY3V0ZSgiU0VMRUNUIGlkZW50aXR5IEZST00gY29uZmlnIikuZmV0Y2hvbmUoKQogICAgICAgIGlmIGlkZW50aXR5ICE9ICgiZjAzMzJkZjkxMDIwYWNlMGZjNDhkODg0OTFmZTk3MzY0ZDA4ODI2NDU4ZDA4YzcxYmE0ZWM2MDUyNGUyNTg2MyIsKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidW5leHBlY3RlZCBleGFjdCBWMyByZWNpcGUgaWRlbnRpdHkiKQogICAgICAgIGlmIGRiLmV4ZWN1dGUoIlNFTEVDVCBjb3VudCgqKSBGUk9NIG1lbWJlcnMgV0hFUkUgY29tcGxldGU9MSBBTkQgbmV4dF9saW5lPWV4cGVjdGVkX3Jvd3MiKS5mZXRjaG9uZSgpWzBdICE9IDI1MToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYWxsIDI1MSBleGFjdCBtZW1iZXIgY3Vyc29ycyBtdXN0IGJlIGNvbXBsZXRlIGJlZm9yZSBhbnRpLWpvaW4iKQogICAgICAgIGlmIGRiLmV4ZWN1dGUoIlNFTEVDVCBjb3VudCgqKSBGUk9NIHNhbXBsZXMiKS5mZXRjaG9uZSgpWzBdICE9IDY1MDIzOTI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImZ1bGwgdW5pcXVlIGNoYXQgc2NhbiBpcyByZXF1aXJlZCBiZWZvcmUgYW50aS1qb2luIikKICAgICAgICBjb2xzID0ge3JbMV0gZm9yIHIgaW4gZGIuZXhlY3V0ZSgiUFJBR01BIHRhYmxlX2luZm8oc291cmNlX3ZlcnNpb25zKSIpfQogICAgICAgIHJlcXVpcmVkID0geyJpZCIsICJyYXdfc2hhIiwgImNhbm9uaWNhbF9zaGEiLCAic2lnbmF0dXJlcyIsICJzb3VyY2Vfc3BsaXQifQogICAgICAgIGlmIG5vdCByZXF1aXJlZCA8PSBjb2xzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVjMgc291cmNlX3ZlcnNpb25zIG1pc3NpbmcgcmVxdWlyZWQgbWV0YWRhdGEgY29sdW1uczoge3NvcnRlZChyZXF1aXJlZC1jb2xzKX0iKQogICAgICAgIGR1cGxpY2F0ZXMgPSBkYi5leGVjdXRlKCJTRUxFQ1QgQ09VTlQoKikgRlJPTSAoU0VMRUNUIGlkIEZST00gc291cmNlX3ZlcnNpb25zIEdST1VQIEJZIGlkIEhBVklORyBDT1VOVCgqKTw+MSkiKS5mZXRjaG9uZSgpWzBdCiAgICAgICAgaWYgZHVwbGljYXRlczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInNvdXJjZSBJRHMgd2l0aCBtdWx0aXBsZSB2ZXJzaW9ucyByZXF1aXJlIHNlcGFyYXRlIGFkanVkaWNhdGlvbjoge2R1cGxpY2F0ZXN9IikKICAgICAgICAjIE5vIHNvdXJjZSBtZW1iZXIgaXMgcmVjb3JkZWQgb24gc291cmNlX3ZlcnNpb25zOiBmYW1pbHkgYXR0cmlidXRpb24gaXMgaW1wb3NzaWJsZSBoZXJlLgogICAgICAgIHJvd3MgPSBkYi5leGVjdXRlKCIiIgogICAgICAgICAgICBTRUxFQ1Qgdi5pZCx2LnJhd19zaGEsdi5jYW5vbmljYWxfc2hhLHYuc291cmNlX3NwbGl0LHYuc2lnbmF0dXJlcywKICAgICAgICAgICAgICAgICAgIENPQUxFU0NFKGMubGlua2VkX2Nhc2VzLDApLENPQUxFU0NFKGMuY2hhdF9yb3dzLDApCiAgICAgICAgICAgIEZST00gc291cmNlX3ZlcnNpb25zIHYKICAgICAgICAgICAgTEVGVCBKT0lOICgKICAgICAgICAgICAgICBTRUxFQ1QgY2FzZV9pZCxDT1VOVChESVNUSU5DVCBjYXNlX2lkKSBBUyBsaW5rZWRfY2FzZXMsQ09VTlQoKikgQVMgY2hhdF9yb3dzCiAgICAgICAgICAgICAgRlJPTSBzYW1wbGVzIEdST1VQIEJZIGNhc2VfaWQKICAgICAgICAgICAgKSBjIE9OIGMuY2FzZV9pZD12LmlkCiAgICAgICAgICAgIFdIRVJFIENPQUxFU0NFKGMubGlua2VkX2Nhc2VzLDApPTAKICAgICAgICAgICAgT1JERVIgQlkgdi5pZCx2LnJhd19zaGEKICAgICAgICAgICAgTElNSVQgNTAwMQogICAgICAgICIiIikuZmV0Y2hhbGwoKQogICAgICAgIGlmIGxlbihyb3dzKSA+IE1BWF9VTkxJTktFRCBvciBsZW4ocm93cykgIT0gYXJncy5leHBlY3RlZF91bmxpbmtlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVubGlua2VkIHNvdXJjZSBjb3VudCB7bGVuKHJvd3MpfSBvdXRzaWRlIGV4cGVjdGVkIGJvdW5kZWQgdG90YWwge2FyZ3MuZXhwZWN0ZWRfdW5saW5rZWR9IChjYXAge01BWF9VTkxJTktFRH0pIikKICAgICAgICBjb3VudHMgPSBDb3VudGVyKCk7IGl0ZW1zID0gW10KICAgICAgICBmb3Igc2lkLCByYXdfc2hhLCBjYW5vbmljYWxfc2hhLCBzcGxpdCwgc2lnbmF0dXJlc19qc29uLCBsaW5rZWRfY2FzZXMsIGNoYXRfcm93cyBpbiByb3dzOgogICAgICAgICAgICBzaWdzID0gc2V0KGpzb24ubG9hZHMoc2lnbmF0dXJlc19qc29uIG9yICJbXSIpKQogICAgICAgICAgICBpZiBzcGxpdCA9PSAicmVncmVzc2lvbl9vbmx5IjoKICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJyZWdyZXNzaW9uX29ubHkiCiAgICAgICAgICAgIGVsaWYgc2lncyAmIHByb3RlY3RlZDoKICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJwcm90ZWN0ZWRfbnVtZXJpYyIKICAgICAgICAgICAgZWxpZiBzaWdzICYgaG9sZG91dDoKICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJzZWxlY3RlZF9ob2xkb3V0IgogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmVhc29uID0gIlVOUkVTT0xWRURfUVVBUkFOVElORSIKICAgICAgICAgICAgY291bnRzW3JlYXNvbl0gKz0gMQogICAgICAgICAgICBpdGVtcy5hcHBlbmQoeyJpZCI6c2lkLCJyYXdfc2hhIjpyYXdfc2hhLCJjYW5vbmljYWxfc2hhIjpjYW5vbmljYWxfc2hhLCJzb3VyY2Vfc3BsaXQiOnNwbGl0LAogICAgICAgICAgICAgICAgICAgICAgICAgICJzaWduYXR1cmVzIjpzb3J0ZWQoc2lncyksImxpbmtlZF9jYXNlX2NvdW50IjpsaW5rZWRfY2FzZXMsImxpbmtlZF9jaGF0X3Jvd3MiOmNoYXRfcm93cywKICAgICAgICAgICAgICAgICAgICAgICAgICAiY2xhc3NpZmljYXRpb24iOnJlYXNvbn0pCiAgICAgICAgcmVwb3J0ID0geyJzY2hlbWEiOiJzb3VyY2VfY2hhdF9hbnRpam9pbl9jbGFzc2lmaWNhdGlvbi52MSIsInJlYWRfb25seSI6VHJ1ZSwKICAgICAgICAgICAgICAgICAgInBheWxvYWRfb3JfcXVlc3Rpb25fY29udGVudF9yZWFkIjpGYWxzZSwic291cmNlX21lbWJlcl9wcm92ZW5hbmNlX2F2YWlsYWJsZSI6RmFsc2UsCiAgICAgICAgICAgICAgICAgICJzb3VyY2VfbWVtYmVyX3Byb3ZlbmFuY2VfbGltaXRhdGlvbiI6IlYzIHNvdXJjZV92ZXJzaW9ucyBoYXMgbm8gbWVtYmVyL2ZhbWlseSBjb2x1bW47IHRoaXMgcmVwb3J0IGNhbm5vdCBhc3NpZ24gemVyby1jaGF0IHNvdXJjZSBJRHMgdG8gcmVjaXBlIGZhbWlsaWVzLiIsCiAgICAgICAgICAgICAgICAgICJpbmRleF9waW5zIjp7InByb3RlY3RlZCI6eyJwYXRoIjpzdHIoYXJncy5wcm90ZWN0ZWRfaW5kZXgpLCJzaGEyNTYiOlBST1RFQ1RFRF9TSEEyNTZ9LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWxlY3RlZF9ob2xkb3V0Ijp7InBhdGgiOnN0cihhcmdzLmhvbGRvdXRfaW5kZXgpLCJzaGEyNTYiOkhPTERPVVRfU0hBMjU2fX0sCiAgICAgICAgICAgICAgICAgICJjb3VudHMiOnsidW5saW5rZWRfc291cmNlX3ZlcnNpb25zIjpsZW4ocm93cyksImNsYXNzaWZpY2F0aW9ucyI6ZGljdChzb3J0ZWQoY291bnRzLml0ZW1zKCkpKX0sCiAgICAgICAgICAgICAgICAgICJ1bmxpbmtlZCI6aXRlbXMsCiAgICAgICAgICAgICAgICAgICJwb2xpY3kiOiJPbmx5IGV4cGxpY2l0IHJlZ3Jlc3Npb24vcHJvdGVjdGVkL2hvbGRvdXQgbWF0Y2hlcyBnZXQgdGhvc2UgY2xhc3NpZmljYXRpb25zLiBFdmVyeSByZXNpZHVhbCBzdGF5cyBxdWFyYW50aW5lZDsgbm8gY2FuZGlkYXRlIGlzIHByb21vdGVkIG9yIGFkZGVkIHRvIHRyYWluLiJ9CiAgICAgICAgYXJncy5vdXRwdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxlbnN1cmVfYXNjaWk9RmFsc2UsaW5kZW50PTIpKyJcbiIsZW5jb2Rpbmc9InV0Zi04IikKICAgIGZpbmFsbHk6CiAgICAgICAgZGIuY2xvc2UoKQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQ0K'))
(qa_dir/"final_recipe_member_metadata.json").write_bytes(base64.b64decode('...'))
(qa_dir/"protected_numeric_signatures.json").write_bytes(base64.b64decode('...'))
(qa_dir/"holdout_signatures.json").write_bytes(base64.b64decode('ew0KICAic2lnbmF0dXJlcyI6IFsNCiAgICAiMDQzNTliNWQ4YmQyNTA2M2U3MTI2MTg0MDhhYTYyYmRhNmM1YmQwYTlkN2RhYTE3MWYzYTgxMmYzNTY3ZWZlYSIsDQogICAgIjBkNzgxYzBlM2Y4ODNkZjZiMjE1MmU0MDZkZWM2YTFmOGVmZGYxYjNlOTNiMTM4YWI3NzJiNjU5NDNiYWM0YjEiLA0KICAgICIwZTI4ZjY0ODBiMGQzYTE2MGJjOGFkYjgxNDQxYzE0NTk0YTMzMzNjZGZjN2RmOWRmMWY5OGM1MWY2YTM2MDA1IiwNCiAgICAiMGU5NWE2NDk5YmU2OTVlYzQ3ZjQzOGNhMjE1YmU2Njg2YzIxYzdmNWI1MGIzYTI2ZTMyNzc4YzEyM2IyZmVhOSIsDQogICAgIjEzNTk1NWMxMTI2OTA1ZDcwYjQ4NDIzMmE0MzBiNzlhMmU1ODE2ZmFmZmNkMTg3ZjczNzU3ZTNkOTE2MTdkNTEiLA0KICAgICIxM2IwZGY2NTY2MWYwNjEyYTY2OWUzYWU4OWZkOGYyZTMxMzZiNzBlNDA3YmE3YTg2YzQ2ZjM4NjA0ZTg3ZGU2IiwNCiAgICAiMTNkNGIxMDMzN2NlN2ZhNzhlYmFhZWNmMjY0ZTBhODNhNWIyZDBhYjk0MGY5Y2VkNzliMDhmNDI0OGQ5MjRhZCIsDQogICAgIjEzZDYwZDA1OWJhNDc1M2I2NDc5Zjg3YjJmMjAxN2M2MTk3N2EwMzFjNDYwMmM2YjhjMDRjYTI1ZjcxODE1Y2UiLA0KICAgICIxNDhiMWQ5NjAyZTZhOTJhYjUxMjZkZjc4NDNiY2E4NTFlMjlhY2Q3YjY5Y2ZhOWM5YTg1NDk0OTA1NDRkOTdmIiwNCiAgICAiMTQ4YmFiNDliODNlMTE5NmE5NTJkMjg3OWEwYzhhYTM1Njc4NmRlMGMyMDY2Nzc2NTUxMzU5OTE0NmZiMzMyYyIsDQogICAgIjE1OTUxNTg1MDVjYWVhNTEyOWY2Y2RiOGZhN2ExNzg1MmU2NTc3YTA5YmM1OGM1NWVjOWI4MDkyZmJiM2YzN2YiLA0KICAgICIxNjJmZDAzYTQ3MGZmY2ViYzI3N2Y4MjM4OTVmZmFhNGYwYzk3ZjEyNzIyMmZmOTA3NDQ3ZWMxY2IwZjE1N2M3IiwNCiAgICAiMTczNTQzNDA0MDAyNzQ5YWNiMTAyMmE4ZmNkZWNhNDZlZTNmYjc3ZjY3OTQ2NGMwNmZlZGM0ODY1YTZjYmQ2NSIsDQogICAgIjE3YzA4NjcxMmZmNWZkZmM0OGE5NjYwMTU1OWUyODA5YjJhMWZjMThjMGJmMzk1ODBiZThkZGQzYzdiYzdjYjgiLA0KICAgICIxN2ZmNGZjOGQxODg3ZjcxNTEyYzBlNzJjZmRmMDk0Y2QzZWRhZTFkNTRjNjBiYzhhNmNkOTI5MDhjMDExNDYwIiwNCiAgICAiMTg3Zjc3YzE0ODlmZjEwMDc3OTczZWQwZGNlMzdlYTIzYzM5ODdkMjZjNTBkMjIwZjMxZmU3OTdiZGNlNGMwOCIsDQogICAgIjFhZjBkNjEyYTEyN2VkMWMzOGNhMTlhOWM3ZGJlNjdhZDNiYmI0YTJjZjZjNjlhNDk1MDc1Mjc5YzEwZjJjYmUiLA0KICAgICIxZTY1YWUzYzgwMGI3ZjQwMWNlNDAxNWVjMjRjMTI2ZjJkYWY1N2QyNDQ1MDIxOGJlYjYzZmRmOTcwYTU0YmNmIiwNCiAgICAiMWZmNGQ4Y2ZkNzMzMTkwOWEzMGE4NDNhMTZmMWY0YjU5NTlkYjgwZTdhODdjNWM1OWZlOWYwZWFiNTQ1ZGRhYiIsDQogICAgIjIzOTI4MjU1ZGQxZDhkM2E0MzliNTM4ZDUzMTkzMzQ1ZTA0OTQ0YjE4Yjc4MTUzM2M1NjEwNDBlMDg3YjBkODciLA0KICAgICIyMzk1NGMxNWMzYjVkOTZlMzU3OTI0ODc0OTdjMWJlYmEzMzFlYmM5MDhhMmI5OWFjMGNhMDAzMDc5YTUyMTdiIiwNCiAgICAiMjQyMTI2YjI3OWJhNGEyMzE5NmFmM2RkMGE2MjEzMGU3MTJlNzg0MGQ3YjZlMmRmM2MxZWU1NmJhMmVmOTMyMiIsDQogICAgIjI0NWI3MmQ1MGQ2MmFlODQzMGVlOWRiNTZlMzAwYWRlOTAzYjE4M2I4ZjYxMTkwZWZkZGI5M2YyZDE4Y2Y5MTUiLA0KICAgICIyNGRjOGUwMGM4YzRmNjBlZWFmYTc3NWRlNDE0MGZhZjA4MTlhNjNiNjM0YTQ0NDE4MTlhMzIyYmYyMzQyZWJlIiwNCiAgICAiMjUxMzIwMzc2NzUwOTE5MDBkMzZkNThlNTU4Y2FlYzVlMDc5NDU1YmY3ZjA2MzFjNGJjOWFiYzVkZDNiNDVjMiIsDQogICAgIjI1NzY0ZjViNjI4MzEzNDViOTllZmQ4NGQzNTU2YzBkZWI3ZjhlYjIzMzlmYmNlNTZiZDUxNmI5OTVhZTRmOWUiLA0KICAgICIyODkwOTc1YzhkOGRiYWNlM2UwZWJiMDM3ZTZhN2Y4YWQ1NTVhM2NhODcwNjU0ODNmM2I2YmM5MzY3ZjI4ZjM0IiwNCiAgICAiMjlkZjI1MWRiYjk4OTBjY2UwYTNlODU5ZmY4ZTg2YmU4ZmZjODliY2FkNDA1NmZmNWY0YWUxOGFiYmM4MjU3ZSIsDQogICAgIjJjNjliYjM2Y2RiNTdiZjA3ZDEwNDM1MGMzZjA1ZTY3YTY5ZDQ0YzdmMWVjMTNjMjRlYTY3NmMyMWJhNzhiNTciLA0KICAgICIyY2JjZDk2ZGFhZTNlYzVlODJhM2VhNjY1MTJjOTY0N2IzNzRhOTMwMmI5NjIwNTkxOTM1YzZlMGFiODkwMjM4IiwNCiAgICAiMmZjZWIxOGM1NGZlNWI1NWEwMTY4M2VlNGMzYmZkZjVjYzdjZGQ2YWMwMTM0MGYzYzViOGY2ZDMyOTBlNzViMSIsDQogICAgIjMxNWI0ZDliMWQwZjYzNTU0Njg0YzcwMjlkNzI3OTMwZjkyZGQwMjI4ODVkYmVhZjdlMmYzZThiOGEyZGE3MzAiLA0KICAgICIzMWY3OGJiYWM0ZjExNjE0YzRkYmU3ZmE0MTViMDg2MzcwZDY2N2MyYjMyMGU5ZWVmYWQ0MDFiMmUxYzIyOTIyIiwNCiAgICAiMzJhMTcyMjg5ZDdlY2QwMGE5N2NkMWNmMDE3NDllNWY5NmUxZmFkY2E2MDlhMTA0ZTQ0ODNlMzczNjQ1Nzc1ZCIsDQogICAgIjMzMWZkOTI2ZmZlOGQzM2ZmNGU4NTlkNWM0MjRhYWE3ZjI5ZGRiY2Y2MzA5ZDc0MzEwOTBiODFmOGNiYTgwOGQiLA0KICAgICIzNDlmODk1MGNlOTFlMzRlOGQzNTAxMzZjYzkxMDZlYTZmZTg2NGUxZDBkMjczZGJlOThjOGUzZmZlMGRjYWI2IiwNCiAgICAiMzRmYTYzYzY1YjE4MDdmMjdlNDdlODExOGVjN2Y4MzYyOGI3MWViMWYzZjQxOGI5ODBhNjU0NDMyMTc3ZWEyMCIsDQogICAgIjM2ZWNlNDVlZTVlNjBjNTc4NTNkMTVlNWI4NzQ0MTBlNmE2MDQ0OTc1NDZiOTc5OTBkMGExYjUxOTkxNTNkOGYiLA0KICAgICIzN2JiOThiMDIwYzBjZmM4M2U3ZjgxMTNjMzc0NDk1NmUxNDQ3NDgxMjU0Mjc0Mjg5ZjQyMWUwZGE1NDFiODFkIiwNCiAgICAiMzdlNDNkNzkyOTVmYWYwM2IzMWQ5ODYyNmU5MGRjYmQ4ODRkNGQ1OGM4NTc1MzE0OTVhMWUxNzQxMzY3MTMxNCIsDQogICAgIjM4ZmMzYWMyNDg1ZDAxODFlMzhmZGJjNmUwZGNmMDMzZjU2MWNkMDI0NWE3ZGIzODg5OTAwODgxYmFhMmNmNzYiLA0KICAgICIzYmJhYmFlNDk1NDFiNjBmZDRhNGZiMWUzMDgyNmQ3MmJiNWIzNzY4MGE5OTYxZTIzYmY5MGIyMTRmOWQ2YmU1IiwNCiAgICAiM2NiOTI2YzhjNzlmY2JhOTI0ZjE0MjY5YTE4MWQ3YzcyNjg2MWVmZWM5NDM0NzlkMjA2NDJhODIxOTdlNzRhNyIsDQogICAgIjNkZGUyNmNmYjJiZDE1OWVlNTQ1ZmU0NmZkN2M2YzM5YjA4YzlhNDc0NTJmNjhkZDQyMDRlZmFjNDJlNDQ3YmMiLA0KICAgICIzZTMyMDcwNDE2ZjBmYmU5NmJjODU3ZmRmMDU4ZjdjMDU2ZGNkZjJhNGEzZWViNTdhOTk2YWQ5Y2FhYzUxNzdjIiwNCiAgICAiNDI2ZjViOTY4NzZkZjQxZWJhOWI5ZmY0YzE4NDVjOTk0NTkxNWMzM2E5MWQ0OWU5NDY2YTMwYzRhOWEzNWYwNyIsDQogICAgIjQ0NTBmODIxMWE0OTk2ZDFhZGFiZjFkM2Q3NTg0NTVjNGY4Y2UwOWM0YjMzNjEwNDYzZmM4YmMyYmI0YjRkZjciLA0KICAgICI0NGZhODBiOWQzZWYwNGQ2ZWVkZTg0OWQ3ZjBkZGQ4M2Y2Zjc0Y2NjYjQ3ZDQ0MTgzY2ZjNmNmZTc1MmE1MTllIiwNCiAgICAiNDYxZTQxMjgzYjRiMjFjZDIyODk1NmZhNDZjOGVjOWQ2OTA4MTM5MDNhOWI2MTljMDFmODdhMGYwMzkzY2JhYSIsDQogICAgIjQ2ZDhjMmI3NzU4MDU1YmZlODUwMzU0Y2VhODJiOTExNDZlMmZmYjZhMTM1MWUzY2I2N2E0NmQ0NDI4MGM2NDEiLA0KICAgICI0OGI2ODFjZTFmMzkwN2VhOTI5NzBiNjdkNzJmNDkwODNjNzFlNzY3YTA2ZGE0MzJkODBmMjhlMzJiY2I0MTU0IiwNCiAgICAiNDliNDcxNzgwM2QxYzI2MjI3ZGNlMGYyY2Q3NjA1MTIxOTU2OTMzOTVkMWI1Y2RmOTEzMjc1MjkyOWIzN2Y2NSIsDQogICAgIjRjYzk2MTlkMDk0M2ZhZmQ0NmNlZTI4NTA5ZWM3NWRmYzAwNmJjNmQ1ZGVmZjQyOGFhMzMyYzFkZTMyN2Y4MzIiLA0KICAgICI0ZTNiMDYzMjllZmYyOWFjMDRiYmYyMjZjMzI5ZmIxMDM4OWQ0ODZjYjk3NjRhMDBkNGI2NGExOWE0OTA4OGVlIiwNCiAgICAiNTA3OTQxYTExMDVkOWMwNDU5OTU0NDRjOGMzNjIzNjE5ZmE4ZTM2MDdkMDA5ZTdkYjNjYmRhNWI4MGUxNzBkZCIsDQogICAgIjUwYjM5MzcwMTUzY2E5MTRjZjk3MWJjYWUzNDk5OWYzM2Y1OGVhZTI3YmViODg1ODNmNThiYWEzZTY4ZGQxZjYiLA0KICAgICI1M2E2MWE3YWMzYjA5OTIzYTYwZTdiNDQzZmI2Mzg3MzY4OTZhNjM1ZmQ1YTNhNjhhN2Q3MmY5NTJlZWY3NTEzIiwNCiAgICAiNTUwZTIxMWQ3MjUyMTQ0NjgxMDBiNTljZjcxODEwNTAyMzQ5MzM3NWQ2ZjUzZTVlNWJhNzFjN2U3MzA3YTYzZSIsDQogICAgIjU2YjUxYmU5ZDdmYjcyNWVmMTc1YjcyYTA2MmNhNjZjYjIwZDI2NTBiYTYyN2NjY2E0OTI4YmE1M2Q3OTU5MjYiLA0KICAgICI1YTBhYjIzMDJlMGJhM2QwMTJlMjM4OWM0NDVkMzE1N2ZmM2M4MzUzZWIzMWQ3YTAxNjRhZjc5Y2U3NTA5ZTU5IiwNCiAgICAiNWQxZGJiNWE2Y2MzY2RkYzcyZTdjMzM4NDM4MGIwNDA5ZDkzYThhMTQ3YTBiZmIxODBhYjZkYWUyZjI5NjFkNyIsDQogICAgIjVlZTVhNTNmZWRmY2UwNzZlNTljOTZhYzkxYzhlMzAyODljMTgzY2IxZjdjM2IwMTVmM2UyMTdkNjc1NWVlODQiLA0KICAgICI1Zjk4N2I0N2RiZjdiMThhZWNjNmU4N2VhYzFjYzlmZDVkYjNhMTVhMDFhNWIzNDllZjFiNmViNzQxZmQwZjkxIiwNCiAgICAiNjM0MmJiMTkzNWMyYjViYTgxMDAxNzAyNWNjY2RkNTdhMjUyYmI3MGI4NTQ3YjUwNWI0MzA3NGU0ZDM3NGE5MSIsDQogICAgIjY2ZmIwNTEwY2ExYzJlZTYwMGQ2MzUxZDA5MWY3YWQ3YzY1NGU5NjQyY2JjOTVmYjM1NWU2MDcxYzhkMjNiOWYiLA0KICAgICI2ODBiN2ZmMDQzZmU1M2VkNmJlN2NhYzFjYmMzZTYyZmFiOTRkNGMzOWIzZmM4YmNiOTY2ZDM5MDM0MTVhYzczIiwNCiAgICAiNjhmNmJlMzYxMjQ5NGU1YjViZjRiOTdkMDRlMjZmZWY0Mjk5MWVkMzI1ZmY4Y2ViYjYyNDIwMzA4NWJjYWM1ZCIsDQogICAgIjY5MmUyNmFmYTM5Mjc0YjBiYTE1YWViNzA3MTQ4NDI1NTBkOWJlYWJmNjI0M2I0MzZjNTE0OTQwMGVhZjM1OTYiLA0KICAgICI2YjZiNjhmYWNlNTdjNTFmNjllMTQxMDUwYjk3ZTE4NmFhNTFkMDhmNjM2NmEzOWI1YWRiOTgwNTE4MmMyY2M2IiwNCiAgICAiNmJhYjUxNTFiMzUwNzI1ZTRiNjdjZDczNTZjODQ5MzdjYWFjZmQ2MTBhMGEzYTU1NGQzMGYwNTkwOGFjYzNiMSIsDQogICAgIjZjODUzYmVjOGFjNzhlYWQ4YTgzNjZiNGE1OGZlNWIyN2JhYzFjNjhkNWI3NzBmMDY0ZjdkNWI4ODkwN2M5NmYiLA0KICAgICI2ZjZkY2RmYjc2MDJlOThlYWFlMmRkNjAwY2I0YzI1NDNiMTRhZWEzMWQ2YWJlYzgyODg1YTcwOWU5MTkyZDc4IiwNCiAgICAiNzE0YTZkYmZlMWI4ODA2ZDE3MGJhODY0ZWJhMzBmMGUyOTJhNjE3YTllNTNhODEyY2ZhMTAyNGU1MzlmMzU4YSIsDQogICAgIjc0ZmY5MTZhOTc2YWU4ZTVmZDJiZTc3Y2NlYTg4NTIzYzZmNWYyMzM3NzMwMGM1NzI2ODk2NDMwMjQ0YzEzOWMiLA0KICAgICI3NTBjZTU5NTViOGQ0YmYxYTIyM2UxN2Y0ODBlNWE2NTUyZjVjMTdhNmViNmY5YTFkYWVmOTBiNGM1NzM1ZTJlIiwNCiAgICAiNzVjYWViYzg0ZDNlMjZiZDE4MjkxYWE5OWEyMWU0MDc1MTM1YTI2YjE1ZWVjZjM5N2YxMDQxYzM5YWY1MmNiZCIsDQogICAgIjc5NmRiYzg3NjFhZGFlNzg3ZTIyMzU5NDExMDExNmI0Y2VhYTE0ZTQ0MzRhYThhOTVkZjBlMWE0YzA2MzJlODgiLA0KICAgICI3Yjg1ZDFlOWJjNjc4NzBmMGNmOWNhOTdhNDk1ODA5N2UyMGUzYzA0NDljNDQxZTZkMTFlYmVhMTE0YjkzNmMxIiwNCiAgICAiN2M2YjFjODI1Yjk1YzRjMjBiYTI3NTg5MWFjOWI0MzY4YjA5YmQyNjg2YzhlZWFiYjY4YjljYzJkYmZlY2QzNSIsDQogICAgIjdjYmJlYzE5MjFiMjgyZDQzYTA4Mjg3MGQ4NGRiMjBhNTg5NzZmNGY0NGFlNjY3YjAwNWI4OGMzZGFiNjE4ODUiLA0KICAgICI3Y2M3ZmU4YjQ4MDBiNDAyYmQ3YTc4NDdiMWI3ZmE3ZDU3ZWEwMzU4OGEyMWFkMzkxODFlNzVhODAxMTMzN2NiIiwNCiAgICAiN2VhYzFmMjg0ZjFkZWNmYjA5M2E2NmJlY2Y1YTQzNGRhNTg3YWM3MjU3M2JmYjliODFlOTU5ZDdlNGMyNDhiMiIsDQogICAgIjdmOTY2NTE5NGViMjM1YjliN2UyNWFiNjAwZjdjMzY3Njg3OTQ2ZTAxMTk5MWNmMTVmMTliMzA1NDhjN2Q0NzUiLA0KICAgICI4MDQ0ZWYwODliZTI1YWM4NGFmYjU1MmU0YjQzNzRkYTFmN2M4OTQwZGZhYzhkM2EzMDlhN2YzMGI4YzlkNGY2IiwNCiAgICAiODE2OTdhZDQ3ZGRhODlkMzJhZmFhNjU1NGVmZDUxYWY0ODNmMDI1NDM2NTM5ODdmMmYzNGIxZjZjN2IzMmM2MSIsDQogICAgIjgxZWNlOWE0NjZhNTcyNTg0ZmE4NWQzZDMyMWVjZTFkOTFkOTRmYTAzOGJhMzY3ZTc1ZWNjZWQ0MGFhZDBhZGUiLA0KICAgICI4MzkwZDgzNmQ0NTNhZDZhNTY3NTVjNzEyYmJiZThkNjJlYTk2MTg0MzVjNWUxOWUyMTNkYTlmZmFiMjBiOGJmIiwNCiAgICAiODQzMTU5ZjQzOGI5MjU2Yjg3NTg3MGE3NDdlOTBkYmQ4NDYzYWFhZmFjZTEyZWY1ODY4MWRhMDQ4YzA1NGJhMSIsDQogICAgIjg0YzQxMDEyNDEyNWFjZDliMmViMDllYjlhM2Q0ZDdhNGMxNjFjNWZiMTkwMTkxOWI3NWYyNDEzNjQ3N2VhZTAiLA0KICAgICI4NTZjMThkNjFmODZjOGY0MWU4NTc3M2IxMDc1MDljMDZmNTRmNTU4MWEwNTIzNWJlZjg4ZjQ5ODUwNGMxYzI3IiwNCiAgICAiODZjYzM3ZTk2M2UxMjRlNDcxNWRkMmY4MzZjNTU4MTZiZTZhMGY2NTZmMWMzNjA5ZWZmOTg0MmRmYjJhNzgwZCIsDQogICAgIjg5YzdmYzU3NTU1ZjUzMzcxY2NlNWQ5NzhlNWQ2NmQzNjk1MTAwOGI2ZTM2YzIyYmE1MzY1N2VlMDQ3OTBmOGIiLA0KICAgICI4YzA3M2M3NzcxM2FjOGMxNjI0MmYzYTE0MzU4NzJmNmE1MDkzMDU0MDU3YWM5MjE1MzgxNDVjZjY3NzY2MmIzIiwNCiAgICAiOGQ4YzVjZDg5YTVkYjI1YjJiNWU4NDA4NmRlZjYyMTMyMTMzNGZhM2JkYjJhZDBkMTNmMjcyMjZkN2Y5NDZlZCIsDQogICAgIjhlNjY3ZWQyNDMxZjQwNTQ0YjVhOTkxODdiOTU2ZGEzNTYzMWM3OGQ3NTcxMzdhMzVlZmRlMmQ3YmQ5MTdkOWYiLA0KICAgICI5MDY4ZGI4NzYzYWViNzA4MDU5YTBiOGI2OWMwYTkzYWYyOTIyMWJhMGVkMTYyYmJkODhhNzBlY2UxMTU5N2NlIiwNCiAgICAiOTNjZjBmNzkwNGM3NWE3ZDVhMWIwZmMyOGY2NDc5OTUwZTdlZGNkNzNiODVkYTExNWE3Zjg1NzU0ZTlkOGMzNCIsDQogICAgIjk1NjczYWViZjcwMjFlNDYzNDI5ZjQzNjcxM2U2MDliYWIxOThhZjYyZjg3NmZhMmQwN2NhMmM2M2FlNmJlMDkiLA0KICAgICI5OGU3NzkwZjY0MDlhMjBmZTUzY2JjM2U3NmMzMjllNzc2NDQ4YzQ2YmY2YjNiNzg5ZTlkZGQ3MjY0YjBjYzViIiwNCiAgICAiOThmNDUzMjdmYjI0YWY3Yjg0M2RiN2M3ZDg3MzA3YmExYmVlNmZiMWExZjJkOTUzZDFiMmMzNGQ5MmJiZjk3NyIsDQogICAgIjk5NGI4OTRmZjNmMzE5ZDMxZTNhY2Q5YzhkNTE1ZWFkZDU3ZjJiMDZjYTU3MzMwOWY3YjU1NDVmMTUyZDYyYTQiLA0KICAgICI5YThmOWViZjU0NWE1ZmQyNzQ4ZmI0NzIxZGE4OWFmOTUxYjAwMjM3MTE5YmU1ZmVlMGM0OTRhODc5OWYxYTFiIiwNCiAgICAiOWQ5NjAyYTIyZGQ2ZWE4ZDNjNjQ0ZjJkMzlmODU4ODk4YjkxZjc3ZDEzNzQ4OTJmZGNjODQ2YzM2OTNhZjQxMiIsDQogICAgImExMDI3MjhlMjJkZjJlY2E5MTViMzJlMjUxYjAxODlkMGM1ZmYxYmYzMmNjZjNkMGQwYWMwMDViZTg1N2E2YjAiLA0KICAgICJhMjEyODA1NDg5ZTlhMzUwZmIwZWZmZTM2YTU2ZDhhNzkzZDRjZjE3MDE1MmY4OGY2ZWI5OWUxYzZmMWQ0ODExIiwNCiAgICAiYTMwNzhkM2NjYWVhMTJjYjAyYjllNzcxNGFjYTA1YTdkZWE1ZWUyZTAzNWIxMTlkMjM4YTJkMjAzNjc5MTg3MiIsDQogICAgImE1OGYyNjE1ZTIxODVkMWNhMmE3NzkzZjU5Y2JiM2MwYTVmZTBmNTU2ZGI0MzQ0ODZjM2RhNDgyZDdiNDE4N2YiLA0KICAgICJhNWQ4YzFiOTc0M2YzY2RhY2JiZDQ4YmE0M2U3NjIwMGViZjIyNjhiZjZmZWM3ZTc0ZTkxMTVjN2YwNzYxOWNhIiwNCiAgICAiYTY2MDJhMzU5ZTQ3NWY3ZTA5MTExYmI1MmQ2YzgzMTZiMDFmNTM1OTlkNTgzZjFmZDNiZDI5YWI0NTQ4YTdkNyIsDQogICAgImE3MTNmYjNhY2E2Mzc3ZGJjYWI4ZGZjM2ZkYzFhZGNiMGJlMTBhN2RkMDhiZWQ2MjkyYTExMmQ4ZjNmNjY4ZTAiLA0KICAgICJhODFlZmNkOGJhMjczNzdiOTI0NjJjMDVjNTNjMmE3NzRhMDQxY2FmMDZjZWRmZTMwMTk5OTIxZWU5OTU2NTJlIiwNCiAgICAiYTg5NWRhMTc1MWM2ZTRlMTc3NTllNjJjMDJkMjUwOTFkY2YyZWRlNWIyOGFhNmIwZjM4ZTZlNzcwMzIzN2FiYyIsDQogICAgImFhNDA0ZDM1ZmIyOWE2YTY5MTgwMWUwYzEyOGEyYjdmOWY5YzFjNTMxOTM5ODI0N2JhNjBmMmMzMGE4NTVkYzgiLA0KICAgICJhYTY3Mzc1ZjhiYjI5Njg3N2VhMGM0ZmM2ZmI1ZGY0MzM0OTY1MmZkZGQ0NjE2ZDBiYjdlN2VlOGNhMGMwMjU2IiwNCiAgICAiYWM5NDQwNDU1M2RmMzg5ZWFhMDlkMTEzMGRhODIzNjhmNjEwYjkxMDk2ZGJmOTk2OGViMmE0ZTc0NjQ3NWIwMCIsDQogICAgImFjY2Y0MGQxZTFmODVjMWRiNmQ0ZWQ4MTZkNmVhYjhmZDIwNTQyNjFlZTJkYjI4OTY0MzZjNDQ1ZTM3Y2Q1MjciLA0KICAgICJhZTI1NTdhZDc0MjA3ZTM3ZDJkNzI2MmZiNTczYWU2NzE3ZDM1NDE2ZGJiNzA2NWU1NmFiYjI5YzY0Yjc3YTY0IiwNCiAgICAiYWY4OTIyZWJhN2IzMzcyNGNkZGJjZDc1MmJlMDg0YTU0Mjk3ZWY1MDRmZjMzMWM1M2M2MDY2MGRjOWY5YjM4ZSIsDQogICAgImFmZDA1OGZhYWQ1ZDRhZWY4Njc0OTJiYWQ1MDUwYzQ5MDYyMmYzYjUyZmI2ZjRkN2Q1YzU4NmM2ZjEzM2EyMjYiLA0KICAgICJiNGFkNDIxZDM2ZTE0ZGQ2MGQzNDJiNTM1MzhlOTdmMGY0NzdhZjhiMGUyMDhlZDJhNGQ4MjMwMTc4ZDEyYTMxIiwNCiAgICAiYjUzZjZjNDM2MTc3NGVmODI2NWZkZjU3NTQ1NzZmYzdlNTEwOWFlNzA0M2IxMTdiMjg5YmJkYWIyM2QxNTNhOSIsDQogICAgImI2ZDlhOThlNGZlZWEyOGFkY2M3NzhiNjUxOTFiYzEzMTRkYzYyMjUzYzI3OTgwOTkyYThjY2RjOWYwMmRjNzYiLA0KICAgICJiN2QxNzliZDRhMDU0YjBmMDA5YTY2MjBkZGRkODE4MTNhNjYwNzY2NzdlM2JiNGUxNzU5NDA1N2QwNDRhMTM3IiwNCiAgICAiYjhiMTBhZGQ4NTAxOTFhZmJhOWJhZWEyMWEzNWQ0ZmI2ZGMzODE4NjgyNGMxZDk3MDdiNDlhNjc3NWU2NDM1YiIsDQogICAgImJhN2JjNzdiNTU1OTY2MzU5YWYwYjhjNTAxOWI4MzMwMGNmZTQ5MjM2NDM0NDRlOGY2NzEzZTViNmNhNGFiZTUiLA0KICAgICJiYmViZDVlOTRjNDQ0MTcxNmZhZmY1YjdmYjc2MGU5MjczNDE1YjQwNjdmMzlmYmY0YjU5NGZiYjg3YzY1MmZjIiwNCiAgICAiYmVhOTNiN2U1NDIzNTkwZmY1Zjc3N2QzM2JmYTYxZmU4ODU1MzhiYTY1NDg3ODBhMzcxYTkxOWMwNjNhNTlhMSIsDQogICAgImJmMTIwYzljMDAzYzY5ODAyNzcyYmYwYWM3NjVlODkxYjA5YmY2ZTgxYTEwMTQzNmM3MWMwMGRjYjRiNjFhMDQiLA0KICAgICJjMTc1NDc0NzNkNDg4MWRjN2ZjMDdmYTBmMjdkMWVlN2MwMzY4YmE1ZWFiOTAyNDVmODVjNGI0ZGY2MTM4NGY3IiwNCiAgICAiYzI0NjhjNjY0MDkxZmVkMGRkZGRlNDNkZmI0MWRiYTdiMzZjNjQzZDRkNWQwOGIzYTk1MTE4YjdhYzlmYTIxZiIsDQogICAgImM1N2I2MjZmZDYwMTVkN2U1MGEyZDM5MGQwOTIxMzM5MzUxNDFmOTNmNDNjOGRlYmZlMmIyYTJhMTY0MWFiYjQiLA0KICAgICJjN2YyMTBhODA1MDkyMGFmYzJiYjhmMjcyMzkwNjhhNjY0NzY5ODA4ZjY5NjE4MTk1NjFlMWYwNTFmZmY4NzVkIiwNCiAgICAiYzg0ODk0NzFkNzdjM2M2ZGE5MjhkZWRkNWNlNzFjMDhlNTIyOTRkMDBkOTc5MmQ4NjM4OGMyMWJjNmM1YWYyMSIsDQogICAgImM4NWY0NTAxNTg5OTUzNDU1Y2I4ZTg1MmNjMTZhZjM0ZDhlNTdkNjRlMzJhYWU4Y2Y1NjFmNTc1MDYxYzkxZDUiLA0KICAgICJjODhiZTY4YWRkNzkyNzgwYzRkOTliZDA4MzE3YzAwZDBmMWE1MTkzMmI5ZjZmMmQ3OGRjYzU3MDBjZmY1NzFjIiwNCiAgICAiYzk1OTNmZWE5N2I0ZjQzOWM1MTBmODg4NDg2ZDg1OWE3Yjg4N2RlYTRiYjhmYjIyNjA4N2M1N2FiYTJmMDdhNiIsDQogICAgImM5OTk3NzEzZjE0Njg0NzA3MjkzYmZlZWMwODhiOTU0ZjJkOTgzOTE0MmRmYzZmOTY0ZDA3MTBiNTRmNTRkYzUiLA0KICAgICJjOWI2YzU4OWNjNzBjNDljYzViYWIyZTJmNTMwZWZjMDY0YTExZjQ4ZjE1M2RmYWViMTY4ZGIzMTdjYmUxYzNiIiwNCiAgICAiY2E4YWJjYjNlNzE1YTlhZjZjODhkZTJmNmRlMWQ5YzZkYzViNTgyMGJlZjJiMGNiMDUyMmYwOTlhNGExOTY3OSIsDQogICAgImNiOGEyYTg3NDVmODY5MTNmYjgyY2Q4ODMyM2I4NTJkYzBiZTUwYzc3ZTU5MTUxNjBhMzkyMmU3ZTQ1YzMwMTUiLA0KICAgICJjY2Y1M2FkNmE0ZWJkOTUxNTMyMjQ0MTMzZTlhZTgzM2I4NjQ3NGExYzA0YWVkMWE5NjQwYWRjZTQ3MzU0NDJiIiwNCiAgICAiY2Q4MzZjYTc5ZjNmNWVjY2E4MzY1NWVjMGVlMTQxYjAyY2U2ZDEyMmJmNjZjNjIxNTJmY2YxYTJlNWE3ODY2MSIsDQogICAgImNkZGMyZTliZDc3ZWVhZmY1NGYwMDRkYTcwMjVhODBmZTY4NWI0NzZkNWRiY2ZkNTM0NjA1NjQ1NjBhMDEyNDUiLA0KICAgICJjZTU2YjA2Y2RkY2FmNmZlMzU2YmRiY2MxNThlZWRmYzdlMWEzN2U4ZTM0OGNiZTRlOTdkYjhmZDI3ZWU0M2M4IiwNCiAgICAiY2VkMDYxNDc0ZDMxYmJhOGViNTJiYzMxYzlhYWRkNzhjMDFkZjEyYmU5ZDIzNzA5NTBiY2NmOTYzNDVlZDU2ZiIsDQogICAgImNlZDdlOGQ0OTZkNWFlYzk0YjAzNDI4YjRmMjZkOWM5ZDE2OWJiYTg5OTlhMmQxZGRiMjY5NWNjZDg5MGZiYjMiLA0KICAgICJkMTdjOTE3MmQzMjM3ODZhMWU3YWZhMjY5M2I0YjhjMDI0OTU5MmYwMDE3ZDg4ZjcyYWQ3MDlkZjk5Nzc1YjFmIiwNCiAgICAiZDM3ZDQ1ZTc2MjBkOTY5MzdkMDA2ZjQ1NmYxZDU5NWZlZTQzN2VlMzAzNmQ2ODk1M2E2OWRkZjEzYmQ1NWMyMSIsDQogICAgImQzYzk5YzU1MWRiODNmN2E4NjdkOWI5ZTE4YWNhZGJkNmQzMzE4Y2I0YTY1MjU5ZWJmNmQyMmUxODY3ZmIzZTkiLA0KICAgICJkNWU2NmIwYzVkNzY0MmIyYjg2Zjg3YzRiZGQyYWU5ZjAwNDBlYzliOTg1YmJmOGNhM2U3YzE5NTgxY2QzYWYyIiwNCiAgICAiZDc5OGI3NTlmZTUzNWMwYjc3YWE1NDU0NTczYjg0YTQ4MzNkN2NhMjlmMTgzOWY5YWI5OWE0MWYzZjRkNzNiYiIsDQogICAgImQ3YjgzMDIyZjhlYjc1N2FhNmI1Y2JmMDc2MjY4YTE5NTMzZjAxMWRhNDE4YjNmNjcwM2Q5NzAxNTQzOTE1NjgiLA0KICAgICJkN2ZlMzYzNGNkZDFiYzIyZjY0NTQ4NmY2NjY5M2YyY2Q1ZjVhMDJhMzc4M2UzYzIwZmUwNDY0NjU0MjkyODRmIiwNCiAgICAiZDg4MzliZTM1NWE3YmU5Mzg5ZTU4OTFhOTdhMmYyNjllOTYxNjZjNTM5ZmYyMTAzNWE2NDAxMzdiMDNhZTFlMyIsDQogICAgImQ4Y2I0Y2Y0N2I5OWQ2NmI4YTBiYmQ5MTUzM2ZmZGUzYzI4ZmVlMTgyNmUxMDk2NTExNDMzMjVhNjM2YTE0YjkiLA0KICAgICJkOGVmMWVmYzU3YWNkMmI0YmE5MGZiODVjZjY0NTNhMDczMTQ2OWU3ZjFiMmNiMWUyY2JiNzg2NDk0YTE1ZmI2IiwNCiAgICAiZGI0OTkwYzliODViN2EzYmIzMDIxOTZmZmY2NGY1ZmQxY2JkY2IyYmVhYzk3MDAxZjRkNGYzYzBkNjc4YjA4MiIsDQogICAgImRiYTQ1YTlkZTYxODhhMWVlZDU0ZDBhNGY5Y2RiYjYyNzljMzg0ZWY4YTM2YmYyNzE3OTJhYjBiN2YyYTgxYjAiLA0KICAgICJkZTU0ZGUwZjhlYzAzZDY5MDBmZDQ0YTZjMmY3YWMzNWFjZjdmNjNjMjEwMThkYmNjZmQwY2MwZDZlZTY4ZDM5IiwNCiAgICAiZGU4ZGFjYTEwMGVjOGRmMWU4OGYxOWY5NDIwZTliYjI1MmViYzEyZDNkZmVkZTJhYWJkMTZhZDkyYTI2MzI0ZSIsDQogICAgImRmMzIzNmMwMTg2NGFhNTQ4YmM3MWQ3ZWUyMDM1OTQ2N2U3M2NiN2VlMTFkNjI0ODFjMDQwMTM4NGM4OTg4ZDciLA0KICAgICJlMDVlZTFkMmVmMWM2NjRmNDBlMjE1MDkxMTQ5MjIzZTlkYWZlOTUwOTM5NGQ0MWQ5ZWZhZjAwNzhiMzAxZGM5IiwNCiAgICAiZTBkNWU5MjE4OTQ2YTU0YjU5NWI4OWYzMDZkOWU4NzdhNjNhMDI4N2IyNjQ2OTZjZGIyYzY5YWFmNDNlZjg4ZCIsDQogICAgImUxMWVlMTc2Y2I0ZjZjMTY2Nzc3ZTMzYThmMDQwNTdlMTI0ZGZhZDQ3MDUzNDlhY2Q0YjE2YWVjNzNjN2NkMmEiLA0KICAgICJlMjc1Mzk1MDFmYjQxMDRjYzhjNDcwMDZjMzJlZTZhMDYzOTU5ZTQzYTY2NTRkODAyYjIxYjMzMzAyNjg2YmEzIiwNCiAgICAiZTNjMjdkYWQzZGJhZDdjOTQ1NGI1NGRhOTc0OWI4NzY3ZGY2ZDc0MjAxMjUwODk0OTlmOTExNTM1YmUwNDJiYiIsDQogICAgImUzY2IwODIyZDg4NjNjNDYwZDgyMTg3OGE4ZDhkNDk4MGI2OTQ1OGM1NmY0MjM5YzY3NGIxMzE5ODljMDUyZGYiLA0KICAgICJlM2YxNjI3ZTc1ZWE1YWUxMWYyNWRmNmRkNzFkMTViMzJkZmFjMWY5MGZmOWQyZGRjMWIwNmYzNWNmMDZiM2QyIiwNCiAgICAiZTc2ZTg0OGEyMTQ4YTliMTA4MWQxYzY4ZjlkOTkzMWUwZjFiNzg1MzNiMjhjZTdiNGQ3NWI5YjM3ZDgyY2JlOCIsDQogICAgImU5YTRjYjI0MmQ2YmFiYzEwZmYwMmU3YTU1ODhiZjg4ODg0NGZlM2JmYWFmZWY5MDRkNWYyOWYzZmMyYzc0ZjYiLA0KICAgICJlYWI0Y2VkZTEzOTQ4NjU0Yzg1YjQyYjI0MmI5M2YxZTdlNjE1N2E5ZDYzMTdmMDM4MjZlNzk1MjM3NmUyMDk2IiwNCiAgICAiZWY0ZGE5ZGM0ZTExNGJlODVhZTBiNTA3YTAwZThhODNhOWVmMzE3YzRmODYwMjc1YzMzZjEyZWJiOTgzMzcxNiIsDQogICAgImYwMWMyZWRiMzQ0MDZjOTFjMTI1Mjg2NmIwYzJjOTM0MTNmOTA2MjBiM2E0OTBjOTUyYThkNDIzOGYwMWZlYjkiLA0KICAgICJmMTA4OTg2YWIwN2M4YzEyMmM5ZjRiZDk4NDBiY2FjZThmY2U3YzY3ZGFkZmEyYjEyNjgwMjRjODMwZWQ2NWEzIiwNCiAgICAiZjE0MDIxYzY1ZTc1ODgyZTU5YTMxMzZjMWUxM2NjYmJjY2JjNjQ1ZGU3MDY0MTgyYzJjYmJlZjMwODFlZjkzOSIsDQogICAgImYxYTJlNDRkNWY3NzVjYzAzOTQ2ZTA3NmNiOGEzM2FmNjk3NzhkZDk2ZWFhY2ZmODVhOWNiZmFhM2ZhMTY4NTUiLA0KICAgICJmMWZiZjg1NTM1Mzk2MTgwMGNjMTA0ZTg0Y2I2YTA0MGY5M2U3ZjM4OGFlNDcyMTRkY2E0ZWIyNzM0YjZjY2I1IiwNCiAgICAiZjRiZDY2NDNlYjBjMTNhNThlODgwZTViNDg2OTVjZWY5MjM5MWRmZTk2M2I3MDRmZGIzZTc1MTQwZmFkM2NhMyIsDQogICAgImY3NTAxOTI2OGUwNGQxNDUzYjIwN2M1MmJlODEwOGU2YTFkZGViODc3NWEyM2VmYzFjMTU1YWVlYTMzMGU4MmQiLA0KICAgICJmOGZjYTQ5NjRjODQ1NmZmZDYyYTZlN2U4NjE5ZjVkMjQzMTdiNTViNmI5ZDkzNDgwNWQ4MDBmMDk5ZGFkZDBlIiwNCiAgICAiZmE0ZGViOTQwODBjZWIwMGEwMTFjYWEzNTMwMDc5MTZiNmVkMGNmMTFmYjU0ZjM4N2IxN2E4MmFjZjdkOWFmMSIsDQogICAgImZmMDY0YWEwMGVlNTBlZTg1NGFiYzVkYjE4NWFmNmI2ZDdjY2MzOWU4YTYxMzdhYzUzY2RjYTc3YWY1NWViMWYiDQogIF0sDQogICJzb3VyY2VzIjogWw0KICAgICJwZWRhZ29neV9jYXNlc192My5qc29ubCIsDQogICAgImhvbGRvdXRfYXJpdGhtZXRpY19jYXNlc192Mi5qc29ubCINCiAgXSwNCiAgInBvbGljeSI6ICJTdWJzZXF1ZW50IGNhbmRpZGF0ZSBzaGFyZHMgZXhjbHVkZSB0aGlzIHVuaW9uOyBwcmlvciBpbW11dGFibGUgY2FuZGlkYXRlIHZpZXdzIHJlbWFpbiBhdWRpdCBhcmNoaXZlcy4gT25seSBmaW5hbCBiYWxhbmNlZCByZWxlYXNlIGlzIGFwcHJvdmVkIGFzIGV2YWx1YXRpb24tZGlzam9pbnQgdHJhaW5pbmcgZGF0YS4iDQp9DQo='))
spec_final=importlib.util.spec_from_file_location('mvk_finalizer',qa_dir/'finalize_current_candidate_metadata.py')
module_final=importlib.util.module_from_spec(spec_final)
spec_final.loader.exec_module(module_final)
final_metadata_thread,final_registered=module_final.register_final_metadata_check(globals())
print(json.dumps({'final_metadata_registered':final_registered,'waiting_thread_alive':final_metadata_thread.is_alive(),'audit_worker_still_alive':full_audit_worker.is_alive(),'training_started':False}))

{"final_metadata_registered": {"state": "WAITING_FOR_EXISTING_FULL_SCAN", "registered_utc": "2026-10-22T15:37:20.567375+00:00", "recipe_identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "recipe_sha256": "1bb69c394c7eaaea7034125c6b3361eebbbe0d77e0715a8454331fbe7cee3ff8", "training_started": false, "new_cases": 0, "new_chat_rows": 0}, "waiting_thread_alive": true, "audit_worker_still_alive": true, "training_started": false}


In [ ]:
import sqlite3,json
from datetime import datetime,timezone
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as p_db:
    pr=p_db.execute("SELECT coalesce(sum(next_line),0),sum(complete) FROM members WHERE role='candidate_chat'").fetchone()
p_db.close()
f_status=qa_dir/'final_metadata_status.json'
print(json.dumps({'observed_utc':datetime.now(timezone.utc).isoformat(),'committed_physical_chat_rows':pr[0],'chat_members_complete':pr[1],'expected_raw_chat_rows':6502392,'worker_alive':full_audit_worker.is_alive(),'worker_state':full_audit_worker_state.get('state'),'error':full_audit_worker_state.get('error'),'final_metadata_status':json.loads(f_status.read_bytes()) if f_status.exists() else None,'last_verified_checkpoint':{k:full_audit_worker_state.get('checkpoint',{}).get(k) for k in ['bytes','sha256','observed_utc']},'query_index_state':globals().get('report_index_evidence',{}).get('state'),'query_indexes_completed':len(globals().get('report_index_evidence',{}).get('completed_indexes',[])),'training_started':False}))

{"observed_utc": "2026-10-22T17:18:55.153077+00:00", "committed_physical_chat_rows": 4427328, "chat_members_complete": 5, "expected_raw_chat_rows": 6502392, "worker_alive": true, "worker_state": "STOPPED_REQUIRES_AUDIT_REPAIR", "error": "Owner requested save and pause until next session; this is not a data QA finding.", "final_metadata_status": {"new_cases": 0, "new_chat_rows": 0, "recipe_identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "recipe_sha256": "1bb69c394c7eaaea7034125c6b3361eebbbe0d77e0715a8454331fbe7cee3ff8", "registered_utc": "2026-10-22T15:37:20.567375+00:00", "state": "WAITING_FOR_EXISTING_FULL_SCAN", "training_started": false}, "last_verified_checkpoint": {"bytes": 6531076096, "sha256": "55e09126557abc3e538455f97cbea9c722a83cdbf0aeacd832dccbbe000f44f5", "observed_utc": "2026-10-22T16:53:09.966298+00:00"}, "query_index_state": "QUERY_INDEXES_COMMITTED", "query_indexes_completed": 3, "training_started": false}


In [ ]:
import json,hashlib,sqlite3
from datetime import datetime,timezone
metadata_names=['span_runtime_evidence.json','source_split_linkage_evidence.json','source_id_inventory_reconciliation.json','latest_compact_progress.json','final_metadata_status.json','final_raw_v3_report_preserved.json','final_candidate_chat_technical_adjudication.json','final_current_text_candidate_manifest.json','final_antijoin_execution.json']
meta_contents={}
for metadata_name in metadata_names:
    metadata_path=qa_dir/metadata_name
    if metadata_path.exists():
        if metadata_path.stat().st_size>300000:raise ValueError('Metadata response bound exceeded')
        meta_contents[metadata_name]=json.loads(metadata_path.read_bytes())
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as obs_db:
    obs_rows=obs_db.execute("SELECT coalesce(sum(next_line),0),sum(complete) FROM members WHERE role='candidate_chat'").fetchone()
    obs_issue=dict(obs_db.execute('SELECT kind,count(*) FROM issues GROUP BY kind'))
obs_db.close()
control_observation={'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':full_audit_worker.is_alive(),'worker_state':full_audit_worker_state.get('state'),'error':full_audit_worker_state.get('error'),'source_versions_indexed':1629906,'actual_committed_physical_chat_rows':obs_rows[0],'chat_members_complete':obs_rows[1],'members_expected':251,'checkpoint':full_audit_worker_state.get('checkpoint'),'current_issues':obs_issue,'recipe_sha256':pointer_all['sha256'],'recipe_identity':V3_ID,'onepass_optimizer_applied':False,'training_started':False,'metadata':meta_contents}
print(json.dumps(control_observation,ensure_ascii=False))

{"observed_utc": "2026-10-22T15:41:58.880425+00:00", "worker_alive": true, "worker_state": "RUNNING_DATA_ONLY", "error": null, "source_versions_indexed": 1629906, "actual_committed_physical_chat_rows": 2696304, "chat_members_complete": 3, "members_expected": 251, "checkpoint": {"id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "bytes": 4574961664, "sha256": "aab811924a15f8f6b900fc40fbb1493dd3de333ad12116b878d473a2a0a863ad", "fresh_raw_readback_verified": true, "report_at_snapshot": {"schema": "MathVisionKid.full_current_archive_audit.v3", "identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863", "derivation_provenance": {"parent_file": "math_data_full_audit_v2.py", "parent_sha256": "81b1e3f04c5886f78282386ecb34d47ba4ba9da63edfec8856bab82034a093ee", "source_label_evidence_file": "exhaustive_primary_math.py", "source_label_evidence_sha256": "d0af1e4adf73a55b8416757ee43de900474f378af7ff8290fffdd0b015acb8b2", "source_label_evidence": "make_row returns train_candidate; chat_r

In [ ]:
import sqlite3,json
sqls={'candidate_link_count':"SELECT count(*) FROM samples s JOIN source_versions v ON s.case_id=v.id AND s.source_raw_sha=v.raw_sha WHERE s.linkage=1 AND v.source_split='train_candidate' AND s.split='train'",'source_label_counts':"SELECT coalesce(source_split,'<missing>'),count(*) FROM source_versions GROUP BY source_split",'source_hash_origin_counts':"SELECT raw_hash_origin,count(*) FROM samples GROUP BY raw_hash_origin",'mode_counts':"SELECT count(*) FROM (SELECT case_id FROM samples GROUP BY case_id HAVING count(*)!=4)"}
with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True,timeout=10) as idx_db:
    plans={name:idx_db.execute('EXPLAIN QUERY PLAN '+sql).fetchall() for name,sql in sqls.items()}
    indices={name:idx_db.execute('PRAGMA index_list('+name+')').fetchall() for name in ['samples','source_versions']}
idx_db.close()
index_observation={'utc':datetime.now(timezone.utc).isoformat(),'query_plans':plans,'indexes':indices,'source_rows_changed':False,'training_started':False}
(qa_dir/'sqlite_report_query_plan_before.json').write_text(json.dumps(index_observation,indent=2))
print(json.dumps(index_observation))

{"utc": "2026-10-22T16:11:35.071034+00:00", "query_plans": {"candidate_link_count": [[5, 0, 0, "SCAN s"], [11, 0, 0, "SEARCH v USING INDEX sqlite_autoindex_source_versions_1 (id=? AND raw_sha=?)"]], "source_label_counts": [[6, 0, 0, "SCAN source_versions"], [8, 0, 0, "USE TEMP B-TREE FOR GROUP BY"]], "source_hash_origin_counts": [[6, 0, 0, "SCAN samples"], [8, 0, 0, "USE TEMP B-TREE FOR GROUP BY"]], "mode_counts": [[2, 0, 0, "CO-ROUTINE (subquery-1)"], [8, 2, 0, "SCAN samples USING COVERING INDEX sqlite_autoindex_samples_2"], [38, 0, 0, "SCAN (subquery-1)"]]}, "indexes": {"samples": [[0, "sqlite_autoindex_samples_2", 1, "u", 0], [1, "sqlite_autoindex_samples_1", 1, "pk", 0]], "source_versions": [[0, "sqlite_autoindex_source_versions_1", 1, "pk", 0]]}, "source_rows_changed": false, "training_started": false}


In [ ]:
"""Add only non-unique query indexes; retain the existing scanner and worker."""
import hashlib, json, shutil, sqlite3, threading, time
from datetime import datetime, timezone

assert full_audit_worker.is_alive()
assert not globals().get('_report_index_upgrade_registered', False)
assert V3_ID == 'f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524e25863'
assert hashlib.sha256(Path(audit_v3.__file__).read_bytes()).hexdigest() == '02431da433060a6574127f9a8e14d90cabffe85d8d0150c3b2a331dc15b4be1f'
assert shutil.disk_usage(ROOT).free > 6*1024**3

REPORT_INDEX_SQL = [
    'CREATE INDEX mvk_report_source_label_id_hash ON source_versions(source_split,id,raw_sha)',
    'CREATE INDEX mvk_report_sample_link_split_case_hash ON samples(linkage,split,case_id,source_raw_sha)',
    'CREATE INDEX mvk_report_sample_hash_origin ON samples(raw_hash_origin)',
]
REPORT_INDEX_QUERIES = {
    'candidate_link_count': "SELECT count(*) FROM samples s JOIN source_versions v ON s.case_id=v.id AND s.source_raw_sha=v.raw_sha WHERE s.linkage=1 AND v.source_split='train_candidate' AND s.split='train'",
    'source_label_counts': "SELECT coalesce(source_split,'<missing>'),count(*) FROM source_versions GROUP BY source_split",
    'source_hash_origin_counts': 'SELECT raw_hash_origin,count(*) FROM samples GROUP BY raw_hash_origin',
}
report_index_ready = threading.Event()
report_index_old_consume = batch_adapter.consume
report_timing_evidence = []
report_index_evidence = {'registered_utc':datetime.now(timezone.utc).isoformat(),
    'state':'WAITING_FOR_WORKER_WRITE_TIMEOUT_CONFIGURATION', 'statements':REPORT_INDEX_SQL,
    'scanner_adapter_files_changed':False, 'new_scanner_started':False,
    'source_or_chat_rows_updated':False, 'non_unique_performance_indexes_only':True,
    'training_started':False, 'completed_indexes':[]}

def save_index_evidence():
    path = qa_dir/'sqlite_report_index_evidence.json'
    temporary = path.with_suffix('.writing')
    temporary.write_text(json.dumps(report_index_evidence,ensure_ascii=False,indent=2))
    temporary.replace(path)

def report_index_safe_consume(audit, entry, path, max_rows=20000, commit_every=256):
    # The original SQLite connection is touched only in its owning worker.
    # A bounded index transaction can then wait between the same 256-row commits.
    audit.db.execute('PRAGMA busy_timeout=1200000')
    if not getattr(audit, '_report_timing_logged', False):
        original_report = audit.report
        def timed_original_report():
            started = time.monotonic()
            state_before = report_index_evidence['state']
            try:
                return original_report()
            finally:
                report_timing_evidence.append({'seconds':time.monotonic()-started,
                    'utc':datetime.now(timezone.utc).isoformat(),
                    'index_state_before':state_before,'index_state_after':report_index_evidence['state']})
                # Optional timing evidence must not turn a valid audit into a
                # failure if its separate metadata file cannot be written.
                try:
                    path=qa_dir/'sqlite_report_timings.json'
                    temporary=path.with_suffix('.writing')
                    temporary.write_text(json.dumps(report_timing_evidence,indent=2))
                    temporary.replace(path)
                except Exception:
                    pass
        audit.report = timed_original_report
        audit._report_timing_logged = True
    report_index_ready.set()
    return report_index_old_consume(audit,entry,path,max_rows=max_rows,commit_every=commit_every)

def build_report_indexes_once():
    connection = None
    try:
        if not report_index_ready.wait(timeout=1800):
            raise TimeoutError('Worker did not reach the next consume; no index write started')
        connection = sqlite3.connect(V3_DB,timeout=1200)
        connection.execute('PRAGMA busy_timeout=1200000')
        if connection.execute('PRAGMA journal_mode').fetchone()[0] != 'wal':
            raise ValueError('WAL required for the existing reader/writer concurrency')
        if connection.execute('SELECT identity FROM config').fetchone() != (V3_ID,):
            raise ValueError('Working database identity changed')
        report_index_evidence.update(state='BUILDING_NON_UNIQUE_QUERY_INDEXES',
            started_utc=datetime.now(timezone.utc).isoformat())
        save_index_evidence()
        for statement in REPORT_INDEX_SQL:
            name = statement.split()[2]
            prior = connection.execute('SELECT sql FROM sqlite_master WHERE type=? AND name=?',('index',name)).fetchone()
            if prior is not None:
                if prior[0] != statement:
                    raise ValueError('Existing index definition differs: '+name)
                continue
            started=time.monotonic()
            connection.execute(statement)
            connection.commit()
            report_index_evidence['completed_indexes'].append({'name':name,
                'seconds':time.monotonic()-started,'completed_utc':datetime.now(timezone.utc).isoformat()})
            save_index_evidence()
        plans = {name:connection.execute('EXPLAIN QUERY PLAN '+sql).fetchall()
                 for name,sql in REPORT_INDEX_QUERIES.items()}
        report_index_evidence.update(state='QUERY_INDEXES_COMMITTED',query_plans_after=plans,
            completed_utc=datetime.now(timezone.utc).isoformat(),worker_alive_at_observation=full_audit_worker.is_alive())
    except Exception as error:
        if connection is not None: connection.rollback()
        report_index_evidence.update(state='INDEX_UPGRADE_STOPPED_WITH_EVIDENCE',error_type=type(error).__name__,error=str(error))
    finally:
        if connection is not None: connection.close()
        save_index_evidence()

batch_adapter.consume = report_index_safe_consume
_report_index_upgrade_registered = True
save_index_evidence()
report_index_thread = threading.Thread(target=build_report_indexes_once,name='MathVisionKidFiniteQueryIndexBuild',daemon=True)
report_index_thread.start()
print(json.dumps({'finite_query_index_upgrade_registered':True,
    'worker_still_alive':full_audit_worker.is_alive(),'training_started':False}))

{"finite_query_index_upgrade_registered": true, "worker_still_alive": true, "training_started": false}


In [ ]:
import json
p_index=qa_dir/'sqlite_report_index_evidence.json';p_times=qa_dir/'sqlite_report_timings.json'
index_live={'index_evidence':json.loads(p_index.read_bytes()) if p_index.exists() else None,'report_timings':json.loads(p_times.read_bytes()) if p_times.exists() else [],'span_calls':span_runtime_evidence.get('calls',[])[-3:],'last_verified_checkpoint':{k:full_audit_worker_state.get('checkpoint',{}).get(k) for k in ['bytes','sha256','observed_utc']},'worker_alive':full_audit_worker.is_alive(),'training_started':False}
print(json.dumps(index_live))

{"index_evidence": {"registered_utc": "2026-10-22T16:16:40.223559+00:00", "state": "QUERY_INDEXES_COMMITTED", "statements": ["CREATE INDEX mvk_report_source_label_id_hash ON source_versions(source_split,id,raw_sha)", "CREATE INDEX mvk_report_sample_link_split_case_hash ON samples(linkage,split,case_id,source_raw_sha)", "CREATE INDEX mvk_report_sample_hash_origin ON samples(raw_hash_origin)"], "scanner_adapter_files_changed": false, "new_scanner_started": false, "source_or_chat_rows_updated": false, "non_unique_performance_indexes_only": true, "training_started": false, "completed_indexes": [{"name": "mvk_report_source_label_id_hash", "seconds": 7.115316506999079, "completed_utc": "2026-10-22T16:21:13.038977+00:00"}, {"name": "mvk_report_sample_link_split_case_hash", "seconds": 19.62488905399732, "completed_utc": "2026-10-22T16:21:32.665010+00:00"}, {"name": "mvk_report_sample_hash_origin", "seconds": 7.425977131999389, "completed_utc": "2026-10-22T16:21:40.109908+00:00"}], "started_utc

In [ ]:
"""Owner requested save-and-pause; never interrupts a SQLite commit or upload."""
import hashlib,json,sqlite3,threading
from datetime import datetime,timezone

assert full_audit_worker.is_alive()
assert not globals().get('_owner_pause_requested',False)
_owner_pause_requested=True
owner_pause_requested_utc=datetime.now(timezone.utc).isoformat()
owner_pause_original_tokenize=batch_module._ORIGINAL_TOKENIZE

class OwnerRequestedDataOnlyPause(Exception):
    pass

def owner_pause_tokenize_gate(row,tokenizer,max_sequence_tokens=40960):
    if threading.current_thread().ident==full_audit_worker.ident and _owner_pause_requested:
        raise OwnerRequestedDataOnlyPause('Owner requested save and pause until next session; this is not a data QA finding.')
    return owner_pause_original_tokenize(row,tokenizer,max_sequence_tokens)

owner_pause_status={'state':'OWNER_PAUSE_REQUESTED_WAITING_FOR_VERIFIED_SAVE',
    'requested_utc':owner_pause_requested_utc,'training_started':False,
    'mechanism':'Stop before a later row token gate; original consume rolls back only the current uncommitted256-row transaction. Original worker saves and raw-verifies its complete portable checkpoint.',
    'data_qa_finding_created':False}

def write_owner_pause_status():
    (qa_dir/'owner_pause_receipt.json').write_text(json.dumps(owner_pause_status,ensure_ascii=False,indent=2))

def confirm_owner_pause_after_save():
    try:
        full_audit_worker.join()
        # The registered technical finalizer must not run acceptance on partial
        # rows. Keep its controlled-stop evidence before recording owner pause.
        final_metadata_thread.join(timeout=60)
        if final_metadata_thread.is_alive():
            raise RuntimeError('The finite metadata checker has not exited')
        if batch_module._ACTIVE or batch_module._OWNER is not None:
            raise RuntimeError('Audit adapter did not uninstall cleanly')
        batch_module._ORIGINAL_TOKENIZE=owner_pause_original_tokenize
        if batch_module.L.tokenize_assistant_only is owner_pause_tokenize_gate:
            batch_module.L.tokenize_assistant_only=owner_pause_original_tokenize
        if batch_module.L.tokenize_assistant_only is not owner_pause_original_tokenize:
            raise RuntimeError('Original tokenizer hook was not restored')
        raw_exit={k:full_audit_worker_state.get(k) for k in ['state','error_type','error','checkpoint_save_error']}
        if raw_exit['error_type']!='OwnerRequestedDataOnlyPause':
            raise RuntimeError('Worker stopped for a reason other than the requested pause')
        proof=full_audit_worker_state['checkpoint']
        if not proof.get('fresh_raw_readback_verified') or proof['observed_utc']<owner_pause_requested_utc:
            raise RuntimeError('No fresh raw-verified checkpoint after the owner pause request')
        snapshot=ROOT/'full_audit_v3_snapshot.sqlite'
        if snapshot.stat().st_size!=proof['bytes'] or file_digest(snapshot)!=proof['sha256']:
            raise RuntimeError('Portable snapshot differs from verified Drive checkpoint')
        with sqlite3.connect(V3_DB.as_uri()+'?mode=ro',uri=True) as a, sqlite3.connect(snapshot.as_uri()+'?mode=ro',uri=True) as b:
            cursors_a=a.execute('SELECT id,role,next_line,complete,expected_rows FROM members ORDER BY id').fetchall()
            cursors_b=b.execute('SELECT id,role,next_line,complete,expected_rows FROM members ORDER BY id').fetchall()
            issues_a=a.execute('SELECT * FROM issues ORDER BY member,line,kind,detail').fetchall()
            issues_b=b.execute('SELECT * FROM issues ORDER BY member,line,kind,detail').fetchall()
            if cursors_a!=cursors_b or issues_a!=issues_b:
                raise RuntimeError('Committed member cursor or issue rows differ from the saved snapshot')
            rows=a.execute('SELECT count(*) FROM samples').fetchone()[0]
            physical=a.execute('SELECT count(*) FROM occurrences').fetchone()[0]
            if rows!=proof['report_at_snapshot']['unique_chat_rows_processed'] or physical!=proof['report_at_snapshot']['physical_chat_occurrences_processed']:
                raise RuntimeError('Saved checkpoint does not include the final committed audit progress')
        a.close();b.close()
        metadata_path=qa_dir/'final_metadata_status.json'
        previous_metadata=json.loads(metadata_path.read_bytes()) if metadata_path.exists() else None
        owner_pause_status.update(state='PAUSED_BY_OWNER_AT_FRESH_RAW_VERIFIED_CHECKPOINT',
            verified_utc=datetime.now(timezone.utc).isoformat(),checkpoint=proof,
            unique_chat_rows_saved=rows,physical_chat_rows_saved=physical,
            member_cursors_equal_saved_snapshot=True,prior_issues_preserved=True,
            member_cursors_sha256=hashlib.sha256(json.dumps(cursors_a,separators=(',',':')).encode()).hexdigest(),
            tokenizer_hook_restored=True,raw_worker_exit_preserved=raw_exit,
            prior_final_metadata_status_preserved=previous_metadata,
            worker_alive=False,final_metadata_worker_alive=False,
            resume_policy='Restore this exact same-ID checkpoint and recipe identity. Do not rerun generation or append64. Re-register the finite final metadata check only after resuming the single worker.')
        full_audit_worker_state['owner_pause_receipt']=owner_pause_status
        full_audit_worker_state['state']='PAUSED_BY_OWNER_AT_FRESH_RAW_VERIFIED_CHECKPOINT'
        write_worker_status()
        metadata_path.write_text(json.dumps({'state':'PAUSED_BY_OWNER_AWAITING_RESUME',
            'owner_pause_requested_utc':owner_pause_requested_utc,'checkpoint_sha256':proof['sha256'],
            'unique_chat_rows_saved':rows,'prior_status_preserved':previous_metadata,
            'training_started':False},ensure_ascii=False,indent=2))
    except Exception as error:
        owner_pause_status.update(state='PAUSE_SAVE_NEEDS_ATTENTION',error_type=type(error).__name__,error=str(error))
    finally:
        write_owner_pause_status()

write_owner_pause_status()
owner_pause_confirmation_thread=threading.Thread(target=confirm_owner_pause_after_save,
    name='MathVisionKidOwnerRequestedPauseSave',daemon=True)
owner_pause_confirmation_thread.start()
batch_module._ORIGINAL_TOKENIZE=owner_pause_tokenize_gate
print(json.dumps({'owner_pause_requested':True,'state':owner_pause_status['state'],
    'not_safe_to_close_before_verification_message':True,'training_started':False}))

{"owner_pause_requested": true, "state": "OWNER_PAUSE_REQUESTED_WAITING_FOR_VERIFIED_SAVE", "not_safe_to_close_before_verification_message": true, "training_started": false}


In [ ]:
import json,sys
from datetime import datetime,timezone
pp=qa_dir/'owner_pause_receipt.json'
save_progress=None
pause_frame=sys._current_frames().get(full_audit_worker.ident) if full_audit_worker.is_alive() else None
while pause_frame is not None and pause_frame.f_code.co_name!='save_checkpoint':pause_frame=pause_frame.f_back
if pause_frame is not None:
    pause_sink=pause_frame.f_locals.get('sink');pause_snapshot=pause_frame.f_locals.get('snapshot')
    save_progress={'raw_readback_bytes':getattr(pause_sink,'size',None),'snapshot_bytes':pause_snapshot.stat().st_size if pause_snapshot is not None and pause_snapshot.exists() else None}
print(json.dumps({'observed_utc':datetime.now(timezone.utc).isoformat(),'worker_alive':full_audit_worker.is_alive(),'worker_state':full_audit_worker_state.get('state'),'owner_pause_confirmation_alive':owner_pause_confirmation_thread.is_alive(),'save_progress':save_progress,'pause_receipt':json.loads(pp.read_bytes()),'training_started':False},ensure_ascii=False))

{"observed_utc": "2026-10-22T17:28:21.580112+00:00", "worker_alive": false, "worker_state": "PAUSED_BY_OWNER_AT_FRESH_RAW_VERIFIED_CHECKPOINT", "owner_pause_confirmation_alive": false, "save_progress": null, "pause_receipt": {"state": "PAUSED_BY_OWNER_AT_FRESH_RAW_VERIFIED_CHECKPOINT", "requested_utc": "2026-10-22T17:17:48.431015+00:00", "training_started": false, "mechanism": "Stop before a later row token gate; original consume rolls back only the current uncommitted256-row transaction. Original worker saves and raw-verifies its complete portable checkpoint.", "data_qa_finding_created": false, "verified_utc": "2026-10-22T17:27:44.294077+00:00", "checkpoint": {"id": "16Q9ft9xhcioconk-5zpxmdRv1ZDNxqFI", "bytes": 7083466752, "sha256": "3c7cb05061c0040812111a6bb73a0709cc59a244493dc135a3b95aa389bf0648", "fresh_raw_readback_verified": true, "report_at_snapshot": {"schema": "MathVisionKid.full_current_archive_audit.v3", "identity": "f0332df91020ace0fc48d88491fe97364d08826458d08c71ba4ec60524

In [1]:
import json,threading
from pathlib import Path
from datetime import datetime,timezone
print(json.dumps({'observed_utc':datetime.now(timezone.utc).isoformat(),'runtime_root_exists':Path('/content/mathvisionkid_data_only').exists(),'globals_present':[k for k in ['ROOT','credentials','recipe','recipe_all','V3_ID','V3_DB','full_audit_worker','batch_module','batch_adapter','tokenizer','qa_dir','finite_full_data_audit_worker_v3','g2_connected_api'] if k in globals()],'threads':[{'name':t.name,'alive':t.is_alive()} for t in threading.enumerate() if 'MathVision' in t.name],'worker_state':globals().get('full_audit_worker_state',{}).get('state'),'training_started':False},ensure_ascii=False))

{"observed_utc": "2026-10-23T01:24:00.037414+00:00", "runtime_root_exists": false, "globals_present": [], "threads": [], "worker_state": null, "training_started": false}
